# Experimental Bayesian Inference with PBS Emulator

This notebook loads a trained PBS emulator artifact, imports experimental Tongji degradation trajectories, performs full-trajectory Bayesian inference, and exports figure panels for trajectory matching, effective-parameter inference, and early-to-late prediction.


## Module 1. Configuration


In [ ]:
from pathlib import Path

import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PBS_DATASET_ID = "pbs_lhs_400"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_ROOT / "figures"
CACHE_ROOT = OUTPUT_ROOT / "cache"
PBS_DATA_ROOT = DATA_ROOT / "pbs" / PBS_DATASET_ID
TONGJI_DATA_ROOT = DATA_ROOT / "tongji"
DATASET_CACHE_DIR = CACHE_ROOT / PBS_DATASET_ID

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DATASET_CACHE_DIR.mkdir(parents=True, exist_ok=True)

EMULATOR_ARTIFACT_PATH = DATASET_CACHE_DIR / f"pbs_emulator_{PBS_DATASET_ID}_multiplier_linearcycle_v1.joblib"
SCAN_METADATA_JSON = PBS_DATA_ROOT / "scan_metadata.json"
TONGJI_CAPACITY_CSV = TONGJI_DATA_ROOT / "tongji_capacity_trajectories.csv"
BATTERY_DATAFRAME_CSV = TONGJI_DATA_ROOT / "battery_dataframe.csv"

N_EXPANDED_SAMPLES = 30000
RANDOM_SEED = 2027
CURVE_SPACING_LOSS_WEIGHT = 1.0
SHAPE_SLOPE_LOSS_WEIGHT = 1.0
SHAPE_SEGMENT_LEVELS = [100, 95, 90, 85, 80]
MINIMUM_ANCHOR_TO_80_SPAN = 200
INCLUDED_TONGJI_MAJOR_GROUPS = ["Tongji2", "Tongji3"]

if not SCAN_METADATA_JSON.exists():
    raise FileNotFoundError(f"PBS scan metadata not found: {SCAN_METADATA_JSON}")
scan_metadata = json.loads(SCAN_METADATA_JSON.read_text(encoding="utf-8"))
EXPANDED_MULTIPLIER_RANGES = {
    str(name): tuple(float(v) for v in values)
    for name, values in scan_metadata.get("variation_ranges", {}).items()
}

print(f"PBS emulator artifact path: {EMULATOR_ARTIFACT_PATH}")
print(f"PBS scan metadata: {SCAN_METADATA_JSON}")
print(f"Tongji capacity CSV: {TONGJI_CAPACITY_CSV}")
print(f"Battery dataframe CSV: {BATTERY_DATAFRAME_CSV}")
print(f"Included Tongji groups: {INCLUDED_TONGJI_MAJOR_GROUPS}")


## Module 2. Imports and surrogate-loading helpers


In [ ]:
def invert_monotonic_forward_targets(y_transformed, soh_count):
    base_cycle = y_transformed[:, [0]]
    delta_cycles = np.clip(y_transformed[:, 1:soh_count], 1e-12, None)
    extra = y_transformed[:, soh_count:]
    soh_cycles = np.concatenate([base_cycle, base_cycle + np.cumsum(delta_cycles, axis=1)], axis=1)
    soh_cycles = np.clip(soh_cycles, 1e-12, None)
    return np.hstack([soh_cycles, extra])


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def load_emulator_artifact(path):
    if not Path(path).exists():
        raise FileNotFoundError(f"PBS emulator artifact not found: {path}. Train the PBS emulator notebook first or update the path.")
    artifact = joblib.load(path)
    required_keys = {"model", "x_scaler", "y_scaler", "metadata"}
    missing = required_keys - set(artifact)
    if missing:
        raise ValueError(f"Artifact is missing required keys: {sorted(missing)}")
    return artifact


artifact = load_emulator_artifact(EMULATOR_ARTIFACT_PATH)
nn_forward_model = artifact["model"]
nn_x_scaler = artifact["x_scaler"]
nn_y_scaler = artifact["y_scaler"]
artifact_meta = artifact["metadata"]
param_names = list(artifact_meta["param_names"])
nn_target_cols = list(artifact_meta["target_cols"])
soh_cols = list(artifact_meta["soh_cols"])
nn_soh_targets = len(soh_cols)

if artifact_meta.get("target_encoding") != "linear_cycle_monotonic_delta":
    raise ValueError(
        f"This notebook expects a linear-cycle emulator artifact, got target_encoding={artifact_meta.get('target_encoding')!r}."
    )

print(json.dumps(artifact_meta, indent=2))


def nn_forward_predict_monotonic_from_normalized(X_norm):
    y_pred_scaled = nn_forward_model.predict(X_norm)
    return nn_y_scaler.inverse_transform(y_pred_scaled)


def nn_forward_predict_raw_from_normalized(X_norm):
    y_pred_monotonic = nn_forward_predict_monotonic_from_normalized(X_norm)
    return invert_monotonic_forward_targets(y_pred_monotonic, nn_soh_targets)


## Module 3. Load Tongji metadata and capacity-based SOH trajectories


In [ ]:
if not BATTERY_DATAFRAME_CSV.exists():
    raise FileNotFoundError(BATTERY_DATAFRAME_CSV)
if not TONGJI_CAPACITY_CSV.exists():
    raise FileNotFoundError(TONGJI_CAPACITY_CSV)

df = pd.read_csv(BATTERY_DATAFRAME_CSV)
tongji_capacity_df = pd.read_csv(TONGJI_CAPACITY_CSV)

tongji_cells_from_df = set(
    df.loc[df["dataset_name"].astype(str).str.contains("TONGJI", case=False, na=False), "cell_id"]
    .astype(str)
    .tolist()
)
if not tongji_cells_from_df:
    raise ValueError("No Tongji cells found in battery_dataframe.csv")

required_cols = ["cell", "cycle", "capacity_Ah", "soh_pct"]
missing_cols = [col for col in required_cols if col not in tongji_capacity_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in Tongji capacity CSV: {missing_cols}")

tongji_soh_raw = tongji_capacity_df.loc[
    tongji_capacity_df["cell"].astype(str).isin(tongji_cells_from_df)
].copy()
for col in ["cycle", "capacity_Ah", "capacity_Ah_raw", "nominal_capacity_Ah", "soh_pct"]:
    if col in tongji_soh_raw.columns:
        tongji_soh_raw[col] = pd.to_numeric(tongji_soh_raw[col], errors="coerce")

tongji_soh_raw = tongji_soh_raw.dropna(subset=["cell", "cycle", "capacity_Ah", "soh_pct"])
tongji_soh_raw = tongji_soh_raw[tongji_soh_raw["cycle"] > 0].copy()
tongji_soh_raw["cell"] = tongji_soh_raw["cell"].astype(str)
tongji_soh_raw = tongji_soh_raw.loc[
    tongji_soh_raw["cell"].str.split("_").str[0].isin(INCLUDED_TONGJI_MAJOR_GROUPS)
].copy()
if tongji_soh_raw.empty:
    raise ValueError(
        f"No Tongji rows remain after filtering to included groups: {INCLUDED_TONGJI_MAJOR_GROUPS}"
    )

tongji_soh_raw["soh"] = tongji_soh_raw["soh_pct"].astype(float)
tongji_soh_raw["dataset"] = "TONGJI(NMC)"
tongji_soh_raw["soh_int"] = tongji_soh_raw["soh"].round().astype(int)

tongji_soh_raw = tongji_soh_raw.sort_values(["cell", "cycle"]).reset_index(drop=True)
first_capacity_by_cell = tongji_soh_raw.groupby("cell")["capacity_Ah"].transform("first")
tongji_soh_raw["self_soh"] = 100.0 * tongji_soh_raw["capacity_Ah"] / first_capacity_by_cell
tongji_soh_raw["self_soh_int"] = tongji_soh_raw["self_soh"].round().astype(int)

print(f"Tongji cells from metadata: {len(tongji_cells_from_df)}")
print(f"Tongji cells in capacity CSV: {tongji_capacity_df['cell'].nunique()}")
print(f"Usable Tongji trajectory rows after filtering: {len(tongji_soh_raw)}")
print(f"Remaining Tongji cells: {tongji_soh_raw['cell'].nunique()}")
display(tongji_soh_raw[["cell", "cycle", "capacity_Ah", "soh", "self_soh", "self_soh_int"]].head())


In [ ]:
if tongji_soh_raw.empty:
    raise ValueError("Tongji trajectory table is empty.")

tongji_plot_df = tongji_soh_raw.sort_values(["cell", "cycle", "soh"]).copy()
tongji_cell_count = int(tongji_plot_df["cell"].nunique())

anchor_summary = (
    tongji_plot_df.groupby("cell", as_index=False)
    .agg(
        start_cycle=("cycle", "min"),
        start_soh=("soh", "max"),
        end_cycle=("cycle", "max"),
        end_soh=("soh", "min"),
        max_capacity_Ah=("capacity_Ah", "max"),
    )
)

cycle_at_80 = (
    tongji_plot_df.loc[tongji_plot_df["soh_int"].eq(80), ["cell", "cycle"]]
    .groupby("cell", as_index=False)["cycle"]
    .min()
    .rename(columns={"cycle": "cycle_at_80"})
)
anchor_summary = anchor_summary.merge(cycle_at_80, on="cell", how="left")
if anchor_summary["cycle_at_80"].isna().all():
    raise ValueError("No Tongji cell reaches 80% SOH, so color mapping cannot be built.")

cmap = plt.cm.coolwarm
norm = plt.Normalize(anchor_summary["cycle_at_80"].min(), anchor_summary["cycle_at_80"].max())
cell_to_color = {
    row.cell: cmap(norm(row.cycle_at_80)) if pd.notna(row.cycle_at_80) else (0.75, 0.75, 0.75, 0.6)
    for row in anchor_summary.itertuples()
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for cell, grp in tongji_plot_df.groupby("cell"):
    color = cell_to_color.get(cell, (0.75, 0.75, 0.75, 0.6))
    axes[0].plot(
        grp["cycle"],
        grp["soh"],
        color=color,
        alpha=0.45,
        linewidth=0.9,
        marker="o",
        markersize=2.8,
        markeredgewidth=0,
    )

axes[0].set_title(f"Tongji raw degradation paths ({tongji_cell_count} cells)")
axes[0].set_xlabel("Cycle")
axes[0].set_ylabel("SOH (%)")
axes[0].grid(alpha=0.25)
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar0 = fig.colorbar(sm, ax=axes[0])
cbar0.set_label("Cycle at 80% SOH")

self_norm_rows = []
for cell, grp in tongji_plot_df.groupby("cell"):
    grp = grp.sort_values("cycle").copy()
    max_capacity = float(grp["capacity_Ah"].max()) if "capacity_Ah" in grp.columns else float("nan")
    if not np.isfinite(max_capacity) or max_capacity <= 0:
        continue
    grp["self_norm_soh"] = grp["capacity_Ah"] / max_capacity * 100.0
    self_norm_rows.append(grp)
    color = cell_to_color.get(cell, (0.75, 0.75, 0.75, 0.6))
    axes[1].plot(
        grp["cycle"],
        grp["self_norm_soh"],
        color=color,
        alpha=0.45,
        linewidth=0.9,
        marker="o",
        markersize=2.8,
        markeredgewidth=0,
    )

axes[1].set_title("Tongji self-normalized degradation paths")
axes[1].set_xlabel("Cycle")
axes[1].set_ylabel("Self-normalized SOH (%)")
axes[1].set_ylim(78, 101)
axes[1].grid(alpha=0.25)
sm1 = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm1.set_array([])
cbar1 = fig.colorbar(sm1, ax=axes[1])
cbar1.set_label("Cycle at 80% SOH")

plt.show()


## Module 4. Build self-normalized Tongji paths (cycle 1 -> 100% SOH)

Each Tongji cell is converted into its own degradation path by setting the first recorded cycle to `(cycle=1, SOH=100%)` in the Tongji space. We then interpolate the cycle positions at integer SOH levels from 100% down to 80%.

Important normalization rule for the later fitting step:
- the Tongji path is normalized only by the Tongji dataset cycle scale
- the surrogate path is normalized only by the surrogate dataset cycle scale
- the match therefore compares relative shape across two different spaces, rather than forcing a shared absolute life axis


In [ ]:
full_surrogate_soh_cols = sorted(
    [col for col in nn_target_cols if str(col).endswith("% SOH")],
    key=lambda col: int(str(col).split("%")[0]),
    reverse=True,
)
full_surrogate_soh_levels = [int(str(col).split("%")[0]) for col in full_surrogate_soh_cols]
level_to_sur_idx = {level: idx for idx, level in enumerate(full_surrogate_soh_levels)}

tongji_dataset_max_cycle = float(tongji_soh_raw["cycle"].max())
if tongji_dataset_max_cycle <= 0:
    raise ValueError("Tongji dataset max cycle must be positive.")


def estimate_cycle_at_soh(levels_desc, cycles, target_level):
    levels_desc = np.asarray(levels_desc, dtype=float)
    cycles = np.asarray(cycles, dtype=float)
    if levels_desc.size < 2:
        return float("nan")
    if target_level > levels_desc[0] or target_level < levels_desc[-1]:
        return float("nan")
    for i in range(len(levels_desc) - 1):
        s0, s1 = float(levels_desc[i]), float(levels_desc[i + 1])
        c0, c1 = float(cycles[i]), float(cycles[i + 1])
        if not (np.isfinite(s0) and np.isfinite(s1) and np.isfinite(c0) and np.isfinite(c1)):
            continue
        if s0 >= target_level >= s1:
            if abs(s0 - s1) < 1e-12:
                return c0
            frac = (target_level - s1) / (s0 - s1)
            return c1 + frac * (c0 - c1)
    return float("nan")


tongji_path_records = []
interp_plot_rows = []
for cell, grp in tongji_soh_raw.groupby("cell"):
    grp = grp.sort_values("cycle").copy()
    self_soh = grp["self_soh"].to_numpy(dtype=float)
    cycles = grp["cycle"].to_numpy(dtype=float)

    valid = np.isfinite(self_soh) & np.isfinite(cycles)
    self_soh = self_soh[valid]
    cycles = cycles[valid]
    if len(self_soh) < 4:
        continue

    self_soh_monotone = np.minimum.accumulate(self_soh)
    cycle1 = float(cycles[0])
    # Keep the experimental path explicitly anchored at (cycle 1, 100% SOH).
    # The cached emulator usually stores crossing cycles from 99% to 80% SOH,
    # so 100% is added manually instead of being filtered by surrogate target columns.
    compare_levels = [100] + [
        level for level in range(99, 79, -1)
        if level in full_surrogate_soh_levels
    ]

    level_cycles = []
    for level in compare_levels:
        if level == 100:
            level_cycles.append(cycle1)
        else:
            cyc = estimate_cycle_at_soh(self_soh_monotone, cycles, float(level))
            level_cycles.append(cyc)

    level_cycles = np.asarray(level_cycles, dtype=float)
    valid_levels = np.isfinite(level_cycles)
    if not np.all(valid_levels):
        first_bad = int(np.flatnonzero(~valid_levels)[0])
        compare_levels = compare_levels[:first_bad]
        level_cycles = level_cycles[:first_bad]

    if len(compare_levels) < 4 or 80 not in compare_levels:
        continue

    relative_cycles = level_cycles - cycle1
    if np.any(~np.isfinite(relative_cycles)) or np.any(relative_cycles < -1e-9):
        continue
    if relative_cycles[-1] < MINIMUM_ANCHOR_TO_80_SPAN:
        continue

    exp_space_norm_cycles = relative_cycles / tongji_dataset_max_cycle

    tongji_path_records.append({
        "cell": cell,
        "anchor_soh": 100,
        "compare_levels": compare_levels,
        "tongji_level_cycles": level_cycles,
        "tongji_relative_cycles": relative_cycles,
        "tongji_exp_space_norm_cycles": exp_space_norm_cycles,
        "tongji_norm_cycles": exp_space_norm_cycles,
        "n_compare_levels": len(compare_levels),
        "span_anchor_to_80": float(relative_cycles[-1]),
    })

    interp_plot_rows.append(pd.DataFrame({
        "cell": cell,
        "cycle": cycles,
        "self_soh": self_soh,
        "self_soh_monotone": self_soh_monotone,
    }))
    interp_plot_rows.append(pd.DataFrame({
        "cell": cell,
        "cycle": level_cycles,
        "self_soh": compare_levels,
        "self_soh_monotone": compare_levels,
        "is_interp_point": True,
    }))

if not tongji_path_records:
    raise ValueError(
        "No Tongji cell has a usable self-normalized path from 100% SOH down to 80% SOH. "
        "Check whether the self-normalized curves actually cross 80% within the available cycle range."
    )

print(f"Built self-normalized Tongji paths for {len(tongji_path_records)} cells")
print(f"Tongji normalization basis (own dataset max cycle): {tongji_dataset_max_cycle:.2f}")
display(pd.DataFrame(tongji_path_records)[["cell", "anchor_soh", "n_compare_levels", "span_anchor_to_80"]].head())

interp_plot_df = pd.concat(interp_plot_rows, ignore_index=True, sort=False)
interp_plot_df["is_interp_point"] = interp_plot_df.get("is_interp_point", False).fillna(False).astype(bool)
interp_summary = pd.DataFrame(tongji_path_records)[["cell", "span_anchor_to_80"]].copy()
interp_summary["cycle_at_80"] = interp_summary["span_anchor_to_80"] + 1.0
cmap = plt.cm.coolwarm
norm = plt.Normalize(interp_summary["cycle_at_80"].min(), interp_summary["cycle_at_80"].max())
cell_to_color = {
    row.cell: cmap(norm(row.cycle_at_80)) if np.isfinite(row.cycle_at_80) else (0.75, 0.75, 0.75, 0.6)
    for row in interp_summary.itertuples()
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for cell, grp in interp_plot_df.groupby("cell"):
    color = cell_to_color.get(cell, (0.75, 0.75, 0.75, 0.6))
    raw_part = grp.loc[~grp["is_interp_point"]].sort_values("cycle")
    interp_part = grp.loc[grp["is_interp_point"]].sort_values("cycle")
    axes[0].plot(
        raw_part["cycle"],
        raw_part["self_soh"],
        color=color,
        alpha=0.35,
        linewidth=0.9,
    )
    axes[0].plot(
        interp_part["cycle"],
        interp_part["self_soh"],
        color=color,
        alpha=0.8,
        linewidth=1.1,
        marker="o",
        markersize=2.8,
        markeredgewidth=0,
    )

axes[0].set_title("Self-normalized Tongji paths with interpolated SOH points")
axes[0].set_xlabel("Cycle")
axes[0].set_ylabel("Self-normalized SOH (%)")
axes[0].grid(alpha=0.25)
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar0 = fig.colorbar(sm, ax=axes[0])
cbar0.set_label("Cycle at 80% SOH")

for cell, grp in interp_plot_df.groupby("cell"):
    color = cell_to_color.get(cell, (0.75, 0.75, 0.75, 0.6))
    interp_part = grp.loc[grp["is_interp_point"]].sort_values("self_soh", ascending=False)
    if interp_part.empty:
        continue
    rel_cycle = interp_part["cycle"].to_numpy(dtype=float) - float(interp_part["cycle"].iloc[0])
    axes[1].plot(
        rel_cycle / tongji_dataset_max_cycle,
        interp_part["self_soh"],
        color=color,
        alpha=0.8,
        linewidth=1.1,
        marker="o",
        markersize=2.8,
        markeredgewidth=0,
    )

axes[1].set_title("Interpolated 100%->80% paths used for fitting")
axes[1].set_xlabel("Tongji normalized cycle (Tongji space)")
axes[1].set_ylabel("Self-normalized SOH (%)")
axes[1].grid(alpha=0.25)
sm1 = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm1.set_array([])
cbar1 = fig.colorbar(sm1, ax=axes[1])
cbar1.set_label("Cycle at 80% SOH")

plt.show()


In [ ]:
if "interp_plot_df" not in globals() or interp_plot_df.empty:
    raise ValueError("Run Module 4 first so interp_plot_df is available.")
if "tongji_dataset_max_cycle" not in globals() or not np.isfinite(tongji_dataset_max_cycle) or tongji_dataset_max_cycle <= 0:
    raise ValueError("Run Module 4 first so tongji_dataset_max_cycle is available.")

def format_crate_token_module4_subgroup(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token

def parse_tongji_subgroup_module4(cell_name):
    cell_name = str(cell_name)
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major, middle, suffix = parts[:3]
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                discharge_token = suffix.split("--")[0]
                temp_group = f"{int(temp_token)}℃" if temp_token.isdigit() else temp_token
                rate_group = (
                    f"{format_crate_token_module4_subgroup(charge_token)}/"
                    f"{format_crate_token_module4_subgroup(discharge_token)}"
                )
                return f"{major} | {temp_group} | {rate_group}"
    except Exception:
        pass
    return "Unknown"

module4_subgroup_plot_df = interp_plot_df.loc[interp_plot_df["is_interp_point"]].copy()
if module4_subgroup_plot_df.empty:
    raise ValueError("No interpolated SOH points are available for subgroup plotting.")

module4_subgroup_plot_df["subgroup"] = module4_subgroup_plot_df["cell"].apply(parse_tongji_subgroup_module4)
subgroup_order = sorted(module4_subgroup_plot_df["subgroup"].dropna().astype(str).unique())
if not subgroup_order:
    raise ValueError("No Tongji subgroup labels could be parsed.")

base_cmap = plt.cm.get_cmap("Set2", max(len(subgroup_order), 1))
subgroup_to_color = {
    subgroup: base_cmap(i % base_cmap.N)
    for i, subgroup in enumerate(subgroup_order)
}

fig, ax = plt.subplots(figsize=(5,4), dpi=500)
seen_subgroups = set()

for cell, grp in module4_subgroup_plot_df.groupby("cell"):
    grp = grp.sort_values("self_soh", ascending=False)
    if grp.empty:
        continue
    subgroup = str(grp["subgroup"].iloc[0])
    color = subgroup_to_color.get(subgroup, (0.6, 0.6, 0.6, 0.8))
    rel_cycle = grp["cycle"].to_numpy(dtype=float) - float(grp["cycle"].iloc[0])
    ax.plot(
        rel_cycle / tongji_dataset_max_cycle,
        grp["self_soh"].to_numpy(dtype=float),
        color=color,
        alpha=0.8,
        linewidth=1,
        marker="o",
        markersize=3,
        markeredgewidth=0,
        label=subgroup if subgroup not in seen_subgroups else None,
    )
    seen_subgroups.add(subgroup)

ax.set_xlabel("Normalized cycle", fontsize=12)
ax.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_ylabel("State of Health (SOH%)", fontsize=12)
ax.set_yticks([80, 85, 90, 95, 100])
ax.set_ylim(79, 101)
ax.legend(
    loc="upper right",
    fontsize=10,
    frameon=False,
)

plt.savefig(FIGURE_DIR / "F3a.tiff", dpi=500, bbox_inches="tight")
plt.show()

print(f"Plotted {module4_subgroup_plot_df['cell'].nunique()} Tongji cells across {len(subgroup_order)} subgroups")


## Module 5. Generate a surrogate candidate pool from the cached NN model


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
missing_ranges = [name for name in param_names if name not in EXPANDED_MULTIPLIER_RANGES]
if missing_ranges:
    available = sorted(EXPANDED_MULTIPLIER_RANGES)
    raise ValueError(
        f"Missing multiplier ranges for: {missing_ranges}. "
        f"Available names from scan_metadata.json are: {available}"
    )

ordered_ranges = [EXPANDED_MULTIPLIER_RANGES[name] for name in param_names]
X_expanded_multiplier = np.column_stack([
    rng.uniform(low, high, size=N_EXPANDED_SAMPLES)
    for (low, high) in ordered_ranges
])
X_expanded_norm = nn_x_scaler.transform(X_expanded_multiplier)
Y_expanded_raw = nn_forward_predict_raw_from_normalized(X_expanded_norm)
Y_expanded_cycles = Y_expanded_raw[:, [nn_target_cols.index(col) for col in full_surrogate_soh_cols]]
surrogate_dataset_max_cycle = float(np.nanmax(Y_expanded_cycles))
if surrogate_dataset_max_cycle <= 0:
    raise ValueError("Surrogate dataset max cycle must be positive.")

print(f"Generated {len(X_expanded_multiplier)} surrogate candidates")
print(f"Parameter ranges used from scan metadata: {dict(zip(param_names, ordered_ranges))}")
print(f"Surrogate max cycle for normalization: {surrogate_dataset_max_cycle:.2f}")


## Module 6. Shape-and-spacing Tongji matching

The fitting objective is kept intentionally simple. Each Tongji cell is matched to the surrogate pool using only two terms:

1. `curve spacing loss`: the mean squared distance between Tongji and surrogate cycle positions at all shared SOH levels.
2. `shape loss`: the mean squared difference of the four coarse slopes defined by `100->95`, `95->90`, `90->85`, and `85->80`.

Both paths are still compared in their own normalized spaces:
- Tongji points are normalized by the Tongji dataset cycle scale
- surrogate points are normalized by the surrogate dataset cycle scale

This keeps the fit focused on trajectory spacing and coarse shape without adding extra higher-order penalties.


In [ ]:
map_rows = []
long_rows = []
required_surrogate_shape_levels = [95, 90, 85, 80]
missing_shape_levels = [level for level in required_surrogate_shape_levels if level not in level_to_sur_idx]
if missing_shape_levels:
    raise ValueError(f"Surrogate SOH grid is missing required shape levels: {missing_shape_levels}")
shape_step = 5.0

for rec in tongji_path_records:
    exp_level_to_norm = dict(zip(rec["compare_levels"], rec["tongji_exp_space_norm_cycles"]))
    exp_level_to_norm[100] = 0.0

    common_levels = [level for level in rec["compare_levels"] if level in level_to_sur_idx]
    if len(common_levels) < 4 or 80 not in common_levels:
        continue
    if not all(level in exp_level_to_norm for level in SHAPE_SEGMENT_LEVELS):
        continue
    if not all(level in common_levels for level in required_surrogate_shape_levels):
        continue

    idxs = [level_to_sur_idx[level] for level in common_levels]

    # Surrogate crossing cycles are already measured from the beginning of life,
    # so the implicit 100% SOH anchor is cycle 0 in the surrogate space.
    sur_relative_cycles = Y_expanded_cycles[:, idxs]
    valid_sur = np.all(np.isfinite(sur_relative_cycles), axis=1) & np.all(sur_relative_cycles >= 0, axis=1)
    if not np.any(valid_sur):
        continue

    valid_indices = np.flatnonzero(valid_sur)
    surrogate_space_norm = sur_relative_cycles / surrogate_dataset_max_cycle
    exp_space_norm = np.asarray([exp_level_to_norm[level] for level in common_levels], dtype=float)

    spacing_sq = (surrogate_space_norm[valid_sur] - exp_space_norm[None, :]) ** 2
    curve_spacing_mse = np.mean(spacing_sq, axis=1)
    curve_spacing_rmse = np.sqrt(curve_spacing_mse)

    exp_shape_points = np.asarray([exp_level_to_norm[level] for level in SHAPE_SEGMENT_LEVELS], dtype=float)
    exp_shape_slopes = np.diff(exp_shape_points) / shape_step

    sur_shape_points_list = [np.zeros(len(valid_indices), dtype=float)]
    for level in required_surrogate_shape_levels:
        level_pos = common_levels.index(level)
        sur_shape_points_list.append(surrogate_space_norm[valid_sur][:, level_pos])
    sur_shape_points = np.column_stack(sur_shape_points_list)
    sur_shape_slopes = np.diff(sur_shape_points, axis=1) / shape_step
    slope_sq = (sur_shape_slopes - exp_shape_slopes[None, :]) ** 2
    shape_slope_mse = np.mean(slope_sq, axis=1)
    shape_slope_rmse = np.sqrt(shape_slope_mse)

    total_loss = (
        CURVE_SPACING_LOSS_WEIGHT * curve_spacing_mse
        + SHAPE_SLOPE_LOSS_WEIGHT * shape_slope_mse
    )

    best_local = int(np.argmin(total_loss))
    best_global = int(valid_indices[best_local])
    best_sur_norm_common = surrogate_space_norm[best_global]
    sur_level_to_norm = {level: float(val) for level, val in zip(common_levels, best_sur_norm_common)}
    sur_level_to_norm[100] = 0.0

    row = {
        "cell": rec["cell"],
        "anchor_soh": 100,
        "surrogate_start_soh": 100,
        "n_compare_levels": rec["n_compare_levels"],
        "n_fit_levels": len(common_levels),
        "span_anchor_to_80": rec["span_anchor_to_80"],
        "tongji_dataset_max_cycle": float(tongji_dataset_max_cycle),
        "surrogate_dataset_max_cycle": float(surrogate_dataset_max_cycle),
        "match_loss_total": float(total_loss[best_local]),
        "curve_spacing_mse": float(curve_spacing_mse[best_local]),
        "curve_spacing_rmse": float(curve_spacing_rmse[best_local]),
        "shape_slope_mse": float(shape_slope_mse[best_local]),
        "shape_slope_rmse": float(shape_slope_rmse[best_local]),
        "best_surrogate_row": best_global,
    }
    for j, name in enumerate(param_names):
        row[f"matched_{name}"] = float(X_expanded_multiplier[best_global, j])

    plot_levels = [100] + [level for level in common_levels if level != 100]
    for level in plot_levels:
        exp_norm_val = float(exp_level_to_norm[level]) if level in exp_level_to_norm else 0.0
        sur_norm_val = float(sur_level_to_norm[level]) if level in sur_level_to_norm else 0.0
        if level in exp_level_to_norm:
            row[f"tongji_exp_space_norm_{level}"] = exp_norm_val
            row[f"tongji_norm_{level}"] = exp_norm_val
        if level in sur_level_to_norm:
            row[f"surrogate_space_norm_expSOH_{level}"] = sur_norm_val
            row[f"sur_norm_expSOH_{level}"] = sur_norm_val
            row[f"surrogate_level_for_expSOH_{level}"] = int(level)
        long_rows.append({
            "cell": rec["cell"],
            "anchor_soh": 100,
            "SOH": int(level),
            "surrogate_SOH": int(level),
            "tongji_exp_space_norm_cycle": exp_norm_val,
            "surrogate_space_norm_cycle": sur_norm_val,
            "tongji_norm_cycle": exp_norm_val,
            "sur_norm_cycle": sur_norm_val,
            "tongji_plot_cycle": float(exp_norm_val * tongji_dataset_max_cycle + 1.0),
            "sur_plot_cycle": float(sur_norm_val * surrogate_dataset_max_cycle + 1.0),
            "is_anchor_point": bool(level == 100),
            "used_in_fit": bool(level in common_levels),
            "match_loss_total": float(total_loss[best_local]),
            "curve_spacing_mse": float(curve_spacing_mse[best_local]),
            "curve_spacing_rmse": float(curve_spacing_rmse[best_local]),
            "shape_slope_mse": float(shape_slope_mse[best_local]),
            "shape_slope_rmse": float(shape_slope_rmse[best_local]),
        })
    map_rows.append(row)

tongji_map_df = pd.DataFrame(map_rows)
tongji_map_long_df = pd.DataFrame(long_rows)

if not tongji_map_long_df.empty and "SOH" in tongji_map_long_df.columns:
    tongji_map_long_df.loc[tongji_map_long_df["SOH"].eq(100), "used_in_fit"] = False

print(f"Tongji cells mapped: {len(tongji_map_df)}")
if tongji_map_df.empty:
    raise ValueError(
        "No Tongji cells were matched under the current spacing-plus-shape criteria. "
        "Check whether the experimental paths contain the required 100/95/90/85/80 levels, "
        "or whether the surrogate pool covers them after filtering."
    )
print(f"Curve-spacing / shape-slope weights = {CURVE_SPACING_LOSS_WEIGHT}, {SHAPE_SLOPE_LOSS_WEIGHT}")
print(f"Tongji normalization basis: {tongji_dataset_max_cycle:.2f} | Surrogate normalization basis: {surrogate_dataset_max_cycle:.2f}")
display(tongji_map_df[["match_loss_total", "curve_spacing_rmse", "shape_slope_rmse", "span_anchor_to_80", "n_fit_levels"]].describe().T)


## Module 7. Matching diagnostics and visualization

The diagnostics below follow the simplified fitting objective.

- The pointwise comparison stays in the normalized spaces used during fitting.
- The SOH-wise error summarizes how far the surrogate curve stays from the Tongji curve across the full path.
- The final panel checks whether long-lived Tongji cells are systematically harder to match under this spacing-plus-shape loss.


In [ ]:
if tongji_map_long_df.empty:
    raise ValueError("No Tongji matches were produced. Check the paths, ranges, or emulator cache.")

fit_only_df = tongji_map_long_df.loc[tongji_map_long_df["used_in_fit"]].copy()
point_rmse = rmse(fit_only_df["tongji_exp_space_norm_cycle"], fit_only_df["surrogate_space_norm_cycle"])
point_mae = float(mean_absolute_error(fit_only_df["tongji_exp_space_norm_cycle"], fit_only_df["surrogate_space_norm_cycle"]))
point_r2 = float(r2_score(fit_only_df["tongji_exp_space_norm_cycle"], fit_only_df["surrogate_space_norm_cycle"]))

print({
    "curve_spacing_rmse": point_rmse,
    "curve_spacing_mae": point_mae,
    "R2": point_r2,
    "n_fit_points": int(len(fit_only_df)),
    "tongji_norm_basis": float(tongji_dataset_max_cycle),
    "surrogate_norm_basis": float(surrogate_dataset_max_cycle),
})

soh_rows = []
for level in sorted(fit_only_df["SOH"].unique(), reverse=True):
    grp = fit_only_df.loc[fit_only_df["SOH"].eq(level)]
    if len(grp) < 2:
        continue
    soh_rows.append({
        "SOH": int(level),
        "curve_spacing_rmse": rmse(grp["tongji_exp_space_norm_cycle"], grp["surrogate_space_norm_cycle"]),
        "curve_spacing_mae": float(mean_absolute_error(grp["tongji_exp_space_norm_cycle"], grp["surrogate_space_norm_cycle"])),
        "R2": float(r2_score(grp["tongji_exp_space_norm_cycle"], grp["surrogate_space_norm_cycle"])),
        "n_points": int(len(grp)),
    })
soh_metric_df = pd.DataFrame(soh_rows)
display(soh_metric_df)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)
min_xy = float(np.nanmin([fit_only_df["tongji_exp_space_norm_cycle"].min(), fit_only_df["surrogate_space_norm_cycle"].min()]))
max_xy = float(np.nanmax([fit_only_df["tongji_exp_space_norm_cycle"].max(), fit_only_df["surrogate_space_norm_cycle"].max()]))
line_x = np.linspace(min_xy, max_xy, 100)

axes[0].scatter(
    fit_only_df["tongji_exp_space_norm_cycle"],
    fit_only_df["surrogate_space_norm_cycle"],
    s=24,
    alpha=0.7,
    color="#32745f",
    edgecolors="none",
)
axes[0].plot(line_x, line_x, linestyle="--", color="black", linewidth=1.0)
axes[0].set_title("Normalized spacing points: surrogate vs Tongji")
axes[0].set_xlabel("Tongji normalized cycle (Tongji space)")
axes[0].set_ylabel("Surrogate normalized cycle (surrogate space)")

axes[1].plot(soh_metric_df["SOH"], soh_metric_df["curve_spacing_rmse"], marker="o", color="#b65f2a", linewidth=1.6)
axes[1].set_title("Curve-spacing RMSE by SOH")
axes[1].set_xlabel("SOH (%)")
axes[1].set_ylabel("RMSE in normalized cycle")
axes[1].invert_xaxis()

axes[2].scatter(tongji_map_df["span_anchor_to_80"], tongji_map_df["curve_spacing_rmse"], s=42, alpha=0.8, color="#2f6f9f", edgecolors="none")
axes[2].set_title("Curve-spacing error vs 100-to-80 span")
axes[2].set_xlabel("100-to-80 span in Tongji cycle")
axes[2].set_ylabel("curve-spacing RMSE")
plt.show()


## Module 7A. All Tongji experimental-versus-surrogate fitted curves in normalized scale

This panel shows the matched Tongji and surrogate trajectories directly in the normalized spaces used by the optimizer. Both curves start at `(normalized cycle=0, SOH=100%)`, so the visual comparison focuses on the fitted trajectory difference itself rather than on separate display axes.


In [ ]:
if tongji_map_long_df.empty:
    raise ValueError("No Tongji matches are available for the all-cell fit overview.")

all_curve_df = tongji_map_long_df.copy()
all_curve_df["cell"] = all_curve_df["cell"].astype(str)
all_curve_df = all_curve_df.sort_values(["cell", "SOH"], ascending=[True, False]).reset_index(drop=True)
cell_order = sorted(all_curve_df["cell"].unique())
n_cells = len(cell_order)
n_cols = 4
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.1 * n_cols, 3.2 * n_rows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, cell_id in zip(axes, cell_order):
    sub = all_curve_df.loc[all_curve_df["cell"].eq(cell_id)].sort_values("SOH", ascending=False).copy()
    spacing_rmse_val = float(sub["curve_spacing_rmse"].iloc[0]) if len(sub) else float("nan")
    slope_rmse_val = float(sub["shape_slope_rmse"].iloc[0]) if len(sub) else float("nan")
    n_fit_levels = int(sub["used_in_fit"].sum()) if "used_in_fit" in sub.columns else 0

    if not sub.empty and 100 not in sub["SOH"].to_numpy(dtype=int):
        anchor_row = sub.iloc[[0]].copy()
        anchor_row["SOH"] = 100
        anchor_row["tongji_exp_space_norm_cycle"] = 0.0
        anchor_row["surrogate_space_norm_cycle"] = 0.0
        anchor_row["is_anchor_point"] = True
        anchor_row["used_in_fit"] = False
        sub = pd.concat([anchor_row, sub], ignore_index=True)
        sub = sub.sort_values("SOH", ascending=False)
    else:
        sub.loc[sub["SOH"].eq(100), "tongji_exp_space_norm_cycle"] = 0.0
        sub.loc[sub["SOH"].eq(100), "surrogate_space_norm_cycle"] = 0.0

    ax.plot(
        sub["tongji_exp_space_norm_cycle"],
        sub["SOH"],
        color="#2f6f9f",
        marker="o",
        markersize=3.6,
        linewidth=1.4,
        label="Exp" if cell_id == cell_order[0] else None,
    )
    ax.plot(
        sub["surrogate_space_norm_cycle"],
        sub["SOH"],
        color="#b65f2a",
        marker="s",
        markersize=3.2,
        linewidth=1.3,
        label="Surrogate" if cell_id == cell_order[0] else None,
    )

    title_bits = [cell_id, f"fit {n_fit_levels} lvls"]
    if np.isfinite(spacing_rmse_val):
        title_bits.append(f"spacing {spacing_rmse_val:.3f}")
    if np.isfinite(slope_rmse_val):
        title_bits.append(f"shape {slope_rmse_val:.3f}")
    ax.set_title(" | ".join(title_bits), fontsize=9)
    ax.set_xlabel("Normalized cycle")
    ax.set_ylabel("SOH (%)")
    ax.grid(alpha=0.22)
    ax.set_ylim(79, 101)
    ax.set_xlim(left=0.0)

for ax in axes[n_cells:]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
handles = [h for h, l in zip(handles, labels) if l]
labels = [l for l in labels if l]
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.01))

plt.show()


## Module 7B. Fitted surrogate parameter scatter by Tongji temperature and cycling-rate groups

This module parses the Tongji cell names to recover the temperature group and the charge/discharge-rate group, then plots each fitted surrogate parameter separately on its original multiplier scale. The first figure is grouped by temperature, and the second figure is grouped by charge/discharge-rate combination.


In [ ]:
if tongji_map_df.empty:
    raise ValueError("No Tongji parameter matches are available for grouped scatter plots.")

matched_param_cols = [col for col in tongji_map_df.columns if col.startswith("matched_")]
if not matched_param_cols:
    raise ValueError("No matched surrogate parameter columns were found in tongji_map_df.")

def format_crate_token(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token

def parse_tongji_cell_groups(cell_name):
    cell_name = str(cell_name)
    out = {
        "major_group": "Unknown",
        "temperature_group": "Unknown",
        "protocol_group": "Unknown",
    }
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major = parts[0]
            middle = parts[1]
            suffix = parts[2]
            out["major_group"] = major
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                temp_group = f"{int(temp_token)}C" if str(temp_token).isdigit() else temp_token
                discharge_token = suffix.split("--")[0]
                rate_group = f"{format_crate_token(charge_token)}/{format_crate_token(discharge_token)}"
                out["temperature_group"] = temp_group
                out["protocol_group"] = f"{major} | {temp_group} | {rate_group}"
    except Exception:
        pass
    return out

def deterministic_spread_positions(group_labels, category_order, width=0.28):
    x = np.zeros(len(group_labels), dtype=float)
    for idx, cat in enumerate(category_order):
        member_idx = np.flatnonzero(np.asarray(group_labels) == cat)
        n = len(member_idx)
        if n == 0:
            continue
        if n == 1:
            offsets = np.array([0.0])
        else:
            offsets = np.linspace(-width, width, n)
        x[member_idx] = idx + offsets
    return x

plot_df = tongji_map_df[["cell"] + matched_param_cols].copy()
parsed = plot_df["cell"].apply(parse_tongji_cell_groups).apply(pd.Series)
plot_df = pd.concat([plot_df, parsed], axis=1)

param_labels = [col.replace("matched_", "") for col in matched_param_cols]
n_params = len(matched_param_cols)
n_cols = 3
n_rows = int(np.ceil(n_params / n_cols))
major_order = [g for g in ["Tongji1", "Tongji2", "Tongji3"] if g in set(plot_df["major_group"].astype(str))]
if not major_order:
    major_order = sorted(plot_df["major_group"].dropna().astype(str).unique())

for major_group in major_order:
    sub_df = plot_df.loc[plot_df["major_group"].astype(str).eq(major_group)].copy()
    protocol_categories = sorted(sub_df["protocol_group"].dropna().astype(str).unique())
    if not protocol_categories:
        continue

    color_cycle = plt.cm.tab10(np.linspace(0, 1, max(len(protocol_categories), 3)))
    color_map = {cat: color_cycle[i] for i, cat in enumerate(protocol_categories)}
    x_positions = deterministic_spread_positions(sub_df["protocol_group"].astype(str).to_numpy(), protocol_categories, width=0.28)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.3 * n_cols, 3.2 * n_rows), constrained_layout=False)
    axes = np.atleast_1d(axes).ravel()

    for ax, param_col, param_label in zip(axes, matched_param_cols, param_labels):
        for cat in protocol_categories:
            mask = sub_df["protocol_group"].astype(str).eq(cat).to_numpy()
            ax.scatter(
                x_positions[mask],
                sub_df.loc[mask, param_col].to_numpy(dtype=float),
                s=28,
                alpha=0.82,
                color=color_map[cat],
                edgecolors="none",
                label=cat,
            )
        ax.set_title(param_label, fontsize=9)
        ax.set_xticks(range(len(protocol_categories)))
        ax.set_xticklabels(protocol_categories, rotation=28, ha="right")
        ax.set_ylabel("Multiplier")
        ax.grid(alpha=0.22)

    for ax in axes[n_params:]:
        ax.set_visible(False)

    handles = [plt.Line2D([0], [0], marker="o", linestyle="", markersize=6, color=color_map[cat], label=cat) for cat in protocol_categories]
    fig.legend(
        handles=handles,
        labels=protocol_categories,
        loc="center left",
        bbox_to_anchor=(0.995, 0.5),
        frameon=False,
        title="Protocol",
    )
    fig.suptitle(f"Fitted surrogate parameter multipliers for {major_group}", y=0.995, fontsize=12)
    fig.subplots_adjust(right=0.82, wspace=0.28, hspace=0.42, top=0.92, bottom=0.18)
    plt.show()


## Module 7C. Group-wise parameter stability and separability diagnostics

This module quantifies whether the fitted surrogate parameters are actually stable within nominally identical Tongji groups. For each matched parameter, it compares within-group dispersion against between-group mean separation under three grouping levels: major Tongji group, temperature group, and fine protocol group. The key diagnostic is a simple signal-to-dispersion ratio, so we can distinguish relatively stable effective parameters from highly degenerate compensation parameters.


In [ ]:
if tongji_map_df.empty:
    raise ValueError("No Tongji parameter matches are available for group-stability diagnostics.")

matched_param_cols = [col for col in tongji_map_df.columns if col.startswith("matched_")]
if not matched_param_cols:
    raise ValueError("No matched surrogate parameter columns were found in tongji_map_df.")

def format_crate_token(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token

def parse_tongji_cell_groups(cell_name):
    cell_name = str(cell_name)
    out = {
        "major_group": "Unknown",
        "temperature_group": "Unknown",
        "protocol_group": "Unknown",
    }
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major, middle, suffix = parts[:3]
            out["major_group"] = major
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                temp_group = f"{int(temp_token)}C" if str(temp_token).isdigit() else temp_token
                discharge_token = suffix.split("--")[0]
                rate_group = f"{format_crate_token(charge_token)}/{format_crate_token(discharge_token)}"
                out["temperature_group"] = temp_group
                out["protocol_group"] = f"{major} | {temp_group} | {rate_group}"
    except Exception:
        pass
    return out

def summarize_group_stability(df, group_col, param_cols):
    rows = []
    valid_df = df.copy()
    valid_df[group_col] = valid_df[group_col].astype(str)
    group_counts = valid_df[group_col].value_counts()
    kept_groups = group_counts[group_counts >= 2].index.tolist()
    valid_df = valid_df.loc[valid_df[group_col].isin(kept_groups)].copy()
    if valid_df.empty or len(kept_groups) < 2:
        return pd.DataFrame()

    for param_col in param_cols:
        sub = valid_df[[group_col, param_col]].dropna().copy()
        if sub.empty:
            continue
        stats = sub.groupby(group_col)[param_col].agg(["mean", "std", "count"]).reset_index()
        stats["std"] = stats["std"].fillna(0.0)
        if len(stats) < 2:
            continue

        within_std_mean = float(stats["std"].mean())
        within_std_median = float(stats["std"].median())
        max_group_mean = float(stats["mean"].max())
        min_group_mean = float(stats["mean"].min())
        between_mean_range = max_group_mean - min_group_mean
        between_mean_std = float(stats["mean"].std(ddof=0)) if len(stats) > 1 else 0.0
        pooled_std = float(np.sqrt(np.average(stats["std"] ** 2, weights=stats["count"].to_numpy(dtype=float)))) if stats["count"].sum() > 0 else float("nan")

        signal_to_dispersion = float(between_mean_range / within_std_mean) if within_std_mean > 0 else float("inf")
        signal_to_pooled = float(between_mean_range / pooled_std) if pooled_std > 0 else float("inf")

        top_groups = stats.sort_values("mean", ascending=False)[[group_col, "mean", "std", "count"]].head(3)
        low_groups = stats.sort_values("mean", ascending=True)[[group_col, "mean", "std", "count"]].head(3)

        rows.append({
            "grouping": group_col,
            "parameter": param_col.replace("matched_", ""),
            "n_groups": int(len(stats)),
            "n_cells": int(len(sub)),
            "within_std_mean": within_std_mean,
            "within_std_median": within_std_median,
            "between_mean_range": between_mean_range,
            "between_mean_std": between_mean_std,
            "pooled_within_std": pooled_std,
            "signal_to_dispersion": signal_to_dispersion,
            "signal_to_pooled": signal_to_pooled,
            "highest_groups": '; '.join([f"{r[1]}={r[2]:.3f}" for r in top_groups.itertuples()]),
            "lowest_groups": '; '.join([f"{r[1]}={r[2]:.3f}" for r in low_groups.itertuples()]),
        })
    return pd.DataFrame(rows)

analysis_df = tongji_map_df[["cell"] + matched_param_cols].copy()
parsed = analysis_df["cell"].apply(parse_tongji_cell_groups).apply(pd.Series)
analysis_df = pd.concat([analysis_df, parsed], axis=1)

summary_frames = []
for group_col in ["major_group", "temperature_group", "protocol_group"]:
    group_summary = summarize_group_stability(analysis_df, group_col, matched_param_cols)
    if not group_summary.empty:
        summary_frames.append(group_summary)

if not summary_frames:
    raise ValueError("No group-stability summaries could be computed; check whether the fitted Tongji cells populate multiple groups with at least two cells each.")

param_group_stability_df = pd.concat(summary_frames, ignore_index=True)

display(param_group_stability_df.sort_values(["grouping", "signal_to_dispersion"], ascending=[True, False]))

for grouping_name in ["major_group", "temperature_group", "protocol_group"]:
    sub = param_group_stability_df.loc[param_group_stability_df["grouping"].eq(grouping_name)].copy()
    if sub.empty:
        continue
    print(f"\nTop parameters for grouping = {grouping_name}")
    display(sub.sort_values("signal_to_dispersion", ascending=False)[[
        "parameter",
        "n_groups",
        "n_cells",
        "within_std_mean",
        "between_mean_range",
        "signal_to_dispersion",
        "signal_to_pooled",
    ]].head(10))

plot_group_order = ["major_group", "temperature_group", "protocol_group"]
fig, axes = plt.subplots(1, len(plot_group_order), figsize=(5.2 * len(plot_group_order), 5.0), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, grouping_name in zip(axes, plot_group_order):
    sub = param_group_stability_df.loc[param_group_stability_df["grouping"].eq(grouping_name)].copy()
    if sub.empty:
        ax.set_visible(False)
        continue
    sub = sub.sort_values("signal_to_dispersion", ascending=False)
    ax.barh(sub["parameter"], sub["signal_to_dispersion"], color="#356a9a", alpha=0.82)
    ax.invert_yaxis()
    ax.set_title(grouping_name.replace("_", " "))
    ax.set_xlabel("Between-group range / mean within-group std")
    ax.grid(axis="x", alpha=0.22)

plt.show()

fig, axes = plt.subplots(1, len(plot_group_order), figsize=(5.2 * len(plot_group_order), 5.0), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, grouping_name in zip(axes, plot_group_order):
    sub = param_group_stability_df.loc[param_group_stability_df["grouping"].eq(grouping_name)].copy()
    if sub.empty:
        ax.set_visible(False)
        continue
    ax.scatter(
        sub["within_std_mean"],
        sub["between_mean_range"],
        s=46,
        alpha=0.82,
        color="#b65f2a",
        edgecolors="none",
    )
    for row in sub.itertuples():
        ax.text(row.within_std_mean, row.between_mean_range, row.parameter, fontsize=8, alpha=0.9)
    ax.set_title(grouping_name.replace("_", " "))
    ax.set_xlabel("Mean within-group std")
    ax.set_ylabel("Between-group mean range")
    ax.grid(alpha=0.22)

plt.show()


## Module 8. Hierarchical Bayesian inference over discrete surrogate candidates

This module performs a discrete hierarchical Bayesian inference over the surrogate candidate library, using only Tongji2 and Tongji3 cells.

### 1. Goal

For each experimental Tongji degradation path $y_i$, we do not assume that a single deterministic inverse solution is trustworthy. Instead, we:

1. construct a discrete candidate set from the surrogate library;
2. assign each candidate a likelihood based on curve mismatch;
3. assign each candidate a hierarchical prior based on group structure in parameter space;
4. assign each candidate a manifold-support prior based on local surrogate density;
5. combine these terms into a posterior-like weight and select the highest-posterior candidate.

So the output is not simply the best-fit curve match. It is the surrogate parameter candidate with the strongest posterior support under the current hierarchical model.

### 2. Candidate set

For each Tongji cell $i$, let $\Theta_i^{(K)} = \{\theta_{i1}, \dots, \theta_{iK}\}$ be the top-$K$ surrogate candidates with the smallest curve-fit loss.

The curve-fit loss is:

$$
\mathcal{L}_{\mathrm{fit}}(\theta)
=
w_{\mathrm{spacing}}\,\mathcal{L}_{\mathrm{spacing}}(\theta)
+
w_{\mathrm{shape}}\,\mathcal{L}_{\mathrm{shape}}(\theta)
$$

where:

- $\mathcal{L}_{\mathrm{spacing}}$ is the mismatch of normalized cycle positions at common SOH levels;
- $\mathcal{L}_{\mathrm{shape}}$ is the mismatch of the four coarse slope segments
  $100\to95$, $95\to90$, $90\to85$, $85\to80$ SOH.

This step gives a discrete inverse-search space that already respects the observed degradation curve.

### 3. Likelihood term

Inside each cell's top-$K$ candidate pool, the fit loss is min-max normalized:

$$
\widetilde{\mathcal{L}}_{\mathrm{fit}}(\theta)
=
\frac{
\mathcal{L}_{\mathrm{fit}}(\theta)-\min \mathcal{L}_{\mathrm{fit}}
}{
\max \mathcal{L}_{\mathrm{fit}}-\min \mathcal{L}_{\mathrm{fit}}
}
$$

We then define a likelihood-like term through an exponential energy model:

$$
p(y_i \mid \theta_{ik})
\propto
\exp\big(
-\beta_{\mathrm{fit}}\,\widetilde{\mathcal{L}}_{\mathrm{fit}}(\theta_{ik})
\big)
$$

where $\beta_{\mathrm{fit}}$ controls how sharply the posterior favors better-fitting candidates.

### 4. Hierarchical prior

Each candidate is represented in standardized surrogate parameter space:

$$
z(\theta) = \frac{\theta - \mu_{\mathrm{sur}}}{\sigma_{\mathrm{sur}}}
$$

The hierarchy is:

$$
z_{i} \sim \mathcal{N}(\mu_{g(i)}, \Sigma_{\mathrm{global}})
$$

$$
\mu_{g} \sim \mathcal{N}(\mu_{m(g)}, \tau_{\mathrm{protocol}}^{-1} \Sigma_{\mathrm{global}})
$$

$$
\mu_{m} \sim \mathcal{N}(\mu_{0}, \tau_{\mathrm{major}}^{-1} \Sigma_{\mathrm{global}})
$$

where:

- $i$ indexes experimental cells;
- $g(i)$ is the protocol group of cell $i$;
- $m(g)$ is the major group (`Tongji2` or `Tongji3`);
- $\mu_0$ is the global center;
- $\Sigma_{\mathrm{global}}$ is the global diagonal variance estimate in standardized parameter space.

In the implementation, the protocol-group and major-group centers are updated by empirical-Bayes shrinkage:

$$
\mu_m
=
\frac{
n_m\,\bar z_m + \lambda_{\mathrm{major}}\,\mu_0
}{
n_m + \lambda_{\mathrm{major}}
}
$$

$$
\mu_g
=
\frac{
n_g\,\bar z_g + \lambda_{\mathrm{protocol}}\,\mu_{m(g)}
}{
n_g + \lambda_{\mathrm{protocol}}
}
$$

So small groups are pulled more strongly toward their parent level, while larger groups are allowed to stay more data-driven.

For candidate $\theta_{ik}$, the hierarchical prior penalty is the diagonal Mahalanobis distance to the selected group center:

$$
\mathcal{L}_{\mathrm{prior}}(\theta_{ik})
=
\frac{1}{P}
\sum_{p=1}^{P}
\frac{
\big(z_p(\theta_{ik}) - \mu_{g(i),p}\big)^2
}{
\sigma^2_{\mathrm{global},p}
}
$$

where $P$ is the number of surrogate parameters.

After min-max normalization within the top-$K$ pool:

$$
\widetilde{\mathcal{L}}_{\mathrm{prior}}(\theta_{ik})
=
\mathrm{MinMax}\left(
\mathcal{L}_{\mathrm{prior}}(\theta_{ik})
\right)
$$

the prior contribution is:

$$
p(\theta_{ik} \mid g(i), m(i))
\propto
\exp\big(
-\beta_{\mathrm{prior}}\,\widetilde{\mathcal{L}}_{\mathrm{prior}}(\theta_{ik})
\big)
$$

### 5. Surrogate-manifold density prior

To avoid selecting isolated edge candidates in parameter space, we define a manifold-support prior using the local $k$-nearest-neighbor distance inside the surrogate library.

For each surrogate candidate:

$$
\mathcal{L}_{\mathrm{density}}(\theta)
=
\frac{1}{k}
\sum_{j=1}^{k}
d\big(z(\theta), z(\theta_{j,\mathrm{NN}})\big)
$$

Smaller values mean the candidate lies in a denser and better-supported region of the surrogate manifold.

After min-max normalization inside the top-$K$ pool:

$$
\widetilde{\mathcal{L}}_{\mathrm{density}}(\theta_{ik})
=
\mathrm{MinMax}\left(
\mathcal{L}_{\mathrm{density}}(\theta_{ik})
\right)
$$

we define:

$$
p_{\mathrm{density}}(\theta_{ik})
\propto
\exp\big(
-\beta_{\mathrm{density}}\,\widetilde{\mathcal{L}}_{\mathrm{density}}(\theta_{ik})
\big)
$$

### 6. Posterior-like weight

The posterior-like candidate weight for cell $i$ and candidate $k$ is:

$$
p(\theta_{ik} \mid y_i, g(i), m(i))
\propto
p(y_i \mid \theta_{ik})
\cdot
p(\theta_{ik} \mid g(i), m(i))
\cdot
p_{\mathrm{density}}(\theta_{ik})
$$

which is equivalent to:

$$
\log p(\theta_{ik} \mid y_i, g(i), m(i))
=
-\beta_{\mathrm{fit}}\,\widetilde{\mathcal{L}}_{\mathrm{fit}}(\theta_{ik})
-\beta_{\mathrm{prior}}\,\widetilde{\mathcal{L}}_{\mathrm{prior}}(\theta_{ik})
-\beta_{\mathrm{density}}\,\widetilde{\mathcal{L}}_{\mathrm{density}}(\theta_{ik})
+ C
$$

Finally, the normalized posterior-like weights are obtained by softmax:

$$
w_{ik}
=
\frac{
\exp(\log p_{ik})
}{
\sum_{k'=1}^{K} \exp(\log p_{ik'})
}
$$

and the selected effective parameter is:

$$
\theta_i^{\star}
=
\arg\max_{\theta_{ik} \in \Theta_i^{(K)}} w_{ik}
$$

### 7. Empirical-Bayes update loop

The hierarchical centers are not fixed in advance. They are updated iteratively:

1. initialize candidate weights from fit and density terms only;
2. compute weighted cell-level parameter means;
3. update global, major-group, and protocol-group centers;
4. recompute the hierarchical prior penalties;
5. update posterior-like weights;
6. repeat for a fixed number of iterations.

This is an empirical-Bayes procedure because the group-level centers and global variances are estimated from the data instead of being fixed externally.

### 8. Interpretation

This module should be interpreted as **discrete hierarchical Bayesian inference over the surrogate candidate set**, not as full continuous-space Bayesian sampling.

What it gives us is:

- a likelihood-informed candidate pool;
- a group-structured prior that couples similar Tongji cells;
- a manifold-support prior that discourages unsupported edge solutions;
- posterior-like weights over multiple plausible inverse solutions.

That makes it more Bayesian than the earlier MAP-style re-ranking, while still remaining computationally compatible with the cached emulator workflow.


In [ ]:
if tongji_map_df.empty:
    raise ValueError("Baseline Tongji fits are required before Module 8 can run.")

from sklearn.neighbors import NearestNeighbors

matched_param_cols = [col for col in tongji_map_df.columns if col.startswith("matched_")]
if not matched_param_cols:
    raise ValueError("No matched surrogate parameter columns were found in tongji_map_df.")

MODULE8_TOPK = 64
HB_N_ITER = 8
HB_FIT_BETA = 18.0
HB_PRIOR_BETA = 10.0
HB_DENSITY_BETA = 2.0
HB_MAJOR_SHRINKAGE = 6.0
HB_PROTOCOL_SHRINKAGE = 4.0
HB_GLOBAL_VAR_FLOOR = 0.20
HB_DENSITY_K = 12

def format_crate_token(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token

def parse_tongji_cell_groups(cell_name):
    cell_name = str(cell_name)
    out = {
        "major_group": "Unknown",
        "temperature_group": "Unknown",
        "protocol_group": "Unknown",
    }
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major, middle, suffix = parts[:3]
            out["major_group"] = major
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                temp_group = f"{int(temp_token)}C" if str(temp_token).isdigit() else temp_token
                discharge_token = suffix.split("--")[0]
                rate_group = f"{format_crate_token(charge_token)}/{format_crate_token(discharge_token)}"
                out["temperature_group"] = temp_group
                out["protocol_group"] = f"{major} | {temp_group} | {rate_group}"
    except Exception:
        pass
    return out

def minmax_scale_array(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return x
    xmin = float(np.nanmin(x))
    xmax = float(np.nanmax(x))
    if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax - xmin < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - xmin) / (xmax - xmin)

def softmax_stable(logits):
    logits = np.asarray(logits, dtype=float)
    logits = logits - np.max(logits)
    exp_logits = np.exp(logits)
    denom = np.sum(exp_logits)
    if denom <= 0 or not np.isfinite(denom):
        return np.full_like(exp_logits, 1.0 / len(exp_logits))
    return exp_logits / denom

candidate_path_records = [rec for rec in tongji_path_records if parse_tongji_cell_groups(rec["cell"])["major_group"] in INCLUDED_TONGJI_MAJOR_GROUPS]
if not candidate_path_records:
    raise ValueError("No Tongji2/Tongji3 path records remain for hierarchical Bayesian inference.")

surrogate_param_mean = np.mean(X_expanded_multiplier, axis=0)
surrogate_param_std = np.std(X_expanded_multiplier, axis=0)
surrogate_param_std = np.where(surrogate_param_std < 1e-12, 1.0, surrogate_param_std)
X_expanded_param_z = (X_expanded_multiplier - surrogate_param_mean[None, :]) / surrogate_param_std[None, :]

knn_k = min(HB_DENSITY_K + 1, len(X_expanded_param_z))
if knn_k < 2:
    raise ValueError("Not enough surrogate candidates to compute the density prior.")
knn = NearestNeighbors(n_neighbors=knn_k)
knn.fit(X_expanded_param_z)
knn_distances, _ = knn.kneighbors(X_expanded_param_z)
surrogate_density_penalty = knn_distances[:, 1:].mean(axis=1)

required_surrogate_shape_levels = [95, 90, 85, 80]
missing_shape_levels = [level for level in required_surrogate_shape_levels if level not in level_to_sur_idx]
if missing_shape_levels:
    raise ValueError(f"Surrogate SOH grid is missing required shape levels: {missing_shape_levels}")
shape_step = 5.0

candidate_pools = []
for rec in candidate_path_records:
    cell_id = str(rec["cell"])
    group_info = parse_tongji_cell_groups(cell_id)

    exp_level_to_norm = dict(zip(rec["compare_levels"], rec["tongji_exp_space_norm_cycles"]))
    exp_level_to_norm[100] = 0.0

    common_levels = [level for level in rec["compare_levels"] if level in level_to_sur_idx]
    if len(common_levels) < 4 or 80 not in common_levels:
        continue
    if not all(level in exp_level_to_norm for level in SHAPE_SEGMENT_LEVELS):
        continue
    if not all(level in common_levels for level in required_surrogate_shape_levels):
        continue

    idxs = [level_to_sur_idx[level] for level in common_levels]
    sur_relative_cycles = Y_expanded_cycles[:, idxs]
    valid_sur = np.all(np.isfinite(sur_relative_cycles), axis=1) & np.all(sur_relative_cycles >= 0, axis=1)
    if not np.any(valid_sur):
        continue

    valid_indices = np.flatnonzero(valid_sur)
    surrogate_space_norm = sur_relative_cycles / surrogate_dataset_max_cycle
    exp_space_norm = np.asarray([exp_level_to_norm[level] for level in common_levels], dtype=float)

    spacing_sq = (surrogate_space_norm[valid_sur] - exp_space_norm[None, :]) ** 2
    curve_spacing_mse = np.mean(spacing_sq, axis=1)
    curve_spacing_rmse = np.sqrt(curve_spacing_mse)

    exp_shape_points = np.asarray([exp_level_to_norm[level] for level in SHAPE_SEGMENT_LEVELS], dtype=float)
    exp_shape_slopes = np.diff(exp_shape_points) / shape_step

    sur_shape_points_list = [np.zeros(len(valid_indices), dtype=float)]
    for level in required_surrogate_shape_levels:
        level_pos = common_levels.index(level)
        sur_shape_points_list.append(surrogate_space_norm[valid_sur][:, level_pos])
    sur_shape_points = np.column_stack(sur_shape_points_list)
    sur_shape_slopes = np.diff(sur_shape_points, axis=1) / shape_step
    slope_sq = (sur_shape_slopes - exp_shape_slopes[None, :]) ** 2
    shape_slope_mse = np.mean(slope_sq, axis=1)
    shape_slope_rmse = np.sqrt(shape_slope_mse)

    fit_loss = (
        CURVE_SPACING_LOSS_WEIGHT * curve_spacing_mse
        + SHAPE_SLOPE_LOSS_WEIGHT * shape_slope_mse
    )

    local_order = np.argsort(fit_loss)
    topk_local = local_order[: min(MODULE8_TOPK, len(local_order))]
    topk_global = valid_indices[topk_local]
    fit_loss_top = fit_loss[topk_local]
    density_top = surrogate_density_penalty[topk_global]
    fit_term = minmax_scale_array(fit_loss_top)
    density_term = minmax_scale_array(density_top)

    candidate_pools.append({
        "cell": cell_id,
        "group_info": group_info,
        "common_levels": common_levels,
        "exp_level_to_norm": exp_level_to_norm,
        "candidate_indices": topk_global,
        "surrogate_space_norm_top": surrogate_space_norm[topk_global],
        "param_raw_top": X_expanded_multiplier[topk_global],
        "param_z_top": X_expanded_param_z[topk_global],
        "fit_loss_top": fit_loss_top,
        "curve_spacing_rmse_top": curve_spacing_rmse[topk_local],
        "shape_slope_rmse_top": shape_slope_rmse[topk_local],
        "density_penalty_top": density_top,
        "fit_term": fit_term,
        "density_term": density_term,
        "posterior_weights": softmax_stable(-HB_FIT_BETA * fit_term - HB_DENSITY_BETA * density_term),
    })

if not candidate_pools:
    raise ValueError("No Tongji2/Tongji3 cells produced valid top-k candidate pools.")

group_counts_protocol = pd.Series([pool["group_info"]["protocol_group"] for pool in candidate_pools]).value_counts().to_dict()
group_counts_major = pd.Series([pool["group_info"]["major_group"] for pool in candidate_pools]).value_counts().to_dict()
param_dim = X_expanded_param_z.shape[1]

global_mean = np.mean([np.average(pool["param_z_top"], axis=0, weights=pool["posterior_weights"]) for pool in candidate_pools], axis=0)
global_var = np.ones(param_dim, dtype=float)

for _ in range(HB_N_ITER):
    cell_means = {}
    for pool in candidate_pools:
        weights = pool["posterior_weights"]
        cell_means[pool["cell"]] = np.average(pool["param_z_top"], axis=0, weights=weights)

    cell_mean_matrix = np.vstack([cell_means[pool["cell"]] for pool in candidate_pools])
    global_mean = np.mean(cell_mean_matrix, axis=0)

    weighted_var_accum = np.zeros(param_dim, dtype=float)
    weight_total = 0.0
    for pool in candidate_pools:
        weights = pool["posterior_weights"]
        diffs = pool["param_z_top"] - global_mean[None, :]
        weighted_var_accum += np.sum(weights[:, None] * (diffs ** 2), axis=0)
        weight_total += np.sum(weights)
    global_var = weighted_var_accum / max(weight_total, 1e-12)
    global_var = np.maximum(global_var, HB_GLOBAL_VAR_FLOOR)

    major_means = {}
    for major_group in sorted(group_counts_major):
        member_cells = [pool["cell"] for pool in candidate_pools if pool["group_info"]["major_group"] == major_group]
        member_matrix = np.vstack([cell_means[cell] for cell in member_cells])
        raw_mean = np.mean(member_matrix, axis=0)
        n_members = len(member_cells)
        major_means[major_group] = (n_members * raw_mean + HB_MAJOR_SHRINKAGE * global_mean) / (n_members + HB_MAJOR_SHRINKAGE)

    protocol_means = {}
    for protocol_group in sorted(group_counts_protocol):
        members = [pool for pool in candidate_pools if pool["group_info"]["protocol_group"] == protocol_group]
        member_cells = [pool["cell"] for pool in members]
        member_matrix = np.vstack([cell_means[cell] for cell in member_cells])
        raw_mean = np.mean(member_matrix, axis=0)
        major_group = members[0]["group_info"]["major_group"]
        major_mean = major_means[major_group]
        n_members = len(member_cells)
        protocol_means[protocol_group] = (n_members * raw_mean + HB_PROTOCOL_SHRINKAGE * major_mean) / (n_members + HB_PROTOCOL_SHRINKAGE)

    for pool in candidate_pools:
        protocol_group = pool["group_info"]["protocol_group"]
        major_group = pool["group_info"]["major_group"]
        if group_counts_protocol.get(protocol_group, 0) >= 2:
            center = protocol_means[protocol_group]
            prior_source = "protocol_group"
        elif group_counts_major.get(major_group, 0) >= 2:
            center = major_means[major_group]
            prior_source = "major_group"
        else:
            center = global_mean
            prior_source = "global"

        pool["hb_prior_center"] = center
        pool["hb_prior_source"] = prior_source
        diffs = pool["param_z_top"] - center[None, :]
        prior_mahal = np.mean((diffs ** 2) / global_var[None, :], axis=1)
        prior_term = minmax_scale_array(prior_mahal)
        pool["hb_prior_mahal"] = prior_mahal
        pool["hb_prior_term"] = prior_term
        log_posterior = (
            -HB_FIT_BETA * pool["fit_term"]
            -HB_PRIOR_BETA * pool["hb_prior_term"]
            -HB_DENSITY_BETA * pool["density_term"]
        )
        pool["log_posterior"] = log_posterior
        pool["posterior_weights"] = softmax_stable(log_posterior)

module8_rows = []
module8_long_rows = []
module8_candidate_rows = []

for pool in candidate_pools:
    best_local = int(np.argmax(pool["posterior_weights"]))
    best_global = int(pool["candidate_indices"][best_local])
    common_levels = pool["common_levels"]
    exp_level_to_norm = pool["exp_level_to_norm"]
    best_sur_norm_common = pool["surrogate_space_norm_top"][best_local]
    sur_level_to_norm = {level: float(val) for level, val in zip(common_levels, best_sur_norm_common)}
    sur_level_to_norm[100] = 0.0

    row = {
        "cell": pool["cell"],
        "anchor_soh": 100,
        "surrogate_start_soh": 100,
        "n_compare_levels": len(common_levels),
        "n_fit_levels": len(common_levels),
        "span_anchor_to_80": float(exp_level_to_norm[80] * tongji_dataset_max_cycle) if 80 in exp_level_to_norm else np.nan,
        "tongji_dataset_max_cycle": float(tongji_dataset_max_cycle),
        "surrogate_dataset_max_cycle": float(surrogate_dataset_max_cycle),
        "module8_fit_loss": float(pool["fit_loss_top"][best_local]),
        "module8_curve_spacing_rmse": float(pool["curve_spacing_rmse_top"][best_local]),
        "module8_shape_slope_rmse": float(pool["shape_slope_rmse_top"][best_local]),
        "module8_density_penalty": float(pool["density_penalty_top"][best_local]),
        "module8_hb_prior_mahal": float(pool["hb_prior_mahal"][best_local]),
        "module8_log_posterior": float(pool["log_posterior"][best_local]),
        "module8_posterior_weight": float(pool["posterior_weights"][best_local]),
        "module8_prior_source": str(pool["hb_prior_source"]),
        "best_surrogate_row": best_global,
    }
    for j, name in enumerate(param_names):
        row[f"matched_{name}"] = float(pool["param_raw_top"][best_local, j])

    plot_levels = [100] + [level for level in common_levels if level != 100]
    for level in plot_levels:
        exp_norm_val = float(exp_level_to_norm[level]) if level in exp_level_to_norm else 0.0
        sur_norm_val = float(sur_level_to_norm[level]) if level in sur_level_to_norm else 0.0
        if level in exp_level_to_norm:
            row[f"tongji_exp_space_norm_{level}"] = exp_norm_val
            row[f"tongji_norm_{level}"] = exp_norm_val
        if level in sur_level_to_norm:
            row[f"surrogate_space_norm_expSOH_{level}"] = sur_norm_val
            row[f"sur_norm_expSOH_{level}"] = sur_norm_val
            row[f"surrogate_level_for_expSOH_{level}"] = int(level)
        module8_long_rows.append({
            "cell": pool["cell"],
            "anchor_soh": 100,
            "SOH": int(level),
            "surrogate_SOH": int(level),
            "tongji_exp_space_norm_cycle": exp_norm_val,
            "surrogate_space_norm_cycle": sur_norm_val,
            "tongji_norm_cycle": exp_norm_val,
            "sur_norm_cycle": sur_norm_val,
            "tongji_plot_cycle": float(exp_norm_val * tongji_dataset_max_cycle + 1.0),
            "sur_plot_cycle": float(sur_norm_val * surrogate_dataset_max_cycle + 1.0),
            "is_anchor_point": bool(level == 100),
            "used_in_fit": bool(level in common_levels and level != 100),
            "module8_fit_loss": float(pool["fit_loss_top"][best_local]),
            "module8_curve_spacing_rmse": float(pool["curve_spacing_rmse_top"][best_local]),
            "module8_shape_slope_rmse": float(pool["shape_slope_rmse_top"][best_local]),
            "module8_density_penalty": float(pool["density_penalty_top"][best_local]),
            "module8_hb_prior_mahal": float(pool["hb_prior_mahal"][best_local]),
            "module8_log_posterior": float(pool["log_posterior"][best_local]),
            "module8_posterior_weight": float(pool["posterior_weights"][best_local]),
            "module8_prior_source": str(pool["hb_prior_source"]),
        })

    for local_rank in range(len(pool["candidate_indices"])):
        module8_candidate_rows.append({
            "cell": pool["cell"],
            "major_group": pool["group_info"]["major_group"],
            "protocol_group": pool["group_info"]["protocol_group"],
            "temperature_group": pool["group_info"]["temperature_group"],
            "candidate_rank_within_topk": int(local_rank + 1),
            "candidate_surrogate_row": int(pool["candidate_indices"][local_rank]),
            "fit_loss": float(pool["fit_loss_top"][local_rank]),
            "curve_spacing_rmse": float(pool["curve_spacing_rmse_top"][local_rank]),
            "shape_slope_rmse": float(pool["shape_slope_rmse_top"][local_rank]),
            "density_penalty": float(pool["density_penalty_top"][local_rank]),
            "fit_term": float(pool["fit_term"][local_rank]),
            "density_term": float(pool["density_term"][local_rank]),
            "hb_prior_mahal": float(pool["hb_prior_mahal"][local_rank]),
            "hb_prior_term": float(pool["hb_prior_term"][local_rank]),
            "log_posterior": float(pool["log_posterior"][local_rank]),
            "posterior_weight": float(pool["posterior_weights"][local_rank]),
            "prior_source": str(pool["hb_prior_source"]),
            "is_selected": bool(local_rank == best_local),
            "is_pure_fit_best": bool(local_rank == int(np.argmin(pool["fit_loss_top"]))),
        })

    module8_rows.append(row)

module8_hbi_df = pd.DataFrame(module8_rows)
module8_hbi_long_df = pd.DataFrame(module8_long_rows)
module8_candidate_df = pd.DataFrame(module8_candidate_rows)

if not module8_hbi_long_df.empty and "SOH" in module8_hbi_long_df.columns:
    module8_hbi_long_df.loc[module8_hbi_long_df["SOH"].eq(100), "used_in_fit"] = False

module8_map_df = module8_hbi_df.copy()
module8_map_long_df = module8_hbi_long_df.copy()

base_compare = tongji_map_df[["cell", "curve_spacing_rmse", "shape_slope_rmse", "match_loss_total"]].rename(columns={
    "curve_spacing_rmse": "module7_curve_spacing_rmse",
    "shape_slope_rmse": "module7_shape_slope_rmse",
    "match_loss_total": "module7_fit_loss",
})
module8_compare = module8_hbi_df[[
    "cell",
    "module8_curve_spacing_rmse",
    "module8_shape_slope_rmse",
    "module8_fit_loss",
    "module8_density_penalty",
    "module8_hb_prior_mahal",
    "module8_log_posterior",
    "module8_posterior_weight",
    "module8_prior_source",
]].copy()
module8_compare_df = base_compare.merge(module8_compare, on="cell", how="inner")
module8_compare_df["delta_curve_spacing_rmse"] = module8_compare_df["module8_curve_spacing_rmse"] - module8_compare_df["module7_curve_spacing_rmse"]
module8_compare_df["delta_shape_slope_rmse"] = module8_compare_df["module8_shape_slope_rmse"] - module8_compare_df["module7_shape_slope_rmse"]
module8_compare_df["delta_fit_loss"] = module8_compare_df["module8_fit_loss"] - module8_compare_df["module7_fit_loss"]

print(f"Module 8 hierarchical Bayes cells mapped: {len(module8_hbi_df)}")
print({
    "included_groups": INCLUDED_TONGJI_MAJOR_GROUPS,
    "topk": MODULE8_TOPK,
    "hb_n_iter": HB_N_ITER,
    "hb_fit_beta": HB_FIT_BETA,
    "hb_prior_beta": HB_PRIOR_BETA,
    "hb_density_beta": HB_DENSITY_BETA,
    "hb_density_k": knn_k - 1,
})
display(module8_compare_df[[
    "module7_curve_spacing_rmse",
    "module8_curve_spacing_rmse",
    "delta_curve_spacing_rmse",
    "module7_shape_slope_rmse",
    "module8_shape_slope_rmse",
    "delta_shape_slope_rmse",
    "module7_fit_loss",
    "module8_fit_loss",
    "delta_fit_loss",
]].describe().T)
display(module8_hbi_df[[
    "module8_curve_spacing_rmse",
    "module8_shape_slope_rmse",
    "module8_fit_loss",
    "module8_density_penalty",
    "module8_hb_prior_mahal",
    "module8_log_posterior",
    "module8_posterior_weight",
]].describe().T)


## Module 8A. All-cell hierarchical-Bayes fit overview in normalized cycle space

This view shows the Tongji2/Tongji3 cells together with the surrogate candidate selected by the hierarchical Bayesian posterior over the discrete top-$K$ candidate pool.


In [ ]:
if module8_hbi_long_df.empty:
    raise ValueError("No Module 8 hierarchical Bayes matches are available for the all-cell fit overview.")

all_curve_df = module8_hbi_long_df.copy()
all_curve_df["cell"] = all_curve_df["cell"].astype(str)
all_curve_df = all_curve_df.sort_values(["cell", "SOH"], ascending=[True, False]).reset_index(drop=True)
cell_order = sorted(all_curve_df["cell"].unique())
n_cells = len(cell_order)
n_cols = 4
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.1 * n_cols, 3.2 * n_rows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, cell_id in zip(axes, cell_order):
    sub = all_curve_df.loc[all_curve_df["cell"].eq(cell_id)].sort_values("SOH", ascending=False).copy()
    spacing_rmse_val = float(sub["module8_curve_spacing_rmse"].iloc[0]) if len(sub) else float("nan")
    slope_rmse_val = float(sub["module8_shape_slope_rmse"].iloc[0]) if len(sub) else float("nan")
    posterior_w = float(sub["module8_posterior_weight"].iloc[0]) if len(sub) else float("nan")
    prior_source = str(sub["module8_prior_source"].iloc[0]) if len(sub) else "na"

    if not sub.empty and 100 not in sub["SOH"].to_numpy(dtype=int):
        anchor_row = sub.iloc[[0]].copy()
        anchor_row["SOH"] = 100
        anchor_row["tongji_exp_space_norm_cycle"] = 0.0
        anchor_row["surrogate_space_norm_cycle"] = 0.0
        anchor_row["is_anchor_point"] = True
        anchor_row["used_in_fit"] = False
        sub = pd.concat([anchor_row, sub], ignore_index=True)
        sub = sub.sort_values("SOH", ascending=False)
    else:
        sub.loc[sub["SOH"].eq(100), "tongji_exp_space_norm_cycle"] = 0.0
        sub.loc[sub["SOH"].eq(100), "surrogate_space_norm_cycle"] = 0.0

    ax.plot(
        sub["tongji_exp_space_norm_cycle"],
        sub["SOH"],
        color="#2f6f9f",
        marker="o",
        markersize=3.6,
        linewidth=1.4,
        label="Exp" if cell_id == cell_order[0] else None,
    )
    ax.plot(
        sub["surrogate_space_norm_cycle"],
        sub["SOH"],
        color="#b65f2a",
        marker="s",
        markersize=3.2,
        linewidth=1.3,
        label="Hierarchical Bayes surrogate" if cell_id == cell_order[0] else None,
    )

    title_bits = [cell_id, f"post {posterior_w:.2f}", f"prior {prior_source}"]
    if np.isfinite(spacing_rmse_val):
        title_bits.append(f"spacing {spacing_rmse_val:.3f}")
    if np.isfinite(slope_rmse_val):
        title_bits.append(f"shape {slope_rmse_val:.3f}")
    ax.set_title(" | ".join(title_bits), fontsize=9)
    ax.set_xlabel("Normalized cycle")
    ax.set_ylabel("SOH (%)")
    ax.grid(alpha=0.22)
    ax.set_ylim(79, 101)
    ax.set_xlim(left=0.0)

for ax in axes[n_cells:]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
handles = [h for h, l in zip(handles, labels) if l]
labels = [l for l in labels if l]
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.01))

plt.show()


## Module 8B. Baseline-versus-hierarchical-Bayes parameter scatter by Tongji group

This view overlays the baseline Module 7 fit and the new hierarchical-Bayes fit for the same Tongji2/Tongji3 cells. Hollow markers and dashed connectors show the old solution; filled markers show the updated hierarchical-Bayes solution.


In [ ]:
if tongji_map_df.empty or module8_hbi_df.empty:
    raise ValueError("Both baseline and hierarchical Bayes fitted tables are required for Module 8B.")

matched_param_cols = [col for col in tongji_map_df.columns if col.startswith("matched_")]
if not matched_param_cols:
    raise ValueError("No matched surrogate parameter columns were found.")

def format_crate_token(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token

def parse_tongji_cell_groups(cell_name):
    cell_name = str(cell_name)
    out = {
        "major_group": "Unknown",
        "temperature_group": "Unknown",
        "protocol_group": "Unknown",
    }
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major = parts[0]
            middle = parts[1]
            suffix = parts[2]
            out["major_group"] = major
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                temp_group = f"{int(temp_token)}C" if str(temp_token).isdigit() else temp_token
                discharge_token = suffix.split("--")[0]
                rate_group = f"{format_crate_token(charge_token)}/{format_crate_token(discharge_token)}"
                out["temperature_group"] = temp_group
                out["protocol_group"] = f"{major} | {temp_group} | {rate_group}"
    except Exception:
        pass
    return out

def deterministic_spread_positions(group_labels, category_order, width=0.22):
    x = np.zeros(len(group_labels), dtype=float)
    for idx, cat in enumerate(category_order):
        member_idx = np.flatnonzero(np.asarray(group_labels) == cat)
        n = len(member_idx)
        if n == 0:
            continue
        offsets = np.array([0.0]) if n == 1 else np.linspace(-width, width, n)
        x[member_idx] = idx + offsets
    return x

baseline_df = tongji_map_df[["cell"] + matched_param_cols].copy()
hb_df = module8_hbi_df[["cell"] + matched_param_cols].copy()
plot_df = baseline_df.merge(hb_df, on="cell", how="inner", suffixes=("_m7", "_m8"))
parsed = plot_df["cell"].apply(parse_tongji_cell_groups).apply(pd.Series)
plot_df = pd.concat([plot_df, parsed], axis=1)

param_labels = [col.replace("matched_", "") for col in matched_param_cols]
n_params = len(matched_param_cols)
n_cols = 3
n_rows = int(np.ceil(n_params / n_cols))
major_order = [g for g in ["Tongji2", "Tongji3"] if g in set(plot_df["major_group"].astype(str))]

for major_group in major_order:
    sub_df = plot_df.loc[plot_df["major_group"].astype(str).eq(major_group)].copy()
    protocol_categories = sorted(sub_df["protocol_group"].dropna().astype(str).unique())
    if not protocol_categories:
        continue

    color_cycle = plt.cm.tab10(np.linspace(0, 1, max(len(protocol_categories), 3)))
    color_map = {cat: color_cycle[i] for i, cat in enumerate(protocol_categories)}
    base_x = deterministic_spread_positions(sub_df["protocol_group"].astype(str).to_numpy(), protocol_categories, width=0.20)
    x_old = base_x - 0.08
    x_new = base_x + 0.08

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.3 * n_cols, 3.2 * n_rows), constrained_layout=False)
    axes = np.atleast_1d(axes).ravel()

    for ax, param_col, param_label in zip(axes, matched_param_cols, param_labels):
        old_col = f"{param_col}_m7"
        new_col = f"{param_col}_m8"
        for row_idx in range(len(sub_df)):
            cat = str(sub_df["protocol_group"].iloc[row_idx])
            color = color_map[cat]
            y_old = float(sub_df[old_col].iloc[row_idx])
            y_new = float(sub_df[new_col].iloc[row_idx])
            ax.plot(
                [x_old[row_idx], x_new[row_idx]],
                [y_old, y_new],
                linestyle="--",
                linewidth=0.9,
                color=color,
                alpha=0.45,
            )
        for cat in protocol_categories:
            mask = sub_df["protocol_group"].astype(str).eq(cat).to_numpy()
            color = color_map[cat]
            ax.scatter(
                x_new[mask],
                sub_df.loc[mask, new_col].to_numpy(dtype=float),
                s=30,
                alpha=0.84,
                color=color,
                edgecolors="none",
                label=cat,
            )
            ax.scatter(
                x_old[mask],
                sub_df.loc[mask, old_col].to_numpy(dtype=float),
                s=28,
                alpha=0.78,
                facecolors="none",
                edgecolors=color,
                linewidths=1.0,
            )
        ax.set_title(param_label, fontsize=9)
        ax.set_xticks(range(len(protocol_categories)))
        ax.set_xticklabels(protocol_categories, rotation=28, ha="right")
        ax.set_ylabel("Multiplier")
        ax.grid(alpha=0.22)

    for ax in axes[n_params:]:
        ax.set_visible(False)

    protocol_handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", markersize=6, color=color_map[cat], label=cat)
        for cat in protocol_categories
    ]
    style_handles = [
        plt.Line2D([0], [0], marker="o", linestyle="--", markersize=6, markerfacecolor="none", markeredgecolor="black", color="black", label="Module 7 baseline"),
        plt.Line2D([0], [0], marker="o", linestyle="", markersize=6, markerfacecolor="black", markeredgecolor="black", color="black", label="Module 8 hierarchical Bayes"),
    ]
    fig.legend(
        handles=protocol_handles + style_handles,
        labels=protocol_categories + ["Module 7 baseline", "Module 8 hierarchical Bayes"],
        loc="center left",
        bbox_to_anchor=(0.995, 0.5),
        frameon=False,
        title="Protocol / style",
    )
    fig.suptitle(f"Baseline vs hierarchical-Bayes parameter multipliers for {major_group}", y=0.995, fontsize=12)
    fig.subplots_adjust(right=0.82, wspace=0.28, hspace=0.42, top=0.92, bottom=0.18)
    plt.show()


## Module 9. Condition-response analysis of inferred effective parameters

This module shifts the focus from trajectory fit quality to parameterization validity. The central question is no longer whether the surrogate can fit the degradation path, but whether the inferred effective parameters show organized and condition-aligned responses to the experimental cycling conditions.

The evidence logic is:

1. if an inferred parameter is meaningful, it should not vary arbitrarily across cells;
2. instead, it should show structured dependence on conditions such as temperature and protocol;
3. the strongest parameters are those that combine:
   - clear between-group separation,
   - directional response to numeric condition severity,
   - and relatively coherent behavior within the same condition group.

In this module we quantify condition response in two complementary ways:

- **group separability** using an ANOVA-style explained-variance score
  $$\eta^2 = \frac{SS_{\mathrm{between}}}{SS_{\mathrm{total}}}$$
  for temperature and protocol groups;
- **directional association** using Pearson correlation with numeric condition descriptors
  (temperature, charge rate, and discharge rate).

The goal is to identify which inferred effective parameters are most strongly aligned with real experimental conditions, and therefore provide the strongest support for the proposed parameterization.


In [ ]:
if module8_hbi_df.empty:
    raise ValueError("Module 8 hierarchical-Bayes results are required before running Module 9.")

matched_param_cols = [col for col in module8_hbi_df.columns if col.startswith("matched_")]
if not matched_param_cols:
    raise ValueError("No matched surrogate parameter columns were found in module8_hbi_df.")

def format_crate_token(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token

def parse_crate_numeric(token):
    token = str(token).strip()
    if token == "025":
        return 0.25
    if token == "05":
        return 0.5
    try:
        return float(token)
    except Exception:
        return np.nan

def parse_tongji_condition_fields(cell_name):
    cell_name = str(cell_name)
    out = {
        "major_group": "Unknown",
        "temperature_group": "Unknown",
        "protocol_group": "Unknown",
        "temperature_C": np.nan,
        "charge_rate_C": np.nan,
        "discharge_rate_C": np.nan,
    }
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major, middle, suffix = parts[:3]
            out["major_group"] = major
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                discharge_token = suffix.split("--")[0]
                temp_group = f"{int(temp_token)}C" if str(temp_token).isdigit() else temp_token
                charge_fmt = format_crate_token(charge_token)
                discharge_fmt = format_crate_token(discharge_token)
                out["temperature_group"] = temp_group
                out["protocol_group"] = f"{major} | {temp_group} | {charge_fmt}/{discharge_fmt}"
                out["temperature_C"] = float(temp_token) if str(temp_token).isdigit() else np.nan
                out["charge_rate_C"] = parse_crate_numeric(charge_token)
                out["discharge_rate_C"] = parse_crate_numeric(discharge_token)
    except Exception:
        pass
    return out

def safe_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    x = x[mask]
    y = y[mask]
    if np.nanstd(x) < 1e-12 or np.nanstd(y) < 1e-12:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def eta_squared(values, groups):
    values = np.asarray(values, dtype=float)
    groups = np.asarray(groups).astype(str)
    mask = np.isfinite(values)
    values = values[mask]
    groups = groups[mask]
    if len(values) < 3:
        return np.nan
    overall_mean = float(np.mean(values))
    ss_total = float(np.sum((values - overall_mean) ** 2))
    if ss_total < 1e-12:
        return 0.0
    ss_between = 0.0
    for g in np.unique(groups):
        grp_vals = values[groups == g]
        if len(grp_vals) == 0:
            continue
        grp_mean = float(np.mean(grp_vals))
        ss_between += len(grp_vals) * (grp_mean - overall_mean) ** 2
    return float(ss_between / ss_total)

analysis_df = module8_hbi_df[["cell"] + matched_param_cols].copy()
parsed = analysis_df["cell"].apply(parse_tongji_condition_fields).apply(pd.Series)
analysis_df = pd.concat([analysis_df, parsed], axis=1)

summary_rows = []
for param_col in matched_param_cols:
    vals = analysis_df[param_col].to_numpy(dtype=float)
    eta_temp = eta_squared(vals, analysis_df["temperature_group"])
    eta_protocol = eta_squared(vals, analysis_df["protocol_group"])
    corr_temp = safe_pearson(vals, analysis_df["temperature_C"])
    corr_charge = safe_pearson(vals, analysis_df["charge_rate_C"])
    corr_discharge = safe_pearson(vals, analysis_df["discharge_rate_C"])
    composite_score = np.nanmean([
        eta_temp,
        eta_protocol,
        abs(corr_temp) if np.isfinite(corr_temp) else np.nan,
        abs(corr_charge) if np.isfinite(corr_charge) else np.nan,
        abs(corr_discharge) if np.isfinite(corr_discharge) else np.nan,
    ])
    summary_rows.append({
        "parameter": param_col.replace("matched_", ""),
        "eta2_temperature": eta_temp,
        "eta2_protocol": eta_protocol,
        "corr_temperature": corr_temp,
        "corr_charge_rate": corr_charge,
        "corr_discharge_rate": corr_discharge,
        "abs_corr_temperature": abs(corr_temp) if np.isfinite(corr_temp) else np.nan,
        "abs_corr_charge_rate": abs(corr_charge) if np.isfinite(corr_charge) else np.nan,
        "abs_corr_discharge_rate": abs(corr_discharge) if np.isfinite(corr_discharge) else np.nan,
        "condition_response_score": composite_score,
    })

module9_condition_response_df = pd.DataFrame(summary_rows).sort_values(
    "condition_response_score", ascending=False
).reset_index(drop=True)

display(module9_condition_response_df)

heatmap_metrics = [
    "eta2_temperature",
    "eta2_protocol",
    "abs_corr_temperature",
    "abs_corr_charge_rate",
    "abs_corr_discharge_rate",
]
heatmap_labels = [
    "eta2 temp",
    "eta2 protocol",
    "|corr| temp",
    "|corr| charge",
    "|corr| discharge",
]
heatmap_data = module9_condition_response_df[heatmap_metrics].to_numpy(dtype=float)
heatmap_data = np.nan_to_num(heatmap_data, nan=0.0)

fig, ax = plt.subplots(figsize=(8.4, max(4.8, 0.5 * len(module9_condition_response_df))), constrained_layout=True)
im = ax.imshow(heatmap_data, aspect="auto", cmap="YlGnBu", vmin=0.0, vmax=max(0.35, float(np.nanmax(heatmap_data))))
ax.set_xticks(np.arange(len(heatmap_labels)))
ax.set_xticklabels(heatmap_labels, rotation=28, ha="right")
ax.set_yticks(np.arange(len(module9_condition_response_df)))
ax.set_yticklabels(module9_condition_response_df["parameter"].tolist())
ax.set_title("Condition-response strength of inferred effective parameters")
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        ax.text(j, i, f"{heatmap_data[i, j]:.2f}", ha="center", va="center", fontsize=7, color="black")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Response strength")
plt.show()

top_params = module9_condition_response_df["parameter"].head(4).tolist()
top_param_cols = []
for name in top_params:
    matches = [col for col in matched_param_cols if col.replace("matched_", "") == name]
    if matches:
        top_param_cols.append(matches[0])

if top_param_cols:
    n_rows = len(top_param_cols)
    fig, axes = plt.subplots(n_rows, 2, figsize=(12.8, 3.6 * n_rows), constrained_layout=True)
    axes = np.atleast_2d(axes)

    protocol_order = sorted(analysis_df["protocol_group"].dropna().astype(str).unique())
    protocol_color_cycle = plt.cm.tab10(np.linspace(0, 1, max(len(protocol_order), 3)))
    protocol_color_map = {cat: protocol_color_cycle[i] for i, cat in enumerate(protocol_order)}

    for row_idx, param_col in enumerate(top_param_cols):
        param_label = param_col.replace("matched_", "")

        ax_temp = axes[row_idx, 0]
        temp_order = sorted(analysis_df["temperature_C"].dropna().unique())
        for xi, temp_val in enumerate(temp_order):
            grp = analysis_df.loc[analysis_df["temperature_C"].eq(temp_val), param_col].dropna().to_numpy(dtype=float)
            if len(grp) == 0:
                continue
            offsets = np.array([0.0]) if len(grp) == 1 else np.linspace(-0.12, 0.12, len(grp))
            ax_temp.scatter(
                np.full(len(grp), xi) + offsets,
                grp,
                s=34,
                alpha=0.82,
                color="#356a9a",
                edgecolors="none",
            )
            ax_temp.plot([xi - 0.16, xi + 0.16], [np.median(grp), np.median(grp)], color="black", linewidth=1.2)
        ax_temp.set_xticks(range(len(temp_order)))
        ax_temp.set_xticklabels([f"{int(v)}C" for v in temp_order])
        ax_temp.set_ylabel("Multiplier")
        ax_temp.set_title(f"{param_label} vs temperature")
        ax_temp.grid(alpha=0.22)

        ax_protocol = axes[row_idx, 1]
        for xi, protocol in enumerate(protocol_order):
            grp = analysis_df.loc[analysis_df["protocol_group"].astype(str).eq(protocol), param_col].dropna().to_numpy(dtype=float)
            if len(grp) == 0:
                continue
            offsets = np.array([0.0]) if len(grp) == 1 else np.linspace(-0.12, 0.12, len(grp))
            ax_protocol.scatter(
                np.full(len(grp), xi) + offsets,
                grp,
                s=34,
                alpha=0.84,
                color=protocol_color_map[protocol],
                edgecolors="none",
            )
            ax_protocol.plot([xi - 0.16, xi + 0.16], [np.median(grp), np.median(grp)], color="black", linewidth=1.1)
        ax_protocol.set_xticks(range(len(protocol_order)))
        ax_protocol.set_xticklabels(protocol_order, rotation=28, ha="right")
        ax_protocol.set_ylabel("Multiplier")
        ax_protocol.set_title(f"{param_label} vs protocol")
        ax_protocol.grid(alpha=0.22)

    plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.4), constrained_layout=True)
numeric_specs = [
    ("temperature_C", "Temperature (C)", "corr_temperature"),
    ("charge_rate_C", "Charge rate (C)", "corr_charge_rate"),
    ("discharge_rate_C", "Discharge rate (C)", "corr_discharge_rate"),
]

highlight_params = module9_condition_response_df["parameter"].head(3).tolist()
highlight_cols = []
for name in highlight_params:
    matches = [col for col in matched_param_cols if col.replace("matched_", "") == name]
    if matches:
        highlight_cols.append(matches[0])

color_cycle = ["#2f6f9f", "#b65f2a", "#32745f"]
for ax, (xcol, xlabel, corr_col) in zip(axes, numeric_specs):
    x = analysis_df[xcol].to_numpy(dtype=float)
    for color, param_col in zip(color_cycle, highlight_cols):
        y = analysis_df[param_col].to_numpy(dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() < 3:
            continue
        corr_val = module9_condition_response_df.loc[
            module9_condition_response_df["parameter"].eq(param_col.replace("matched_", "")),
            corr_col,
        ].iloc[0]
        ax.scatter(x[mask], y[mask], s=34, alpha=0.75, color=color, edgecolors="none", label=f"{param_col.replace('matched_', '')} (r={corr_val:.2f})")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Multiplier")
    ax.set_title(f"Top parameter response to {xlabel.lower()}")
    ax.grid(alpha=0.22)
axes[0].legend(frameon=False, loc="best")
plt.show()


# Module 10. Temperature-informed and protocol-consensus hierarchical Bayesian inference inside the inverse mapping

This module replaces the previous post-hoc temperature regression. The key update is that temperature structure now participates directly in the surrogate-candidate inference, instead of being analyzed only after the parameters have already been selected. In addition, repeated cells within the same protocol group are softly encouraged to land in a more coherent parameter region.

### 1. What is updated relative to the old Module 10

Previously, Module 10 only took the parameters inferred by Module 8 and then fit temperature-response curves afterwards. That was useful for interpretation, but it did **not** influence which surrogate candidate was selected for each experimental cell.

Here we move the temperature information upstream into the inference itself:

1. Module 8 still provides a discrete top-$K$ candidate pool for each cell.
2. Module 10 keeps the same candidate pool, but re-scores each candidate with an additional temperature-structured prior.
3. The final selected candidate is therefore supported by:
   - curve-fit quality,
   - surrogate-manifold density,
   - hierarchical group shrinkage,
   - temperature-consistent effective-parameter behavior,
   - and protocol-level consensus among repeated cells.

So this module is no longer a pure after-the-fact interpretation step. It is a **temperature-informed inverse inference** step with an additional **protocol-consensus prior**.

### 1A. Constrained versus unconstrained parameters

In this updated version, the temperature prior is applied only to the two parameters for which the physical direction is both clearer and more useful in the present surrogate candidate pool:

- **SEI solvent diffusivity**, encouraged to become larger at higher temperature;
- **Lithium plating kinetic rate constant**, encouraged to become larger at lower temperature.

We do **not** impose temperature priors on:

- **Dead lithium decay constant**, because it still behaves more like an effective proxy parameter in the current inversion;
- **LAM-related parameters**, because their temperature trend is not yet separated robustly enough in the present surrogate manifold.

So Module 10 is intentionally selective: a small subset receives strong physics-aware temperature guidance, while the remaining parameters stay data-adaptive.

### 1B. Why protocol consensus is added

The temperature prior alone acts on each cell independently. That can improve physical directionality, but it can also create an undesirable pattern where one repeated cell in a protocol group moves a lot while the others barely move.

To reduce that instability, we add a protocol-level consensus prior in the same constrained subspace. Repeated cells under the same protocol are softly encouraged to choose candidates that remain close to the current protocol-group center for the constrained parameters.

This does **not** force all repeated cells to become identical. It simply discourages isolated jumps that enlarge within-group spread without improving the overall interpretation.

### 2. Candidate-space formulation

For each experimental cell $i$, let the candidate pool from Module 8 be

$$
\Theta_i^{(K)} = \{\theta_{i1}, \theta_{i2}, \dots, \theta_{iK}\}.
$$

Each candidate already has a curve-fit loss

$$
\mathcal{L}_{\mathrm{fit}}(\theta_{ik}),
$$

a density penalty

$$
\mathcal{L}_{\mathrm{dens}}(\theta_{ik}),
$$

and a hierarchical-group penalty

$$
\mathcal{L}_{\mathrm{hb}}(\theta_{ik}).
$$

We now add a temperature-consistency penalty

$$
\mathcal{L}_{T}(\theta_{ik} \mid T_i, \phi),
$$

where $T_i$ is the cell temperature and $\phi$ denotes the global temperature-response hyperparameters shared across cells.

We also add a protocol-consensus penalty

$$
\mathcal{L}_{\mathrm{proto}}(\theta_{ik} \mid g(i), \mu_{g(i)}),
$$

where $g(i)$ is the protocol group of cell $i$ and $\mu_{g(i)}$ is the current protocol-group center in the constrained parameter subspace.

### 3. Temperature coordinate and effective-parameter model

We use the same Arrhenius-style temperature coordinate

$$
x(T) = \frac{1}{T_{\mathrm{ref}}} - \frac{1}{T},
$$

with $T$ in Kelvin and $T_{\mathrm{ref}} = 25^\circ\mathrm{C}$.

For a temperature-sensitive effective parameter $\theta^{(m)}$, we model the log-multiplier of candidate $k$ for cell $i$ as

$$
z_{ik}^{(m)} = \log \theta_{ik}^{(m)}.
$$

Conditioned on temperature, we use a Gaussian prior

$$
z_{ik}^{(m)} \mid T_i
\sim
\mathcal{N}\left(\alpha_m + \beta_m x(T_i), \sigma_m^2\right).
$$

The slope and spread are not left completely free. Instead, each constrained parameter is regularized by:

- a **minimum slope magnitude** $\beta_{\min}$, preventing the temperature prior from collapsing to an almost flat line;
- a **maximum spread** $\sigma_{\max}$, preventing the prior from becoming so wide that it no longer meaningfully distinguishes candidates.

The sign of $\beta_m$ can encode qualitative physics:

- for SEI-related transport parameters expected to increase with temperature, $\beta_m \ge \beta_{\min} > 0$;
- for plating-related parameters expected to be stronger at lower temperature, $\beta_m \le -\beta_{\min} < 0$.

### 4. Joint scoring used for inverse inference

Inside each cell's candidate pool, each penalty is min-max normalized. The candidate score is then

$$
\mathcal{E}_{ik}
=
\lambda_{\mathrm{fit}} \widetilde{\mathcal{L}}_{\mathrm{fit},ik}
+
\lambda_{\mathrm{dens}} \widetilde{\mathcal{L}}_{\mathrm{dens},ik}
+
\lambda_{\mathrm{hb}} \widetilde{\mathcal{L}}_{\mathrm{hb},ik}
+
\lambda_{T} \widetilde{\mathcal{L}}_{T,ik}
+
\lambda_{\mathrm{proto}} \widetilde{\mathcal{L}}_{\mathrm{proto},ik}.
$$

The posterior-like weight becomes

$$
w_{ik}
=
\frac{\exp(-\mathcal{E}_{ik})}
{\sum_{j=1}^{K}\exp(-\mathcal{E}_{ij})}.
$$

The selected surrogate candidate is the one with the highest posterior-like weight.

### 5. Empirical-Bayes alternating update

The temperature hyperparameters $\phi = \{(\alpha_m,\beta_m,\sigma_m)\}_m$ are not fixed beforehand. Instead, we update them iteratively:

1. initialize from the Module 8 candidate ranking;
2. fit temperature-response priors from the currently selected candidates, but only for the constrained parameter subset, while enforcing slope floors and spread caps;
3. estimate protocol-group centers in the same constrained subspace;
4. recompute candidate posterior weights using both the updated temperature penalty and the protocol-consensus penalty;
5. repeat until the candidate selection stabilizes.

This is an empirical-Bayes approximation to a full hierarchical Bayesian model. It is not a full continuous posterior sampler over the surrogate parameter space, but it is already much closer to physics-aware inverse inference than a pure nearest-fit rule.

### 6. What to look for in the diagnostics

- **Module 10A** checks whether the new inference improves the fitted degradation trajectories in normalized cycle space.
- **Module 10B** checks how the inferred effective parameters shift relative to Module 8 and whether they become more organized by temperature/protocol group.
- **Module 10C** checks whether the surrogate-vs-experimental cycle positions move closer to the identity line, which is the most direct diagnostic of scale agreement.
- **Module 10D** checks whether the small or large changes are limited by candidate-pool spread or by weak re-ranking.


In [ ]:
if module8_candidate_df.empty or module8_hbi_df.empty:
    raise ValueError("Module 8 candidate pools and fitted results are required before running Module 10.")

TEMP_REF_C = 25.0
TEMP_REF_K = TEMP_REF_C + 273.15
MODULE10_N_ITER = 10
MODULE10_FIT_WEIGHT = 18.0
MODULE10_DENSITY_WEIGHT = 2.0
MODULE10_HB_WEIGHT = 10.0
MODULE10_TEMP_WEIGHT = 14.0
MODULE10_PROTOCOL_WEIGHT = 8.0
MODULE10_GROUP_SWEEPS = 4
MODULE10_SIGMA_FLOOR = 0.08
MODULE10_SIGMA_MAX = 0.22
MODULE10_RIDGE = 1e-4
MODULE10_PROTOCOL_VAR_FLOOR = 0.12
MODULE10_BETA_MIN_POS = 2.5
MODULE10_BETA_MIN_NEG = 0.8

def format_crate_token(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token

def parse_tongji_condition_fields(cell_name):
    cell_name = str(cell_name)
    out = {
        "major_group": "Unknown",
        "temperature_group": "Unknown",
        "protocol_group": "Unknown",
        "temperature_C": np.nan,
    }
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major, middle, suffix = parts[:3]
            out["major_group"] = major
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                discharge_token = suffix.split("--")[0]
                temp_group = f"{int(temp_token)}C" if str(temp_token).isdigit() else temp_token
                out["temperature_group"] = temp_group
                out["protocol_group"] = f"{major} | {temp_group} | {format_crate_token(charge_token)}/{format_crate_token(discharge_token)}"
                out["temperature_C"] = float(temp_token) if str(temp_token).isdigit() else np.nan
    except Exception:
        pass
    return out

def minmax_scale_array(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return x
    xmin = float(np.nanmin(x))
    xmax = float(np.nanmax(x))
    if (not np.isfinite(xmin)) or (not np.isfinite(xmax)) or abs(xmax - xmin) < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - xmin) / (xmax - xmin)

def softmax_stable(logits):
    logits = np.asarray(logits, dtype=float)
    logits = logits - np.nanmax(logits)
    exp_logits = np.exp(logits)
    denom = np.nansum(exp_logits)
    if (not np.isfinite(denom)) or denom <= 0:
        return np.full(len(logits), 1.0 / max(len(logits), 1))
    return exp_logits / denom

def fit_signed_temperature_prior(
    x,
    y,
    sign="none",
    beta_min=0.0,
    sigma_max=MODULE10_SIGMA_MAX,
    ridge=MODULE10_RIDGE,
    sigma_floor=MODULE10_SIGMA_FLOOR,
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 3:
        return {
            "alpha": float(np.nanmean(y)) if len(y) else 0.0,
            "beta": 0.0,
            "sigma": 1.0,
            "n_obs": int(len(x)),
        }

    X = np.column_stack([np.ones_like(x), x])
    XtX = X.T @ X + ridge * np.eye(2)
    Xty = X.T @ y
    coef = np.linalg.solve(XtX, Xty)
    alpha = float(coef[0])
    beta = float(coef[1])
    if sign == "positive":
        beta = max(beta, float(beta_min))
    elif sign == "negative":
        beta = min(beta, -float(beta_min))
    yhat = alpha + beta * x
    sigma = float(np.sqrt(np.mean((y - yhat) ** 2)))
    sigma = max(sigma, sigma_floor)
    sigma = min(sigma, sigma_max)
    return {
        "alpha": alpha,
        "beta": beta,
        "sigma": sigma,
        "n_obs": int(len(x)),
    }

def estimate_protocol_centers(selected_rows, active_specs, var_floor=MODULE10_PROTOCOL_VAR_FLOOR):
    selected_rows = selected_rows.copy()
    protocol_centers = {}
    global_center = {}
    global_scale = {}
    for spec in active_specs:
        col = f"log_cand_{spec['param_name']}"
        vals = selected_rows[col].to_numpy(dtype=float)
        global_center[col] = float(np.nanmean(vals))
        global_scale[col] = max(float(np.nanstd(vals, ddof=0)), var_floor)

    for protocol_group, sub in selected_rows.groupby("protocol_group", sort=True):
        protocol_centers[str(protocol_group)] = {}
        for spec in active_specs:
            col = f"log_cand_{spec['param_name']}"
            vals = sub[col].to_numpy(dtype=float)
            if len(vals) == 0 or not np.isfinite(vals).any():
                protocol_centers[str(protocol_group)][col] = global_center[col]
            else:
                protocol_centers[str(protocol_group)][col] = float(np.nanmean(vals))
    return protocol_centers, global_center, global_scale

def compute_temperature_penalty(candidate_frame, temp_prior_models, active_specs):
    penalty = np.zeros(len(candidate_frame), dtype=float)
    for spec in active_specs:
        model = temp_prior_models[spec["param_name"]]
        col = f"log_cand_{spec['param_name']}"
        pred = model["alpha"] + model["beta"] * candidate_frame["temp_arrhenius_x"].to_numpy(dtype=float)
        resid = (candidate_frame[col].to_numpy(dtype=float) - pred) / model["sigma"]
        penalty += spec["weight"] * (resid ** 2)
    return penalty

def compute_protocol_penalty_for_cell(cell_sub, peer_rows, active_specs, global_center, global_scale):
    penalty = np.zeros(len(cell_sub), dtype=float)
    for spec in active_specs:
        col = f"log_cand_{spec['param_name']}"
        if peer_rows is not None and len(peer_rows):
            center_val = float(np.nanmean(peer_rows[col].to_numpy(dtype=float)))
        else:
            center_val = global_center[col]
        resid = (cell_sub[col].to_numpy(dtype=float) - center_val) / global_scale[col]
        penalty += spec["weight"] * (resid ** 2)
    return penalty

candidate_df = module8_candidate_df.copy()
candidate_df["cell"] = candidate_df["cell"].astype(str)
parsed = candidate_df["cell"].apply(parse_tongji_condition_fields).apply(pd.Series)
for col in parsed.columns:
    candidate_df[col] = parsed[col].to_numpy()
candidate_df = candidate_df.loc[candidate_df["major_group"].astype(str).isin(INCLUDED_TONGJI_MAJOR_GROUPS)].copy()
candidate_df["temperature_K"] = candidate_df["temperature_C"] + 273.15
candidate_df["temp_arrhenius_x"] = (1.0 / TEMP_REF_K) - (1.0 / candidate_df["temperature_K"])

param_index = {name: idx for idx, name in enumerate(param_names)}

TEMP_PRIOR_SPECS = [
    {
        "param_name": "SEI solvent diffusivity [m2.s-1]",
        "sign": "positive",
        "weight": 1.8,
        "label": "SEI diffusivity",
        "beta_min": MODULE10_BETA_MIN_POS,
        "sigma_max": MODULE10_SIGMA_MAX,
    },
    {
        "param_name": "Lithium plating kinetic rate constant [m.s-1]",
        "sign": "negative",
        "weight": 2.2,
        "label": "Plating kinetic rate",
        "beta_min": MODULE10_BETA_MIN_NEG,
        "sigma_max": MODULE10_SIGMA_MAX,
    },
]

active_temp_specs = [spec for spec in TEMP_PRIOR_SPECS if spec["param_name"] in param_index]
if not active_temp_specs:
    raise ValueError("No temperature-prior parameters are available in the surrogate parameter list.")

for spec in active_temp_specs:
    pname = spec["param_name"]
    pidx = param_index[pname]
    col = f"cand_{pname}"
    candidate_df[col] = X_expanded_multiplier[candidate_df["candidate_surrogate_row"].to_numpy(dtype=int), pidx]
    candidate_df[f"log_{col}"] = np.log(np.clip(candidate_df[col].to_numpy(dtype=float), 1e-12, None))

candidate_df["fit_norm"] = candidate_df.groupby("cell")["fit_loss"].transform(minmax_scale_array)
candidate_df["density_norm"] = candidate_df.groupby("cell")["density_penalty"].transform(minmax_scale_array)
candidate_df["hb_norm"] = candidate_df.groupby("cell")["hb_prior_mahal"].transform(minmax_scale_array)

selected_by_cell = {}
for cell_id, sub in candidate_df.groupby("cell", sort=True):
    if "posterior_weight" in sub.columns and np.isfinite(sub["posterior_weight"]).any():
        best_idx = sub["posterior_weight"].astype(float).idxmax()
    elif "log_posterior" in sub.columns and np.isfinite(sub["log_posterior"]).any():
        best_idx = sub["log_posterior"].astype(float).idxmax()
    else:
        best_idx = sub["fit_loss"].astype(float).idxmin()
    selected_by_cell[cell_id] = int(best_idx)

module10_prior_rows = []
for iteration in range(MODULE10_N_ITER):
    selected_rows = candidate_df.loc[list(selected_by_cell.values())].copy()
    temp_prior_models = {}
    for spec in active_temp_specs:
        col = f"log_cand_{spec['param_name']}"
        fit_result = fit_signed_temperature_prior(
            selected_rows["temp_arrhenius_x"].to_numpy(dtype=float),
            selected_rows[col].to_numpy(dtype=float),
            sign=spec["sign"],
            beta_min=spec.get("beta_min", 0.0),
            sigma_max=spec.get("sigma_max", MODULE10_SIGMA_MAX),
        )
        fit_result["param_name"] = spec["param_name"]
        fit_result["label"] = spec["label"]
        fit_result["sign"] = spec["sign"]
        fit_result["weight"] = spec["weight"]
        fit_result["iteration"] = iteration + 1
        temp_prior_models[spec["param_name"]] = fit_result
        module10_prior_rows.append(fit_result.copy())

    protocol_centers, global_center, global_scale = estimate_protocol_centers(selected_rows, active_temp_specs)

    temp_penalty = compute_temperature_penalty(candidate_df, temp_prior_models, active_temp_specs)
    candidate_df["module10_temp_penalty_raw"] = temp_penalty
    candidate_df["module10_temp_penalty_norm"] = candidate_df.groupby("cell")["module10_temp_penalty_raw"].transform(minmax_scale_array)

    working_selected = dict(selected_by_cell)
    cell_order_by_group = {}
    for protocol_group, sub in candidate_df.groupby("protocol_group", sort=True):
        cell_order_by_group[str(protocol_group)] = sorted(sub["cell"].astype(str).unique())

    for _sweep in range(MODULE10_GROUP_SWEEPS):
        changed_in_sweep = False
        for protocol_group, cell_ids in cell_order_by_group.items():
            for cell_id in cell_ids:
                cell_sub = candidate_df.loc[candidate_df["cell"].astype(str).eq(cell_id)].copy()
                peer_indices = [
                    working_selected[peer_id]
                    for peer_id in cell_ids
                    if peer_id != cell_id and peer_id in working_selected
                ]
                peer_rows = candidate_df.loc[peer_indices].copy() if peer_indices else None
                proto_raw = compute_protocol_penalty_for_cell(cell_sub, peer_rows, active_temp_specs, global_center, global_scale)
                proto_norm = minmax_scale_array(proto_raw)
                total_energy_cell = (
                    MODULE10_FIT_WEIGHT * cell_sub["fit_norm"].to_numpy(dtype=float)
                    + MODULE10_DENSITY_WEIGHT * cell_sub["density_norm"].to_numpy(dtype=float)
                    + MODULE10_HB_WEIGHT * cell_sub["hb_norm"].to_numpy(dtype=float)
                    + MODULE10_TEMP_WEIGHT * cell_sub["module10_temp_penalty_norm"].to_numpy(dtype=float)
                    + MODULE10_PROTOCOL_WEIGHT * proto_norm
                )
                best_local = int(np.argmin(total_energy_cell))
                best_index = int(cell_sub.index[best_local])
                if working_selected.get(cell_id) != best_index:
                    changed_in_sweep = True
                    working_selected[cell_id] = best_index
        if not changed_in_sweep:
            break

    final_selected_rows = candidate_df.loc[list(working_selected.values())].copy()
    protocol_centers, global_center, global_scale = estimate_protocol_centers(final_selected_rows, active_temp_specs)

    protocol_penalty_raw_full = np.zeros(len(candidate_df), dtype=float)
    protocol_penalty_norm_full = np.zeros(len(candidate_df), dtype=float)
    total_energy_full = np.zeros(len(candidate_df), dtype=float)
    posterior_weights = np.zeros(len(candidate_df), dtype=float)
    new_selected = {}

    for protocol_group, cell_ids in cell_order_by_group.items():
        for cell_id in cell_ids:
            cell_sub = candidate_df.loc[candidate_df["cell"].astype(str).eq(cell_id)].copy()
            peer_indices = [
                working_selected[peer_id]
                for peer_id in cell_ids
                if peer_id != cell_id and peer_id in working_selected
            ]
            peer_rows = candidate_df.loc[peer_indices].copy() if peer_indices else None
            proto_raw = compute_protocol_penalty_for_cell(cell_sub, peer_rows, active_temp_specs, global_center, global_scale)
            proto_norm = minmax_scale_array(proto_raw)
            total_energy_cell = (
                MODULE10_FIT_WEIGHT * cell_sub["fit_norm"].to_numpy(dtype=float)
                + MODULE10_DENSITY_WEIGHT * cell_sub["density_norm"].to_numpy(dtype=float)
                + MODULE10_HB_WEIGHT * cell_sub["hb_norm"].to_numpy(dtype=float)
                + MODULE10_TEMP_WEIGHT * cell_sub["module10_temp_penalty_norm"].to_numpy(dtype=float)
                + MODULE10_PROTOCOL_WEIGHT * proto_norm
            )
            log_post_cell = -total_energy_cell
            weights = softmax_stable(log_post_cell)
            idxs = cell_sub.index.to_numpy(dtype=int)
            protocol_penalty_raw_full[idxs] = proto_raw
            protocol_penalty_norm_full[idxs] = proto_norm
            total_energy_full[idxs] = total_energy_cell
            posterior_weights[idxs] = weights
            new_selected[str(cell_id)] = int(idxs[int(np.argmax(weights))])

    candidate_df["module10_protocol_penalty_raw"] = protocol_penalty_raw_full
    candidate_df["module10_protocol_penalty_norm"] = protocol_penalty_norm_full
    candidate_df["module10_total_energy"] = total_energy_full
    candidate_df["module10_log_posterior"] = -total_energy_full
    candidate_df["module10_posterior_weight"] = posterior_weights

    if new_selected == selected_by_cell:
        selected_by_cell = new_selected
        break
    selected_by_cell = new_selected

module10_prior_df = pd.DataFrame(module10_prior_rows)

candidate_df["is_module10_selected"] = False
candidate_df.loc[list(selected_by_cell.values()), "is_module10_selected"] = True

cell_to_path = {str(rec["cell"]): rec for rec in tongji_path_records if parse_tongji_condition_fields(rec["cell"])["major_group"] in INCLUDED_TONGJI_MAJOR_GROUPS}

module10_rows = []
module10_long_rows = []
for cell_id, idx in selected_by_cell.items():
    rec = cell_to_path.get(str(cell_id))
    if rec is None:
        continue
    cand = candidate_df.loc[idx]
    surrogate_row = int(cand["candidate_surrogate_row"])
    exp_level_to_norm = dict(zip(rec["compare_levels"], rec["tongji_exp_space_norm_cycles"]))
    exp_level_to_norm[100] = 0.0
    common_levels = sorted([100] + [level for level in rec["compare_levels"] if level in level_to_sur_idx], reverse=True)
    sur_level_to_norm = {}
    for level in common_levels:
        if level == 100:
            sur_level_to_norm[level] = 0.0
        elif level in level_to_sur_idx:
            sur_level_to_norm[level] = float(Y_expanded_cycles[surrogate_row, level_to_sur_idx[level]] / surrogate_dataset_max_cycle)

    row = {
        "cell": str(cell_id),
        "anchor_soh": 100,
        "surrogate_start_soh": 100,
        "n_compare_levels": len(common_levels),
        "n_fit_levels": int(sum(level != 100 for level in common_levels)),
        "span_anchor_to_80": float(exp_level_to_norm[80] * tongji_dataset_max_cycle) if 80 in exp_level_to_norm else np.nan,
        "tongji_dataset_max_cycle": float(tongji_dataset_max_cycle),
        "surrogate_dataset_max_cycle": float(surrogate_dataset_max_cycle),
        "module10_fit_loss": float(cand["fit_loss"]),
        "module10_curve_spacing_rmse": float(cand["curve_spacing_rmse"]),
        "module10_shape_slope_rmse": float(cand["shape_slope_rmse"]),
        "module10_density_penalty": float(cand["density_penalty"]),
        "module10_hb_prior_mahal": float(cand["hb_prior_mahal"]),
        "module10_temp_penalty_raw": float(cand["module10_temp_penalty_raw"]),
        "module10_temp_penalty_norm": float(cand["module10_temp_penalty_norm"]),
        "module10_protocol_penalty_raw": float(cand["module10_protocol_penalty_raw"]),
        "module10_protocol_penalty_norm": float(cand["module10_protocol_penalty_norm"]),
        "module10_total_energy": float(cand["module10_total_energy"]),
        "module10_log_posterior": float(cand["module10_log_posterior"]),
        "module10_posterior_weight": float(cand["module10_posterior_weight"]),
        "module10_prior_source": "hierarchy+density+temperature+protocol",
        "best_surrogate_row": surrogate_row,
    }
    for name in param_names:
        row[f"matched_{name}"] = float(X_expanded_multiplier[surrogate_row, param_index[name]])

    for level in common_levels:
        exp_norm_val = float(exp_level_to_norm[level]) if level in exp_level_to_norm else np.nan
        sur_norm_val = float(sur_level_to_norm[level]) if level in sur_level_to_norm else np.nan
        row[f"tongji_exp_space_norm_{level}"] = exp_norm_val
        row[f"surrogate_space_norm_expSOH_{level}"] = sur_norm_val
        module10_long_rows.append({
            "cell": str(cell_id),
            "anchor_soh": 100,
            "SOH": int(level),
            "surrogate_SOH": int(level),
            "tongji_exp_space_norm_cycle": exp_norm_val,
            "surrogate_space_norm_cycle": sur_norm_val,
            "tongji_norm_cycle": exp_norm_val,
            "sur_norm_cycle": sur_norm_val,
            "tongji_plot_cycle": float(exp_norm_val * tongji_dataset_max_cycle + 1.0),
            "sur_plot_cycle": float(sur_norm_val * surrogate_dataset_max_cycle + 1.0),
            "is_anchor_point": bool(level == 100),
            "used_in_fit": bool(level != 100),
            "module10_fit_loss": float(cand["fit_loss"]),
            "module10_curve_spacing_rmse": float(cand["curve_spacing_rmse"]),
            "module10_shape_slope_rmse": float(cand["shape_slope_rmse"]),
            "module10_density_penalty": float(cand["density_penalty"]),
            "module10_hb_prior_mahal": float(cand["hb_prior_mahal"]),
            "module10_temp_penalty_raw": float(cand["module10_temp_penalty_raw"]),
            "module10_temp_penalty_norm": float(cand["module10_temp_penalty_norm"]),
            "module10_protocol_penalty_raw": float(cand["module10_protocol_penalty_raw"]),
            "module10_protocol_penalty_norm": float(cand["module10_protocol_penalty_norm"]),
            "module10_total_energy": float(cand["module10_total_energy"]),
            "module10_log_posterior": float(cand["module10_log_posterior"]),
            "module10_posterior_weight": float(cand["module10_posterior_weight"]),
            "module10_prior_source": "hierarchy+density+temperature+protocol",
        })
    module10_rows.append(row)

module10_hbi_df = pd.DataFrame(module10_rows)
module10_hbi_long_df = pd.DataFrame(module10_long_rows)
module10_candidate_df = candidate_df.copy()

module10_compare_df = module8_hbi_df[[
    "cell",
    "module8_curve_spacing_rmse",
    "module8_shape_slope_rmse",
    "module8_fit_loss",
    "module8_posterior_weight",
]].merge(
    module10_hbi_df[[
        "cell",
        "module10_curve_spacing_rmse",
        "module10_shape_slope_rmse",
        "module10_fit_loss",
        "module10_posterior_weight",
        "module10_temp_penalty_norm",
        "module10_protocol_penalty_norm",
    ]],
    on="cell",
    how="inner",
)
module10_compare_df["delta_curve_spacing_rmse"] = module10_compare_df["module10_curve_spacing_rmse"] - module10_compare_df["module8_curve_spacing_rmse"]
module10_compare_df["delta_shape_slope_rmse"] = module10_compare_df["module10_shape_slope_rmse"] - module10_compare_df["module8_shape_slope_rmse"]
module10_compare_df["delta_fit_loss"] = module10_compare_df["module10_fit_loss"] - module10_compare_df["module8_fit_loss"]

print(f"Module 10 temperature-informed cells mapped: {len(module10_hbi_df)}")
print({
    "n_iter": MODULE10_N_ITER,
    "fit_weight": MODULE10_FIT_WEIGHT,
    "density_weight": MODULE10_DENSITY_WEIGHT,
    "hb_weight": MODULE10_HB_WEIGHT,
    "temp_weight": MODULE10_TEMP_WEIGHT,
    "protocol_weight": MODULE10_PROTOCOL_WEIGHT,
    "active_temp_params": [spec["param_name"] for spec in active_temp_specs],
})
display(module10_compare_df[[
    "module8_curve_spacing_rmse",
    "module10_curve_spacing_rmse",
    "delta_curve_spacing_rmse",
    "module8_shape_slope_rmse",
    "module10_shape_slope_rmse",
    "delta_shape_slope_rmse",
    "module8_fit_loss",
    "module10_fit_loss",
    "delta_fit_loss",
    "module10_temp_penalty_norm",
    "module10_protocol_penalty_norm",
]].describe().T)

if not module10_prior_df.empty:
    prior_summary = (
        module10_prior_df
        .sort_values(["param_name", "iteration"])
        .groupby("param_name", as_index=False)
        .tail(1)
        [[ "param_name", "label", "sign", "alpha", "beta", "sigma", "n_obs", "iteration" ]]
        .reset_index(drop=True)
    )
    display(prior_summary)


## Module 10A. All-cell temperature-informed fit overview in normalized cycle space

This view uses exactly the same plotting format as Module 7A. The only change is the data source: the selected surrogate curves come from the Module 10 Bayesian/temperature-informed inference instead of the Module 7 likelihood-only matching.


In [ ]:
if module10_hbi_long_df.empty:
    raise ValueError("No Module 10 matches are available for the all-cell fit overview.")

all_curve_df = module10_hbi_long_df.copy()
all_curve_df["cell"] = all_curve_df["cell"].astype(str)
all_curve_df = all_curve_df.sort_values(["cell", "SOH"], ascending=[True, False]).reset_index(drop=True)
cell_order = sorted(all_curve_df["cell"].unique())
n_cells = len(cell_order)
n_cols = 4
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.1 * n_cols, 3.2 * n_rows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, cell_id in zip(axes, cell_order):
    sub = all_curve_df.loc[all_curve_df["cell"].eq(cell_id)].sort_values("SOH", ascending=False).copy()
    spacing_rmse_val = float(sub["module10_curve_spacing_rmse"].iloc[0]) if len(sub) else float("nan")
    slope_rmse_val = float(sub["module10_shape_slope_rmse"].iloc[0]) if len(sub) else float("nan")
    n_fit_levels = int(sub["used_in_fit"].sum()) if "used_in_fit" in sub.columns else 0

    if not sub.empty and 100 not in sub["SOH"].to_numpy(dtype=int):
        anchor_row = sub.iloc[[0]].copy()
        anchor_row["SOH"] = 100
        anchor_row["tongji_exp_space_norm_cycle"] = 0.0
        anchor_row["surrogate_space_norm_cycle"] = 0.0
        anchor_row["is_anchor_point"] = True
        anchor_row["used_in_fit"] = False
        sub = pd.concat([anchor_row, sub], ignore_index=True)
        sub = sub.sort_values("SOH", ascending=False)
    else:
        sub.loc[sub["SOH"].eq(100), "tongji_exp_space_norm_cycle"] = 0.0
        sub.loc[sub["SOH"].eq(100), "surrogate_space_norm_cycle"] = 0.0

    ax.plot(
        sub["tongji_exp_space_norm_cycle"],
        sub["SOH"],
        color="#2f6f9f",
        marker="o",
        markersize=3.6,
        linewidth=1.4,
        label="Exp" if cell_id == cell_order[0] else None,
    )
    ax.plot(
        sub["surrogate_space_norm_cycle"],
        sub["SOH"],
        color="#b65f2a",
        marker="s",
        markersize=3.2,
        linewidth=1.3,
        label="Surrogate" if cell_id == cell_order[0] else None,
    )

    title_bits = [cell_id, f"fit {n_fit_levels} lvls"]
    if np.isfinite(spacing_rmse_val):
        title_bits.append(f"spacing {spacing_rmse_val:.3f}")
    if np.isfinite(slope_rmse_val):
        title_bits.append(f"shape {slope_rmse_val:.3f}")
    ax.set_title(" | ".join(title_bits), fontsize=9)
    ax.set_xlabel("Normalized cycle")
    ax.set_ylabel("SOH (%)")
    ax.grid(alpha=0.22)
    ax.set_ylim(79, 101)
    ax.set_xlim(left=0.0)

for ax in axes[n_cells:]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
handles = [h for h, l in zip(handles, labels) if l]
labels = [l for l in labels if l]
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.01))

plt.show()


## Module 10B. Module 7 versus Module 10 parameter scatter by Tongji group

This view uses the non-Bayesian Module 7 mapping as the baseline. Hollow markers and dashed connectors show the Module 7 solution, while filled markers show the updated temperature-informed hierarchical-Bayesian Module 10 solution.


In [ ]:
if tongji_map_df.empty or module10_hbi_df.empty:
    raise ValueError("Both Module 7 and Module 10 fitted tables are required for Module 10B.")

matched_param_cols = [col for col in tongji_map_df.columns if col.startswith("matched_")]
if not matched_param_cols:
    raise ValueError("No matched surrogate parameter columns were found.")

def format_crate_token(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token

def parse_tongji_cell_groups(cell_name):
    cell_name = str(cell_name)
    out = {
        "major_group": "Unknown",
        "temperature_group": "Unknown",
        "protocol_group": "Unknown",
    }
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major, middle, suffix = parts[:3]
            out["major_group"] = major
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                temp_group = f"{int(temp_token)}C" if str(temp_token).isdigit() else temp_token
                discharge_token = suffix.split("--")[0]
                rate_group = f"{format_crate_token(charge_token)}/{format_crate_token(discharge_token)}"
                out["temperature_group"] = temp_group
                out["protocol_group"] = f"{major} | {temp_group} | {rate_group}"
    except Exception:
        pass
    return out

def deterministic_spread_positions(group_labels, category_order, width=0.22):
    x = np.zeros(len(group_labels), dtype=float)
    for idx, cat in enumerate(category_order):
        member_idx = np.flatnonzero(np.asarray(group_labels) == cat)
        n = len(member_idx)
        if n == 0:
            continue
        offsets = np.array([0.0]) if n == 1 else np.linspace(-width, width, n)
        x[member_idx] = idx + offsets
    return x

baseline_df = tongji_map_df[["cell"] + matched_param_cols].copy()
new_df = module10_hbi_df[["cell"] + matched_param_cols].copy()
plot_df = baseline_df.merge(new_df, on="cell", how="inner", suffixes=("_m7", "_m10"))
parsed = plot_df["cell"].apply(parse_tongji_cell_groups).apply(pd.Series)
plot_df = pd.concat([plot_df, parsed], axis=1)

param_labels = [col.replace("matched_", "") for col in matched_param_cols]
n_params = len(matched_param_cols)
n_cols = 3
n_rows = int(np.ceil(n_params / n_cols))
major_order = [g for g in ["Tongji2", "Tongji3"] if g in set(plot_df["major_group"].astype(str))]

for major_group in major_order:
    sub_df = plot_df.loc[plot_df["major_group"].astype(str).eq(major_group)].copy()
    protocol_categories = sorted(sub_df["protocol_group"].dropna().astype(str).unique())
    if not protocol_categories:
        continue

    color_cycle = plt.cm.tab10(np.linspace(0, 1, max(len(protocol_categories), 3)))
    color_map = {cat: color_cycle[i] for i, cat in enumerate(protocol_categories)}
    base_x = deterministic_spread_positions(sub_df["protocol_group"].astype(str).to_numpy(), protocol_categories, width=0.20)
    x_old = base_x - 0.08
    x_new = base_x + 0.08

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.3 * n_cols, 3.2 * n_rows), constrained_layout=False)
    axes = np.atleast_1d(axes).ravel()

    for ax, param_col, param_label in zip(axes, matched_param_cols, param_labels):
        old_col = f"{param_col}_m7"
        new_col = f"{param_col}_m10"
        for row_idx in range(len(sub_df)):
            cat = str(sub_df["protocol_group"].iloc[row_idx])
            color = color_map[cat]
            y_old = float(sub_df[old_col].iloc[row_idx])
            y_new = float(sub_df[new_col].iloc[row_idx])
            ax.plot([x_old[row_idx], x_new[row_idx]], [y_old, y_new], linestyle="--", linewidth=0.9, color=color, alpha=0.45)
        for cat in protocol_categories:
            mask = sub_df["protocol_group"].astype(str).eq(cat).to_numpy()
            color = color_map[cat]
            ax.scatter(x_new[mask], sub_df.loc[mask, new_col].to_numpy(dtype=float), s=30, alpha=0.84, color=color, edgecolors="none", label=cat)
            ax.scatter(x_old[mask], sub_df.loc[mask, old_col].to_numpy(dtype=float), s=28, alpha=0.78, facecolors="none", edgecolors=color, linewidths=1.0)
        ax.set_title(param_label, fontsize=9)
        ax.set_xticks(range(len(protocol_categories)))
        ax.set_xticklabels(protocol_categories, rotation=28, ha="right")
        ax.set_ylabel("Multiplier")
        ax.grid(alpha=0.22)

    for ax in axes[n_params:]:
        ax.set_visible(False)

    protocol_handles = [
        plt.Line2D([0], [0], marker="o", color="none", markerfacecolor=color_map[cat], markersize=6, label=cat)
        for cat in protocol_categories
    ]
    style_handles = [
        plt.Line2D([0], [0], marker="o", linestyle="--", color="#555555", markerfacecolor="white", markeredgecolor="#555555", markersize=6, label="Module 7"),
        plt.Line2D([0], [0], marker="o", linestyle="None", color="#555555", markerfacecolor="#555555", markeredgecolor="#555555", markersize=6, label="Module 10"),
    ]
    fig.legend(handles=style_handles + protocol_handles, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    fig.suptitle(f"{major_group}: Module 7 vs Module 10 effective parameters", y=0.995, fontsize=12)
    fig.tight_layout(rect=[0, 0, 0.86, 0.965])
    plt.show()


In [ ]:
if "module10_hbi_df" not in globals() or module10_hbi_df.empty:
    raise ValueError("Run Module 10 before plotting the Module 10 subgroup parameter summary.")

module10_param_cols = [col for col in module10_hbi_df.columns if col.startswith("matched_")]
if not module10_param_cols:
    raise ValueError("No Module 10 matched parameter columns were found.")

# Keep the requested 3 x 2 panel layout. If more parameters exist in the model,
# this panel shows the first six in the current surrogate parameter order.
module10_panel_param_cols = module10_param_cols[:6]
if len(module10_param_cols) > len(module10_panel_param_cols):
    print(f"Found {len(module10_param_cols)} parameters; plotting the first 6 for the requested 3 x 2 layout.")

def format_crate_token_module10b_summary(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token

def parse_tongji_groups_module10b_summary(cell_name):
    cell_name = str(cell_name)
    out = {
        "major_group": "Unknown",
        "subgroup": "Unknown",
        "temperature_C": np.nan,
        "charge_rate_C": np.nan,
        "discharge_rate_C": np.nan,
    }
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major, middle, suffix = parts[:3]
            out["major_group"] = major
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                discharge_token = suffix.split("--")[0]
                temp_group = f"{int(temp_token)}℃" if temp_token.isdigit() else temp_token
                rate_group = (
                    f"{format_crate_token_module10b_summary(charge_token)}/"
                    f"{format_crate_token_module10b_summary(discharge_token)}"
                )
                out["subgroup"] = f"{major} | {temp_group} | {rate_group}"
                out["temperature_C"] = float(temp_token) if temp_token.isdigit() else np.nan
                out["charge_rate_C"] = parse_crate_numeric_module10b(charge_token)
                out["discharge_rate_C"] = parse_crate_numeric_module10b(discharge_token)
    except Exception:
        pass
    return out

def parse_crate_numeric_module10b(token):
    token = str(token).strip()
    if token == "025":
        return 0.25
    if token == "05":
        return 0.5
    try:
        return float(token)
    except Exception:
        return np.nan

def deterministic_offsets_module10b(n, width=0.18):
    if n <= 1:
        return np.array([0.0])
    return np.linspace(-width, width, n)

module10_param_plot_df = module10_hbi_df[["cell"] + module10_panel_param_cols].copy()
parsed = module10_param_plot_df["cell"].apply(parse_tongji_groups_module10b_summary).apply(pd.Series)
module10_param_plot_df = pd.concat([module10_param_plot_df, parsed], axis=1)
module10_param_plot_df = module10_param_plot_df.loc[
    module10_param_plot_df["major_group"].astype(str).isin(["Tongji2", "Tongji3"])
].copy()
if module10_param_plot_df.empty:
    raise ValueError("No Tongji2/Tongji3 Module 10 parameter results are available.")

subgroup_meta_df = (
    module10_param_plot_df
    .groupby(["major_group", "subgroup"], as_index=False)
    .agg(
        temperature_C=("temperature_C", "mean"),
        charge_rate_C=("charge_rate_C", "mean"),
        discharge_rate_C=("discharge_rate_C", "mean"),
    )
)
tongji2_order = (
    subgroup_meta_df.loc[subgroup_meta_df["major_group"].eq("Tongji2")]
    .sort_values(["temperature_C", "charge_rate_C", "discharge_rate_C", "subgroup"])["subgroup"]
    .astype(str)
    .tolist()
)
tongji3_order = (
    subgroup_meta_df.loc[subgroup_meta_df["major_group"].eq("Tongji3")]
    .sort_values(["discharge_rate_C", "charge_rate_C", "temperature_C", "subgroup"])["subgroup"]
    .astype(str)
    .tolist()
)
subgroup_order = tongji2_order + tongji3_order
if len(subgroup_order) > 6:
    print(f"Found {len(subgroup_order)} subgroups; plotting the first 6 after major-group ordering.")
    subgroup_order = subgroup_order[:6]
module10_param_plot_df = module10_param_plot_df.loc[module10_param_plot_df["subgroup"].isin(subgroup_order)].copy()
tongji2_x = [i for i, subgroup in enumerate(subgroup_order) if subgroup in set(tongji2_order)]
tongji3_x = [i for i, subgroup in enumerate(subgroup_order) if subgroup in set(tongji3_order)]
major_boundary_x = (
    (max(tongji2_x) + min(tongji3_x)) / 2.0
    if tongji2_x and tongji3_x
    else None
)

# Match the Module 4 / Figure 2A subgroup palette exactly: Set2 indexed by subgroup_order.
base_cmap = plt.cm.get_cmap("Set2", max(len(subgroup_order), 1))
subgroup_color_map = {
    subgroup: base_cmap(idx % base_cmap.N)
    for idx, subgroup in enumerate(subgroup_order)
}
major_marker_map = {"Tongji2": "o", "Tongji3": "s"}

fig, axes = plt.subplots(2, 3, figsize=(14.5, 7.8), dpi=500, constrained_layout=True)
fig.set_constrained_layout_pads(hspace=0.12, wspace=0.06)
axes = axes.ravel()
x_positions = np.arange(len(subgroup_order), dtype=float)

for ax, param_col in zip(axes, module10_panel_param_cols):
    param_label = param_col.replace("matched_", "")
    if param_label == "Negative electrode critical stress [Pa]":
        param_label = "NE critical stress [Pa]"
    if param_label == "Negative electrode LAM constant proportional term [s-1]":
        param_label = "NE LAM constant proportional term [s-1]"

    for x_idx, subgroup in enumerate(subgroup_order):
        sub = module10_param_plot_df.loc[module10_param_plot_df["subgroup"].astype(str).eq(subgroup)].copy()
        if sub.empty:
            continue
        color = subgroup_color_map[subgroup]
        offsets = deterministic_offsets_module10b(len(sub), width=0.16)
        sub = sub.sort_values(["major_group", "cell"]).reset_index(drop=True)
        for offset, (_, point) in zip(offsets, sub.iterrows()):
            major = str(point["major_group"])
            ax.scatter(
                x_idx + offset,
                float(point[param_col]),
                s=80,
                marker=major_marker_map.get(major, "o"),
                facecolor=color,
                edgecolor="none",
                alpha=0.6,
                zorder=2,
            )

        mean_val = float(pd.to_numeric(sub[param_col], errors="coerce").mean())
        mean_major = str(sub["major_group"].mode().iloc[0]) if not sub["major_group"].mode().empty else "Tongji2"
        ax.scatter(
            x_idx,
            mean_val,
            s=120,
            marker=major_marker_map.get(mean_major, "o"),
            facecolor=color,
            edgecolor="black",
            linewidth=1.25,
            alpha=0.8,
            zorder=4,
        )

    if major_boundary_x is not None:
        ax.axvline(major_boundary_x, color="0.70", linestyle="--", linewidth=1.0, zorder=1)

    xaxis_trans = ax.get_xaxis_transform()
    if tongji2_x:
        ax.annotate(
            "",
            xy=(max(tongji2_x) + 0.28, 0.965),
            xytext=(min(tongji2_x) - 0.28, 0.965),
            xycoords=xaxis_trans,
            textcoords=xaxis_trans,
            arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5),
            clip_on=False,
        )
        ax.text(
            np.mean(tongji2_x),
            0.92,
            "temperature increase",
            transform=xaxis_trans,
            ha="center",
            va="top",
            fontsize=10,
            color="black",
        )
    if tongji3_x:
        ax.annotate(
            "",
            xy=(max(tongji3_x) + 0.28, 0.965),
            xytext=(min(tongji3_x) - 0.28, 0.965),
            xycoords=xaxis_trans,
            textcoords=xaxis_trans,
            arrowprops=dict(arrowstyle="->", color="black", linewidth=1.5),
            clip_on=False,
        )
        ax.text(
            np.mean(tongji3_x),
            0.92,
            "discharge rate increase",
            transform=xaxis_trans,
            ha="center",
            va="top",
            fontsize=10,
            color="black",
        )

    param_range = EXPANDED_MULTIPLIER_RANGES.get(param_col.replace("matched_", ""))
    if param_range is not None:
        tick_lo, tick_hi = [float(v) for v in param_range]
        tick_span = tick_hi - tick_lo
        if np.isfinite(tick_span) and tick_span > 0:
            ax.set_ylim(tick_lo - 0.1 * tick_span, tick_hi + 0.18 * tick_span)
            ax.set_yticks(np.linspace(tick_lo, tick_hi, 3))
    else:
        y_lo, y_hi = ax.get_ylim()
        y_span = y_hi - y_lo
        if np.isfinite(y_span) and y_span > 0:
            ax.set_ylim(y_lo - 0.1 * y_span, y_hi + 0.18 * y_span)
            tick_lo, tick_hi = ax.get_ylim()
            ax.set_yticks(np.linspace(tick_lo, tick_hi, 5))

    ax.set_title(param_label, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels([label.replace(" | ", "\n") for label in subgroup_order], rotation=0, ha="center", fontsize=8)
    ax.set_ylabel("Multiplier", fontsize=12)

for ax in axes[len(module10_panel_param_cols):]:
    ax.set_visible(False)

# subgroup_handles = [
#     plt.Line2D(
#         [0],
#         [0],
#         marker="o",
#         linestyle="",
#         markersize=7,
#         markerfacecolor=subgroup_color_map[subgroup],
#         markeredgecolor="none",
#         color=subgroup_color_map[subgroup],
#         label=subgroup,
#     )
#     for subgroup in subgroup_order
# ]
# marker_handles = [
#     plt.Line2D([0], [0], marker="o", linestyle="", markersize=7, color="black", markerfacecolor="white", label="Tongji2 cell"),
#     plt.Line2D([0], [0], marker="s", linestyle="", markersize=7, color="black", markerfacecolor="white", label="Tongji3 cell"),
#     plt.Line2D([0], [0], marker="o", linestyle="", markersize=9, color="black", markerfacecolor="white", label="Subgroup mean"),
# ]
# fig.legend(
#     handles=subgroup_handles + marker_handles,
#     loc="upper center",
#     bbox_to_anchor=(0.5, 1.04),
#     ncol=3,
#     frameon=False,
#     fontsize=8,
# )
plt.savefig(FIGURE_DIR / "F4.tiff", dpi=500, bbox_inches="tight")
plt.show()

module10_param_mean_df = (
    module10_param_plot_df
    .groupby(["subgroup", "major_group"], as_index=False)[module10_panel_param_cols]
    .mean()
)
display(module10_param_mean_df)


## Module 10C. Surrogate-versus-experimental normalized cycle scatter

This view uses exactly the same plotting format as the Module 7 normalized cycle scatter. It evaluates the Module 10 fitted cycle positions against the experimental cycle positions on the same normalized fitting scale.


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np
from matplotlib.colors import Normalize

# ---------- Data preparation ----------
if "module10_hbi_long_df" not in globals() or module10_hbi_long_df.empty:
    raise ValueError("Run Module 10 before plotting the Module 10 normalized cycle scatter.")

module10_scatter_df = module10_hbi_long_df.loc[module10_hbi_long_df["used_in_fit"]].copy()
required_scatter_cols = ["tongji_exp_space_norm_cycle", "surrogate_space_norm_cycle"]
missing_scatter_cols = [col for col in required_scatter_cols if col not in module10_scatter_df.columns]
if missing_scatter_cols:
    raise ValueError(f"Missing required columns for Module 10 scatter: {missing_scatter_cols}")

module10_scatter_df = module10_scatter_df.replace([np.inf, -np.inf], np.nan).dropna(subset=required_scatter_cols)
if module10_scatter_df.empty:
    raise ValueError("No finite fitting points are available for the Module 10 normalized cycle scatter.")

# These values are already normalized by the original fitting scale:
# Tongji cycles by tongji_dataset_max_cycle and surrogate cycles by surrogate_dataset_max_cycle.
# Therefore, do not renormalize by the local max; keeping this scale preserves the original upper range.
y_true_cycle = module10_scatter_df["tongji_exp_space_norm_cycle"].to_numpy(dtype=float).ravel()
y_pred_cycle = module10_scatter_df["surrogate_space_norm_cycle"].to_numpy(dtype=float).ravel()

y_true_norm = y_true_cycle
y_pred_norm = y_pred_cycle

rmse = np.sqrt(mean_squared_error(y_true_norm, y_pred_norm))
r2 = r2_score(y_true_norm, y_pred_norm)

# Compute errors (in units of 1e-3)
errors = (y_pred_norm - y_true_norm) * 1000

lo, hi = 0.0, 1.0
margin = 0.05

# ---------- Colormap: light red for small residuals, dark red for large residuals ----------
# Use Reds_r (reversed) so that low values → light, high values → dark
red_cmap = plt.cm.Reds_r
# Take a subset from 0.3 to 1.0 to avoid the very palest colours
red_cmap_subset = red_cmap.from_list(
    'dark_reds_subset',
    [red_cmap(0.0), red_cmap(0.4)],
    N=256
)

norm_scatter = Normalize(vmin=errors.min(), vmax=errors.max())

fig, ax = plt.subplots(figsize=(5, 4), dpi=500)

# ---------- Scatter plot with colour mapped to error ----------
ax.scatter(y_true_norm, y_pred_norm, s=40, alpha=0.8,
           edgecolor="none", c=errors, cmap=red_cmap_subset, norm=norm_scatter)
ax.plot([lo, hi], [lo, hi], color="black", linestyle="--", linewidth=1.1)

ax.set_xlabel("True cycle (normalized)", fontsize=12)
ax.set_ylabel("Emulator predicted cycle (normalized)", fontsize=12)
ax.set_xlim(lo - margin, hi + margin)
ax.set_ylim(lo - margin, hi + margin)
ax.set_aspect("equal", adjustable="box")

# ---------- Text box in the bottom-right corner ----------
ax.text(
    0.95,
    0.05,
    f"RMSE = {rmse:.4f}\n $R^2$ = {r2:.4f}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    linespacing=1.5,
    fontsize=12,
)

# ---------- Inset subplot in the top-left corner ----------
left, bottom = 0.15, 0.75
width, height = 0.30, 0.20

ax_inset = ax.inset_axes([left, bottom, width, height])

# Compute histogram counts and bin edges
counts, bin_edges = np.histogram(errors, bins=30)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]

# Normalise counts for colour mapping (0 to 1)
norm_counts = Normalize(vmin=counts.min(), vmax=counts.max())
counts_norm = norm_counts(counts)

# Plot each bar individually with colour mapped to its count
for center, count, c_norm in zip(bin_centers, counts, counts_norm):
    color = red_cmap_subset(c_norm)
    ax_inset.bar(center, count, width=bin_width, color=color, edgecolor='white', linewidth=0.3)

ax_inset.spines["top"].set_visible(False)
ax_inset.spines["right"].set_visible(False)
ax_inset.spines["left"].set_linewidth(0.8)
ax_inset.spines["bottom"].set_linewidth(0.8)

ax_inset.set_xlabel("Residual ($10^{-3}$)", fontsize=8)
ax_inset.set_xlim(-32, 32)
ax_inset.set_ylim(0, 50)
ax_inset.set_xticks([-30, -15, 0, 15, 30])
ax_inset.set_yticks([0, 25, 50])
ax_inset.set_xticklabels([-30, -15, 0, 15, 30], fontsize=8)
ax_inset.set_yticklabels([0, 25, 50], fontsize=8)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "F3b.tiff", dpi=500, format="tiff", bbox_inches="tight")
plt.show()

tongji_cycle_equiv_rmse = rmse * float(tongji_dataset_max_cycle)
surrogate_cycle_equiv_rmse = rmse * float(surrogate_dataset_max_cycle)

print(f"RMSE in normalized cycle: {rmse:.4f}")
print(f"Tongji-cycle-equivalent RMSE: {tongji_cycle_equiv_rmse:.2f} cycles")
print(f"Surrogate-cycle-equivalent RMSE: {surrogate_cycle_equiv_rmse:.2f} cycles")
print(
    "Normalized cycle range: "
    f"true [{np.nanmin(y_true_norm):.4f}, {np.nanmax(y_true_norm):.4f}], "
    f"pred [{np.nanmin(y_pred_norm):.4f}, {np.nanmax(y_pred_norm):.4f}]"
)
print(
    "Cycle normalization basis: "
    f"Tongji = {float(tongji_dataset_max_cycle):.2f} cycles, "
    f"surrogate = {float(surrogate_dataset_max_cycle):.2f} cycles"
)
print(f"n fitting points: {len(module10_scatter_df)}")


In [ ]:
# ---------- Data loading and preparation (Module 10 source) ----------
if "module10_hbi_df" not in globals() or module10_hbi_df.empty:
    raise ValueError("Run Module 10 before plotting subgroup best fits.")
if "module10_hbi_long_df" not in globals() or module10_hbi_long_df.empty:
    raise ValueError("Run Module 10 before plotting subgroup best fits.")

def format_crate_token_module10_best(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token

def parse_tongji_subgroup_module10_best(cell_name):
    cell_name = str(cell_name)
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major, middle, suffix = parts[:3]
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                discharge_token = suffix.split("--")[0]
                temp_group = f"{int(temp_token)}℃" if temp_token.isdigit() else temp_token
                rate_group = (
                    f"{format_crate_token_module10_best(charge_token)}/"
                    f"{format_crate_token_module10_best(discharge_token)}"
                )
                return f"{major} | {temp_group} | {rate_group}"
    except Exception:
        pass
    return "Unknown"

module10_best_summary_df = module10_hbi_df.copy()
module10_best_summary_df["subgroup"] = module10_best_summary_df["cell"].apply(parse_tongji_subgroup_module10_best)

# Local aliases keep the plotting block identical to the Module 7 best-fit style.
if "module10_curve_spacing_rmse" in module10_best_summary_df.columns:
    module10_best_summary_df["curve_spacing_rmse"] = module10_best_summary_df["module10_curve_spacing_rmse"]
if "module10_shape_slope_rmse" in module10_best_summary_df.columns:
    module10_best_summary_df["shape_slope_rmse"] = module10_best_summary_df["module10_shape_slope_rmse"]
if "module10_total_energy" in module10_best_summary_df.columns:
    module10_best_summary_df["match_loss_total"] = module10_best_summary_df["module10_total_energy"]
elif "module10_fit_loss" in module10_best_summary_df.columns:
    module10_best_summary_df["match_loss_total"] = module10_best_summary_df["module10_fit_loss"]

if "match_loss_total" in module10_best_summary_df.columns:
    best_metric_col = "match_loss_total"
elif "curve_spacing_rmse" in module10_best_summary_df.columns:
    best_metric_col = "curve_spacing_rmse"
else:
    raise ValueError("module10_hbi_df must contain module10_total_energy/module10_fit_loss or module10_curve_spacing_rmse.")

module10_best_summary_df = (
    module10_best_summary_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["subgroup", best_metric_col])
    .sort_values(["subgroup", best_metric_col, "cell"])
)
if module10_best_summary_df.empty:
    raise ValueError("No finite Module 10 subgroup fit metrics are available.")

module10_best_cells_df = (
    module10_best_summary_df
    .groupby("subgroup", as_index=False, sort=True)
    .head(1)
    .reset_index(drop=True)
)

if len(module10_best_cells_df) > 6:
    print(f"Found {len(module10_best_cells_df)} subgroups; plotting the first 6 in sorted subgroup order.")
module10_best_cells_df = module10_best_cells_df.head(6).copy()

# ---------- Plotting with per-subplot legends and manual x-ticks ----------
fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=500, constrained_layout=True)
fig.set_constrained_layout_pads(hspace=0.1)
axes = axes.ravel()

for i, (ax, row) in enumerate(zip(axes, module10_best_cells_df.itertuples(index=False))):
    cell = str(row.cell)
    subgroup = str(row.subgroup)
    sub_long = (
        module10_hbi_long_df
        .loc[module10_hbi_long_df["cell"].astype(str).eq(cell)]
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["SOH", "tongji_exp_space_norm_cycle", "surrogate_space_norm_cycle"])
        .sort_values("SOH", ascending=False)
    )
    if sub_long.empty:
        ax.text(0.5, 0.5, f"No curve data\n{subgroup}", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        continue

    # Plot experimental and surrogate curves
    ax.plot(
        sub_long["tongji_exp_space_norm_cycle"],
        sub_long["SOH"],
        color="#6f6f6f",
        alpha=0.8,
        linewidth=2.5,
        marker="^",
        markersize=10.0,
        markeredgewidth=0,
        label="TONGJI target",
    )
    ax.plot(
        sub_long["surrogate_space_norm_cycle"],
        sub_long["SOH"],
        color="#15517f",
        alpha=0.6,
        linewidth=2.5,
        marker="o",
        markersize=10,
        markeredgewidth=0,
        label="Best emulator fit",
    )

    # Text box with metrics (bottom-left)
    rmse_value = float(getattr(row, "curve_spacing_rmse")) if "curve_spacing_rmse" in module10_best_summary_df.columns else float("nan")
    err_text = [
        subgroup,
        f"RMSE: ${rmse_value*1000:.2f} \\times 10^{{-3}}$",
    ]

    ax.text(
        0.03,
        0.05,
        "\n".join(err_text),
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=12,
        linespacing=1.5,
    )

    # Subplot title and axis limits
    ax.set_xlim(0, 1.0)
    ax.set_ylim(79, 101)
    ax.set_yticks([80, 85, 90, 95, 100])

    if i == 0:
        ax.set_xticks([0.0, 0.2, 0.4])
        ax.set_xlim(-0.02, 0.42)
    elif i == 1:
        ax.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8])
        ax.set_xlim(-0.02, 0.82)
    elif i == 2:
        ax.set_xticks([0.0, 0.2, 0.4, 0.6])
        ax.set_xlim(-0.02, 0.62)
    else:
        ax.set_xticks([0.0, 0.2, 0.4])
        ax.set_xlim(-0.02, 0.42)

    # ---------- Legend in the upper-right corner of each subplot ----------
    ax.legend(loc="upper right", frameon=False, fontsize=12)

# Hide unused subplots if fewer than 6
for idx in range(len(module10_best_cells_df), len(axes)):
    axes[idx].set_axis_off()

# Set y-labels for the first column and x-labels for the bottom row
for idx, ax in enumerate(axes):
    if idx % 3 == 0:
        ax.set_ylabel("State of health (SOH%)", fontsize=12)
for ax in axes[-3:]:
    ax.set_xlabel("Normalized cycle", fontsize=12)

plt.savefig(FIGURE_DIR / "F3c.tiff", dpi=500, format="tiff", bbox_inches="tight")
plt.show()

# Display the selected best cells
module10_best_display_cols = [
    col for col in ["subgroup", "cell", best_metric_col, "match_loss_total", "curve_spacing_rmse", "shape_slope_rmse"]
    if col in module10_best_cells_df.columns
]
display(module10_best_cells_df[module10_best_display_cols])


## Module 12. Group-anchor posterior-manifold-constrained early-to-late prediction

This module evaluates whether PBS-inferred parameters have predictive value when only the early part of a degradation trajectory is visible. For each protocol group, one arbitrary repeated cell is used as an anchor and the remaining cells are treated as test cells.

The anchor is first fitted over its full 100-80% SOH trajectory in Module 10. Instead of collapsing the anchor candidate pool to a single mean parameter, Module 12 keeps the anchor top-k candidates as a local posterior manifold. For each test cell, PBS candidates are first restricted to the nearest fixed-size neighborhood of this anchor posterior manifold.

After this hard feasible-region filter, candidate selection uses the observed early-SOH segment, density regularization, temperature regularization, and an anchor-manifold penalty. The anchor penalty is window-dependent: shorter early windows rely more strongly on the anchor manifold, while longer early windows rely more on the target-cell trajectory.

The early-trajectory matching term includes absolute cycle spacing, segment-shape slope, and relative cycle spacing. The final parameter reported for parameterization is the MAP candidate with the lowest posterior energy. The prediction curve is a posterior-weighted average over the best local candidates, which reduces single-candidate instability when several parameters fit the early segment similarly.


In [ ]:
if module10_candidate_df.empty or module10_hbi_df.empty:
    raise ValueError("Module 10 outputs are required before running Module 12.")

MODULE12_MANIFOLD_TOPK = 12
MODULE12_ANCHOR_FILTER_TOPN = 128
MODULE12_ANCHOR_SCALE_FLOOR_FRACTION = 0.12
MODULE12_ABSOLUTE_FIT_WEIGHT = 1.00
MODULE12_SHAPE_FIT_WEIGHT = 1.00
MODULE12_RELATIVE_FIT_WEIGHT = 0.35
MODULE12_RELATIVE_EPS_CYCLE = 1.0
MODULE12_USE_GROUP_PENALTY_AFTER_FILTER = True
MODULE12_GROUP_WEIGHT_BY_WINDOW = {95: 0.60, 90: 0.35, 85: 0.15}
MODULE12_POSTERIOR_TOPK = 24
MODULE12_POSTERIOR_TEMPERATURE = 0.75
MODULE12_INCLUDE_MAJOR_GROUPS = INCLUDED_TONGJI_MAJOR_GROUPS
MODULE12_TAIL_WINDOWS = [95, 90, 85]
MODULE12_EVAL_NORM_BASIS = float(tongji_dataset_max_cycle)
MODULE12_TEMP_REF_K = float(globals().get("TEMP_REF_K", 25.0 + 273.15))
MODULE12_N_ANCHOR_SPLITS = 5


def parse_tongji_condition_fields_module12(cell_name):
    if "parse_tongji_condition_fields" in globals():
        return ensure_major_prefixed_protocol_group(parse_tongji_condition_fields(cell_name))
    cell_name = str(cell_name)
    out = {
        "major_group": "Unknown",
        "temperature_group": "Unknown",
        "protocol_group": "Unknown",
        "temperature_C": np.nan,
    }
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major, middle, suffix = parts[:3]
            out["major_group"] = major
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                discharge_token = suffix.split("--")[0]
                if temp_token.isdigit():
                    temp_group = f"{int(temp_token)}C"
                    out["temperature_C"] = float(temp_token)
                else:
                    temp_group = temp_token

                def _fmt_crate(token):
                    token = str(token).strip()
                    if token == "025":
                        return "0.25C"
                    if token == "05":
                        return "0.5C"
                    if token.isdigit():
                        return f"{int(token)}C"
                    return token

                out["temperature_group"] = temp_group
                out["protocol_group"] = f"{major} | {temp_group} | {_fmt_crate(charge_token)}/{_fmt_crate(discharge_token)}"
    except Exception:
        pass
    return out


def ensure_major_prefixed_protocol_group(cond):
    cond = dict(cond)
    major = str(cond.get("major_group", "Unknown"))
    protocol = str(cond.get("protocol_group", "Unknown"))
    if major != "Unknown" and not protocol.startswith(f"{major} | "):
        cond["protocol_group"] = f"{major} | {protocol}"
    return cond


def module12_softmax_from_energy(energy, temperature=1.0):
    energy = np.asarray(energy, dtype=float)
    if energy.size == 0:
        return energy
    temperature = max(float(temperature), 1e-8)
    logits = -(energy - np.nanmin(energy)) / temperature
    logits = logits - np.nanmax(logits)
    weights = np.exp(logits)
    denom = np.nansum(weights)
    if (not np.isfinite(denom)) or denom <= 0:
        return np.full(len(energy), 1.0 / max(len(energy), 1), dtype=float)
    return weights / denom


def minmax_scale_array_module12(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return x
    xmin = float(np.nanmin(x))
    xmax = float(np.nanmax(x))
    if (not np.isfinite(xmin)) or (not np.isfinite(xmax)) or abs(xmax - xmin) < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - xmin) / (xmax - xmin)


def compute_anchor_posterior_manifold_distance(candidate_param_matrix, anchor_param_matrix, anchor_scale):
    """Distance from each candidate to the nearest anchor posterior candidate."""
    candidate_param_matrix = np.asarray(candidate_param_matrix, dtype=float)
    anchor_param_matrix = np.asarray(anchor_param_matrix, dtype=float)
    anchor_scale = np.asarray(anchor_scale, dtype=float)
    anchor_scale = np.where(np.isfinite(anchor_scale) & (anchor_scale > 1e-12), anchor_scale, 1.0)
    scaled_candidates = candidate_param_matrix / anchor_scale[None, :]
    scaled_anchors = anchor_param_matrix / anchor_scale[None, :]
    diff = scaled_candidates[:, None, :] - scaled_anchors[None, :, :]
    dist = np.sqrt(np.mean(diff ** 2, axis=2))
    return np.nanmin(dist, axis=1)


matched_param_cols = [col for col in module10_hbi_df.columns if col.startswith("matched_")]
if not matched_param_cols:
    raise ValueError("No matched parameter columns were found in Module 10 outputs.")

module12_anchor_candidate_df = module10_candidate_df.copy()
module12_anchor_candidate_df["cell"] = module12_anchor_candidate_df["cell"].astype(str)
module12_selected_df = module10_hbi_df.copy()
module12_selected_df["cell"] = module12_selected_df["cell"].astype(str)

if "major_group" not in module12_selected_df.columns or "protocol_group" not in module12_selected_df.columns:
    parsed = module12_selected_df["cell"].apply(parse_tongji_condition_fields_module12).apply(pd.Series)
    for col in parsed.columns:
        module12_selected_df[col] = parsed[col].to_numpy()

module12_selected_df = module12_selected_df.loc[
    module12_selected_df["major_group"].astype(str).isin(MODULE12_INCLUDE_MAJOR_GROUPS)
].copy()
module12_anchor_candidate_df = module12_anchor_candidate_df.loc[
    module12_anchor_candidate_df["major_group"].astype(str).isin(MODULE12_INCLUDE_MAJOR_GROUPS)
].copy()

cell_to_path = {
    str(rec["cell"]): rec
    for rec in tongji_path_records
    if parse_tongji_condition_fields_module12(rec["cell"])["major_group"] in MODULE12_INCLUDE_MAJOR_GROUPS
}

global_param_mean = np.mean(X_expanded_multiplier, axis=0)
global_param_std = np.std(X_expanded_multiplier, axis=0)
global_param_std = np.where(global_param_std < 1e-12, 1.0, global_param_std)
param_name_to_idx = {name: idx for idx, name in enumerate(param_names)}

module12_active_temp_specs = list(active_temp_specs) if "active_temp_specs" in globals() else []
module12_temp_models = {}
if "module10_prior_df" in globals() and isinstance(module10_prior_df, pd.DataFrame) and not module10_prior_df.empty:
    latest_priors = (
        module10_prior_df
        .sort_values(["param_name", "iteration"])
        .groupby("param_name", as_index=False)
        .tail(1)
    )
    for row in latest_priors.itertuples(index=False):
        module12_temp_models[str(row.param_name)] = {
            "alpha": float(row.alpha),
            "beta": float(row.beta),
            "sigma": float(row.sigma),
        }

module12_density_penalty = (
    np.asarray(surrogate_density_penalty, dtype=float)
    if "surrogate_density_penalty" in globals()
    else np.zeros(len(X_expanded_multiplier), dtype=float)
)


def select_module12_anchor_cells(selected_df, n_splits=MODULE12_N_ANCHOR_SPLITS):
    """Build deterministic protocol-wise anchor splits shared by Modules 12 and 13."""
    anchor_rows = []
    selected_df = selected_df.copy()
    selected_df["cell"] = selected_df["cell"].astype(str)
    for protocol_group, sub in selected_df.groupby("protocol_group", sort=True):
        cells = sorted(sub["cell"].astype(str).unique())
        if len(cells) < 2:
            continue
        for split_idx in range(int(n_splits)):
            anchor_cell = cells[split_idx % len(cells)]
            test_cells = [cell for cell in cells if cell != anchor_cell]
            row0 = sub.loc[sub["cell"].astype(str).eq(anchor_cell)].iloc[0]
            anchor_rows.append({
                "anchor_split": int(split_idx),
                "protocol_group": str(protocol_group),
                "anchor_cell": str(anchor_cell),
                "major_group": str(row0["major_group"]),
                "temperature_group": str(row0["temperature_group"]),
                "n_group_cells": int(len(cells)),
                "n_test_cells": int(len(test_cells)),
                "anchor_is_unique_within_protocol": bool(split_idx < len(cells)),
                "anchor_best_surrogate_row": int(row0["best_surrogate_row"]),
                "anchor_module10_fit_loss": float(row0.get("module10_fit_loss", row0.get("fit_loss", np.nan))),
            })
    return pd.DataFrame(anchor_rows)


def build_surrogate_curve_matrix(row_indices, levels_desc):
    row_indices = np.asarray(row_indices, dtype=int)
    curves = []
    for level in levels_desc:
        level = int(level)
        if level == 100:
            curves.append(np.zeros(len(row_indices), dtype=float))
        else:
            curves.append(Y_expanded_cycles[row_indices, level_to_sur_idx[level]].astype(float))
    return np.column_stack(curves)


def compute_module12_temp_penalty(row_indices, temp_arrhenius_x):
    row_indices = np.asarray(row_indices, dtype=int)
    penalty = np.zeros(len(row_indices), dtype=float)
    for spec in module12_active_temp_specs:
        pname = spec["param_name"]
        if pname not in module12_temp_models or pname not in param_name_to_idx:
            continue
        pidx = param_name_to_idx[pname]
        raw_vals = np.clip(X_expanded_multiplier[row_indices, pidx].astype(float), 1e-12, None)
        log_vals = np.log(raw_vals)
        model = module12_temp_models[pname]
        sigma = max(float(model["sigma"]), 1e-8)
        pred = float(model["alpha"]) + float(model["beta"]) * float(temp_arrhenius_x)
        resid = (log_vals - pred) / sigma
        penalty += float(spec.get("weight", 1.0)) * (resid ** 2)
    return penalty


def summarize_prediction_points(scatter_df, group_cols=None, prefix=""):
    if group_cols is None:
        group_cols = []
    rows = []
    grouped = [((), scatter_df)] if not group_cols else scatter_df.groupby(group_cols, sort=True)
    for key, sub in grouped:
        if sub.empty:
            continue
        err_norm = sub["pred_norm_cycle"].to_numpy(dtype=float) - sub["true_norm_cycle"].to_numpy(dtype=float)
        err_raw = sub["pred_raw_cycle"].to_numpy(dtype=float) - sub["true_raw_cycle"].to_numpy(dtype=float)
        true_raw = sub["true_raw_cycle"].to_numpy(dtype=float)
        row = {
            f"{prefix}rmse norm": float(np.sqrt(np.mean(err_norm ** 2))),
            f"{prefix}mae norm": float(np.mean(np.abs(err_norm))),
            f"{prefix}rmse raw": float(np.sqrt(np.mean(err_raw ** 2))),
            f"{prefix}mae raw": float(np.mean(np.abs(err_raw))),
            f"{prefix}mape": float(np.mean(100.0 * np.abs(err_raw) / np.maximum(np.abs(true_raw), 1e-12))),
            "n_points": int(len(sub)),
        }
        if group_cols:
            if not isinstance(key, tuple):
                key = (key,)
            row.update(dict(zip(group_cols, key)))
        rows.append(row)
    return pd.DataFrame(rows)


anchor_rows = []
prediction_rows = []
prediction_long_rows = []
scatter_rows = []
compare_rows = []
window_summary_rows = []

print("Module 12 fit basis: independent early-stage inference with shared multi-anchor splits.")
print("Prediction windows: 95->80, 90->80, and 85->80.")
print("For each window, only SOH points strictly above the window start are used for fitting.")
print(f"Early-fit weights: absolute={MODULE12_ABSOLUTE_FIT_WEIGHT:.2f}, shape={MODULE12_SHAPE_FIT_WEIGHT:.2f}, relative={MODULE12_RELATIVE_FIT_WEIGHT:.2f}")
print(f"Anchor hard-filter: topN={MODULE12_ANCHOR_FILTER_TOPN}; soft group penalty after filter={MODULE12_USE_GROUP_PENALTY_AFTER_FILTER}")
print(f"Window-dependent anchor weights: {MODULE12_GROUP_WEIGHT_BY_WINDOW}")
print(f"Posterior-weighted prediction: topK={MODULE12_POSTERIOR_TOPK}, temperature={MODULE12_POSTERIOR_TEMPERATURE:.2f}")
print(f"Anchor splits: {MODULE12_N_ANCHOR_SPLITS}; Module 12 and Module 13 use the same anchor plan.")
print(f"Evaluation normalization basis is shared: Tongji dataset max cycle = {MODULE12_EVAL_NORM_BASIS:.2f}")

module12_anchor_split_plan_df = select_module12_anchor_cells(module12_selected_df, MODULE12_N_ANCHOR_SPLITS)
if module12_anchor_split_plan_df.empty:
    raise ValueError("No valid Module 12 anchor split plan could be built.")

for anchor_plan in module12_anchor_split_plan_df.itertuples(index=False):
    split_idx = int(anchor_plan.anchor_split)
    protocol_group = str(anchor_plan.protocol_group)
    anchor_cell = str(anchor_plan.anchor_cell)
    group_df = module12_selected_df.loc[module12_selected_df["protocol_group"].astype(str).eq(protocol_group)].copy()
    test_cells = [cell for cell in sorted(group_df["cell"].astype(str).unique()) if cell != anchor_cell]
    if not test_cells:
        continue

    anchor_candidates = module12_anchor_candidate_df.loc[module12_anchor_candidate_df["cell"].eq(anchor_cell)].copy()
    if anchor_candidates.empty:
        continue
    sort_col = "module10_total_energy" if "module10_total_energy" in anchor_candidates.columns else "fit_loss"
    anchor_candidates = anchor_candidates.sort_values(sort_col, ascending=True).head(MODULE12_MANIFOLD_TOPK).copy()
    if anchor_candidates.empty:
        continue

    anchor_sur_rows = anchor_candidates["candidate_surrogate_row"].to_numpy(dtype=int)
    anchor_param_matrix = X_expanded_multiplier[anchor_sur_rows]
    anchor_center = np.nanmean(anchor_param_matrix, axis=0)
    anchor_std = np.nanstd(anchor_param_matrix, axis=0)
    anchor_scale = np.maximum(anchor_std, MODULE12_ANCHOR_SCALE_FLOOR_FRACTION * global_param_std)
    anchor_rows.append(anchor_plan._asdict())

    for cell_id in test_cells:
        rec = cell_to_path.get(cell_id)
        if rec is None:
            continue

        exp_level_to_raw = dict(zip(rec["compare_levels"], rec["tongji_relative_cycles"]))
        exp_level_to_raw[100] = 0.0
        common_levels = sorted([100] + [level for level in rec["compare_levels"] if level in level_to_sur_idx], reverse=True)

        parsed_info = parse_tongji_condition_fields_module12(cell_id)
        temp_arrhenius_x = (1.0 / MODULE12_TEMP_REF_K) - (1.0 / (float(parsed_info["temperature_C"]) + 273.15))
        base_module10 = module12_selected_df.loc[module12_selected_df["cell"].eq(cell_id)]

        for window_start in MODULE12_TAIL_WINDOWS:
            train_levels = [level for level in common_levels if level >= window_start]
            pred_levels = [level for level in common_levels if window_start > level >= 80]
            if len(train_levels) < 2 or len(pred_levels) < 2:
                continue

            required_levels = sorted(set(train_levels + pred_levels), reverse=True)
            required_non_anchor_levels = [level for level in required_levels if level != 100]
            if not required_non_anchor_levels:
                continue

            required_idxs = [level_to_sur_idx[level] for level in required_non_anchor_levels]
            sur_cycles_needed = Y_expanded_cycles[:, required_idxs]
            valid_mask = np.all(np.isfinite(sur_cycles_needed), axis=1) & np.all(sur_cycles_needed >= 0, axis=1)
            valid_indices = np.flatnonzero(valid_mask)
            if len(valid_indices) == 0:
                continue

            n_candidates_before_filter = int(len(valid_indices))
            cand_param_matrix_all = X_expanded_multiplier[valid_indices]
            group_penalty_raw_all = compute_anchor_posterior_manifold_distance(
                cand_param_matrix_all,
                anchor_param_matrix,
                anchor_scale,
            )
            n_keep = min(n_candidates_before_filter, int(MODULE12_ANCHOR_FILTER_TOPN))
            anchor_distance_order = np.argsort(group_penalty_raw_all)
            keep_local = anchor_distance_order[:n_keep]
            valid_indices = valid_indices[keep_local]
            group_penalty_raw = group_penalty_raw_all[keep_local]
            anchor_filter_cutoff = float(np.nanmax(group_penalty_raw)) if len(group_penalty_raw) else np.nan
            anchor_rank_lookup = np.empty(n_candidates_before_filter, dtype=int)
            anchor_rank_lookup[anchor_distance_order] = np.arange(n_candidates_before_filter)
            selected_anchor_rank_local = anchor_rank_lookup[keep_local]

            sur_train_raw = build_surrogate_curve_matrix(valid_indices, train_levels)
            exp_train_raw = np.asarray([exp_level_to_raw[level] for level in train_levels], dtype=float)
            sur_train_norm = sur_train_raw / MODULE12_EVAL_NORM_BASIS
            exp_train_norm = exp_train_raw / MODULE12_EVAL_NORM_BASIS

            train_spacing_rmse = np.sqrt(np.mean((sur_train_norm - exp_train_norm[None, :]) ** 2, axis=1))
            relative_denom = np.maximum(np.abs(exp_train_raw[None, :]), MODULE12_RELATIVE_EPS_CYCLE)
            train_relative_rmse = np.sqrt(np.mean(((sur_train_raw - exp_train_raw[None, :]) / relative_denom) ** 2, axis=1))
            if len(train_levels) >= 2:
                sur_train_slope = np.diff(sur_train_norm, axis=1)
                exp_train_slope = np.diff(exp_train_norm)
                train_shape_rmse = np.sqrt(np.mean((sur_train_slope - exp_train_slope[None, :]) ** 2, axis=1))
            else:
                train_shape_rmse = np.zeros(len(valid_indices), dtype=float)
            train_fit_loss = (
                MODULE12_ABSOLUTE_FIT_WEIGHT * train_spacing_rmse
                + MODULE12_SHAPE_FIT_WEIGHT * train_shape_rmse
                + MODULE12_RELATIVE_FIT_WEIGHT * train_relative_rmse
            )

            cand_param_matrix = X_expanded_multiplier[valid_indices]
            group_penalty_norm = minmax_scale_array_module12(group_penalty_raw)
            density_raw = module12_density_penalty[valid_indices]
            density_norm = minmax_scale_array_module12(density_raw)
            temp_penalty_raw = compute_module12_temp_penalty(valid_indices, temp_arrhenius_x)
            temp_penalty_norm = minmax_scale_array_module12(temp_penalty_raw)
            fit_norm = minmax_scale_array_module12(train_fit_loss)

            group_weight = float(MODULE12_GROUP_WEIGHT_BY_WINDOW.get(int(window_start), 0.0))
            total_energy = (
                MODULE10_FIT_WEIGHT * fit_norm
                + MODULE10_DENSITY_WEIGHT * density_norm
                + MODULE10_TEMP_WEIGHT * temp_penalty_norm
                + group_weight * group_penalty_norm
            )
            best_local = int(np.argmin(total_energy))
            chosen_surrogate_row = int(valid_indices[best_local])

            posterior_topk = min(int(MODULE12_POSTERIOR_TOPK), len(valid_indices))
            posterior_local = np.argsort(total_energy)[:posterior_topk]
            posterior_weights = module12_softmax_from_energy(
                total_energy[posterior_local],
                temperature=MODULE12_POSTERIOR_TEMPERATURE,
            )
            posterior_rows = valid_indices[posterior_local]

            pred_level_to_raw = {100: 0.0}
            map_level_to_raw = {100: 0.0}
            for level in required_levels:
                if level == 100:
                    continue
                level_idx = level_to_sur_idx[level]
                pred_level_to_raw[level] = float(np.average(Y_expanded_cycles[posterior_rows, level_idx], weights=posterior_weights))
                map_level_to_raw[level] = float(Y_expanded_cycles[chosen_surrogate_row, level_idx])

            exp_pred_raw = np.asarray([exp_level_to_raw[level] for level in pred_levels], dtype=float)
            pred_pred_raw = np.asarray([pred_level_to_raw[level] for level in pred_levels], dtype=float)
            exp_pred_norm = exp_pred_raw / MODULE12_EVAL_NORM_BASIS
            pred_pred_norm = pred_pred_raw / MODULE12_EVAL_NORM_BASIS

            rmse_norm = float(np.sqrt(np.mean((pred_pred_norm - exp_pred_norm) ** 2)))
            mae_norm = float(np.mean(np.abs(pred_pred_norm - exp_pred_norm)))
            rmse_raw = float(np.sqrt(np.mean((pred_pred_raw - exp_pred_raw) ** 2)))
            mae_raw = float(np.mean(np.abs(pred_pred_raw - exp_pred_raw)))
            mape = float(np.mean(100.0 * np.abs(pred_pred_raw - exp_pred_raw) / np.maximum(np.abs(exp_pred_raw), 1e-12)))

            row = {
                "anchor_split": split_idx,
                "cell": cell_id,
                "protocol_group": protocol_group,
                "anchor_cell": anchor_cell,
                "window": f"{window_start}-80",
                "fit_basis": f"100_to_{window_start}",
                "best_surrogate_row": chosen_surrogate_row,
                "module12_total_energy": float(total_energy[best_local]),
                "module12_group_penalty_raw": float(group_penalty_raw[best_local]),
                "module12_group_penalty_norm": float(group_penalty_norm[best_local]),
                "module12_group_weight": float(group_weight),
                "module12_posterior_topk": int(posterior_topk),
                "module12_posterior_entropy": float(-np.sum(posterior_weights * np.log(np.maximum(posterior_weights, 1e-12)))),
                "module12_anchor_distance_rank": int(selected_anchor_rank_local[best_local] + 1),
                "module12_anchor_distance_percentile": float((selected_anchor_rank_local[best_local] + 1) / max(n_candidates_before_filter, 1)),
                "module12_anchor_filter_candidates_before": n_candidates_before_filter,
                "module12_anchor_filter_candidates_after": int(len(valid_indices)),
                "module12_anchor_filter_topn": int(MODULE12_ANCHOR_FILTER_TOPN),
                "module12_anchor_filter_cutoff": anchor_filter_cutoff,
                "module12_train_spacing_rmse": float(train_spacing_rmse[best_local]),
                "module12_train_shape_rmse": float(train_shape_rmse[best_local]),
                "module12_train_relative_rmse": float(train_relative_rmse[best_local]),
                "module12_density_penalty": float(density_raw[best_local]),
                "module12_temp_penalty_norm": float(temp_penalty_norm[best_local]),
                "module12_rmse_norm": rmse_norm,
                "module12_mae_norm": mae_norm,
                "module12_mape": mape,
                "module12_rmse_raw": rmse_raw,
                "module12_mae_raw": mae_raw,
                "n_train_levels": int(len(train_levels)),
                "n_pred_levels": int(len(pred_levels)),
            }
            for name in param_names:
                row[f"matched_{name}"] = float(X_expanded_multiplier[chosen_surrogate_row, param_name_to_idx[name]])
            prediction_rows.append(row)

            for name in param_names:
                m10_val = float(base_module10.iloc[0][f"matched_{name}"]) if len(base_module10) else np.nan
                compare_rows.append({
                    "anchor_split": split_idx,
                    "cell": cell_id,
                    "protocol_group": protocol_group,
                    "anchor_cell": anchor_cell,
                    "window": f"{window_start}-80",
                    "param_name": name,
                    "module10_value": m10_val,
                    "module12_value": float(row[f"matched_{name}"]),
                    "abs_delta": float(abs(row[f"matched_{name}"] - m10_val)) if np.isfinite(m10_val) else np.nan,
                    "rel_delta": float(abs(row[f"matched_{name}"] - m10_val) / max(abs(m10_val), 1e-12)) if np.isfinite(m10_val) else np.nan,
                })

            for level in train_levels + pred_levels:
                segment_role = "train" if level in train_levels else "pred"
                prediction_long_rows.append({
                    "anchor_split": split_idx,
                    "cell": cell_id,
                    "protocol_group": protocol_group,
                    "anchor_cell": anchor_cell,
                    "window": f"{window_start}-80",
                    "segment_role": segment_role,
                    "SOH": int(level),
                    "exp_norm_cycle": float(exp_level_to_raw[level] / MODULE12_EVAL_NORM_BASIS),
                    "pred_norm_cycle": float(pred_level_to_raw[level] / MODULE12_EVAL_NORM_BASIS),
                    "map_norm_cycle": float(map_level_to_raw[level] / MODULE12_EVAL_NORM_BASIS),
                    "exp_raw_cycle": float(exp_level_to_raw[level]),
                    "pred_raw_cycle": float(pred_level_to_raw[level]),
                    "map_raw_cycle": float(map_level_to_raw[level]),
                    "module12_rmse_norm": rmse_norm,
                    "module12_rmse_raw": rmse_raw,
                    "is_anchor_point": bool(level == 100),
                })

            window_summary_rows.append({
                "anchor_split": split_idx,
                "cell": cell_id,
                "protocol_group": protocol_group,
                "anchor_cell": anchor_cell,
                "window": f"{window_start}-80",
                "n_points": int(len(pred_levels)),
                "rmse_norm": rmse_norm,
                "mae_norm": mae_norm,
                "mape": mape,
                "rmse_raw": rmse_raw,
                "mae_raw": mae_raw,
            })

            for level in pred_levels:
                scatter_rows.append({
                    "anchor_split": split_idx,
                    "cell": cell_id,
                    "protocol_group": protocol_group,
                    "anchor_cell": anchor_cell,
                    "window": f"{window_start}-80",
                    "SOH": int(level),
                    "true_norm_cycle": float(exp_level_to_raw[level] / MODULE12_EVAL_NORM_BASIS),
                    "pred_norm_cycle": float(pred_level_to_raw[level] / MODULE12_EVAL_NORM_BASIS),
                    "map_norm_cycle": float(map_level_to_raw[level] / MODULE12_EVAL_NORM_BASIS),
                    "true_raw_cycle": float(exp_level_to_raw[level]),
                    "pred_raw_cycle": float(pred_level_to_raw[level]),
                    "map_raw_cycle": float(map_level_to_raw[level]),
                })

module12_anchor_df = pd.DataFrame(anchor_rows)
module12_prediction_df = pd.DataFrame(prediction_rows)
module12_prediction_long_df = pd.DataFrame(prediction_long_rows)
module12_scatter_df = pd.DataFrame(scatter_rows)
module12_param_compare_long_df = pd.DataFrame(compare_rows)
module12_window_scatter_df = module12_scatter_df.copy()
module12_window_summary_df = pd.DataFrame(window_summary_rows)

if module12_prediction_df.empty:
    raise ValueError("Module 12 produced no independent prediction results. Check group sizes, SOH coverage, or surrogate validity.")

module12_split_metric_df = summarize_prediction_points(module12_scatter_df, ["anchor_split"])
module12_error_summary_df = pd.DataFrame([{
    "scope": "five_anchor_split_mean",
    "n_anchor_splits": int(module12_split_metric_df["anchor_split"].nunique()),
    "rmse norm": float(module12_split_metric_df["rmse norm"].mean()),
    "rmse norm std": float(module12_split_metric_df["rmse norm"].std(ddof=0)),
    "mae norm": float(module12_split_metric_df["mae norm"].mean()),
    "mae norm std": float(module12_split_metric_df["mae norm"].std(ddof=0)),
    "rmse raw": float(module12_split_metric_df["rmse raw"].mean()),
    "rmse raw std": float(module12_split_metric_df["rmse raw"].std(ddof=0)),
    "mae raw": float(module12_split_metric_df["mae raw"].mean()),
    "mae raw std": float(module12_split_metric_df["mae raw"].std(ddof=0)),
    "mape": float(module12_split_metric_df["mape"].mean()),
    "mape std": float(module12_split_metric_df["mape"].std(ddof=0)),
    "n_points": int(module12_scatter_df.shape[0]),
}])

module12_window_split_metric_df = summarize_prediction_points(module12_scatter_df, ["window", "anchor_split"])
module12_window_overall_df = (
    module12_window_split_metric_df.groupby("window", as_index=False)
    .agg(
        n_splits=("anchor_split", "nunique"),
        **{
            "rmse norm": ("rmse norm", "mean"),
            "rmse norm std": ("rmse norm", lambda s: float(np.std(s, ddof=0))),
            "mae norm": ("mae norm", "mean"),
            "mae norm std": ("mae norm", lambda s: float(np.std(s, ddof=0))),
            "rmse raw": ("rmse raw", "mean"),
            "rmse raw std": ("rmse raw", lambda s: float(np.std(s, ddof=0))),
            "mae raw": ("mae raw", "mean"),
            "mae raw std": ("mae raw", lambda s: float(np.std(s, ddof=0))),
            "mape": ("mape", "mean"),
            "mape std": ("mape", lambda s: float(np.std(s, ddof=0))),
        }
    )
)

module12_group_summary_df = module12_prediction_df.groupby(["anchor_split", "protocol_group", "anchor_cell", "window"], as_index=False).agg(
    n_test_cells=("cell", "count"),
    mean_rmse_norm=("module12_rmse_norm", "mean"),
    mean_rmse_raw=("module12_rmse_raw", "mean"),
    mean_group_penalty=("module12_group_penalty_raw", "mean"),
)

display(module12_anchor_split_plan_df)
display(module12_error_summary_df)
display(module12_window_overall_df)
display(module12_group_summary_df.head(20))
display(
    module12_prediction_df[[
        "anchor_split",
        "window",
        "fit_basis",
        "n_train_levels",
        "n_pred_levels",
        "module12_rmse_norm",
        "module12_mae_norm",
        "module12_rmse_raw",
        "module12_mae_raw",
        "module12_mape",
        "module12_group_penalty_raw",
        "module12_total_energy",
    ]]
    .rename(columns={
        "module12_rmse_norm": "rmse norm",
        "module12_mae_norm": "mae norm",
        "module12_rmse_raw": "rmse raw",
        "module12_mae_raw": "mae raw",
        "module12_mape": "mape",
    })
    .describe(include="all").T
)

# Use split 0 for detailed curve overlays to keep the plot readable; aggregate metrics above use all five splits.
module12_plot_split = 0
plot_scatter_df = module12_window_scatter_df.loc[module12_window_scatter_df["anchor_split"].eq(module12_plot_split)].copy()
fig, axes = plt.subplots(len(MODULE12_TAIL_WINDOWS), 2, figsize=(11.2, 4.0 * len(MODULE12_TAIL_WINDOWS)), constrained_layout=True)
axes = np.atleast_2d(axes)
for row_idx, window_start in enumerate(MODULE12_TAIL_WINDOWS):
    window_label = f"{window_start}-80"
    sub = plot_scatter_df.loc[plot_scatter_df["window"].eq(window_label)].copy()
    if sub.empty:
        axes[row_idx, 0].set_visible(False)
        axes[row_idx, 1].set_visible(False)
        continue

    lo_norm = float(min(sub["true_norm_cycle"].min(), sub["pred_norm_cycle"].min()))
    hi_norm = float(max(sub["true_norm_cycle"].max(), sub["pred_norm_cycle"].max()))
    axes[row_idx, 0].plot([lo_norm, hi_norm], [lo_norm, hi_norm], linestyle="--", color="black", linewidth=1.0)
    axes[row_idx, 0].scatter(sub["true_norm_cycle"], sub["pred_norm_cycle"], s=24, alpha=0.76, edgecolors="none", color="#2f6f9f")
    axes[row_idx, 0].set_title(f"Split {module12_plot_split} | {window_label} normalized")
    axes[row_idx, 0].set_xlabel("True normalized cycle")
    axes[row_idx, 0].set_ylabel("Predicted normalized cycle")
    axes[row_idx, 0].grid(alpha=0.22)

    lo_raw = float(min(sub["true_raw_cycle"].min(), sub["pred_raw_cycle"].min()))
    hi_raw = float(max(sub["true_raw_cycle"].max(), sub["pred_raw_cycle"].max()))
    axes[row_idx, 1].plot([lo_raw, hi_raw], [lo_raw, hi_raw], linestyle="--", color="black", linewidth=1.0)
    axes[row_idx, 1].scatter(sub["true_raw_cycle"], sub["pred_raw_cycle"], s=24, alpha=0.76, edgecolors="none", color="#b65f2a")
    axes[row_idx, 1].set_title(f"Split {module12_plot_split} | {window_label} raw")
    axes[row_idx, 1].set_xlabel("True raw cycle from 100% anchor")
    axes[row_idx, 1].set_ylabel("Predicted raw cycle from 100% anchor")
    axes[row_idx, 1].grid(alpha=0.22)
plt.show()

fit_plot_df = module12_prediction_long_df.loc[module12_prediction_long_df["anchor_split"].eq(module12_plot_split)].copy()
fit_plot_df = fit_plot_df.sort_values(["cell", "SOH"], ascending=[True, False]).reset_index(drop=True)
fig, axes = plt.subplots(1, len(MODULE12_TAIL_WINDOWS), figsize=(5.0 * len(MODULE12_TAIL_WINDOWS), 4.8), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
for ax, window_start in zip(axes, MODULE12_TAIL_WINDOWS):
    window_label = f"{window_start}-80"
    for cell_id, sub in fit_plot_df.loc[fit_plot_df["window"].eq(window_label)].groupby("cell", sort=True):
        sub = sub.sort_values("SOH", ascending=False).copy()
        train_sub = sub.loc[sub["segment_role"].eq("train")].copy()
        pred_sub = sub.loc[sub["segment_role"].eq("pred")].copy()
        exp_sub = sub.copy()

        if len(train_sub) >= 2:
            ax.plot(train_sub["pred_norm_cycle"], train_sub["SOH"], color="#d97904", alpha=0.35, linewidth=1.0)
        if len(pred_sub) >= 2:
            ax.plot(pred_sub["pred_norm_cycle"], pred_sub["SOH"], color="#2f6f9f", alpha=0.35, linewidth=1.0)
        ax.scatter(exp_sub["exp_norm_cycle"], exp_sub["SOH"], s=10, color="#8f8f8f", alpha=0.28, edgecolors="none")

    ax.set_title(f"Split {module12_plot_split}: fit 100->{window_start}, predict {window_start - 1}->80")
    ax.set_xlabel("Normalized cycle")
    ax.set_ylabel("SOH (%)")
    ax.set_ylim(79, 101)
    ax.set_xlim(left=0.0)
    ax.grid(alpha=0.22)

legend_handles = [
    plt.Line2D([0], [0], color="#d97904", linewidth=1.5, label="Train segment"),
    plt.Line2D([0], [0], color="#2f6f9f", linewidth=1.5, label="Prediction segment"),
    plt.Line2D([0], [0], linestyle="none", marker="o", markersize=5, color="#8f8f8f", label="Experimental points"),
]
fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
plt.show()


## Module 12 overview. Early-fit and late-prediction overview in raw cycle scale

This overview combines the Module 12 early-fit and late-prediction curves for the three prediction windows. Main panels show raw-cycle SOH trajectories, and inset panels show predicted versus true raw cycle positions for the prediction segment.


In [ ]:
if "module12_prediction_long_df" not in globals() or module12_prediction_long_df.empty:
    raise ValueError("Run Module 12 before the early-fit / late-prediction overview.")
if "module12_scatter_df" not in globals() or module12_scatter_df.empty:
    raise ValueError("Run Module 12 before the true-versus-predicted inset plots.")

module12_available_splits = sorted(module12_prediction_long_df["anchor_split"].dropna().unique())
if not module12_available_splits:
    raise ValueError("No Module 12 anchor split is available for plotting.")

module12_overview_split = globals().get("module12_plot_split", module12_available_splits[0])
if module12_overview_split not in module12_available_splits:
    module12_overview_split = module12_available_splits[0]

overview_long_df = (
    module12_prediction_long_df
    .loc[module12_prediction_long_df["anchor_split"].eq(module12_overview_split)]
    .copy()
    .sort_values(["window", "cell", "SOH"], ascending=[True, True, False])
)
overview_scatter_df = (
    module12_scatter_df
    .loc[module12_scatter_df["anchor_split"].eq(module12_overview_split)]
    .copy()
)

window_order = [f"{w}-80" for w in MODULE12_TAIL_WINDOWS if f"{w}-80" in set(overview_long_df["window"].astype(str))]
if not window_order:
    window_order = sorted(overview_long_df["window"].astype(str).unique())

fig, axes = plt.subplots(1, len(window_order), figsize=(5.2 * len(window_order), 4.4), constrained_layout=True, dpi=500)
axes = np.atleast_1d(axes).ravel()

train_color = "darkorange"   # yellow for train segment
pred_color = "cornflowerblue"    # blue for prediction segment
exp_color = "#7a7a7a"     # grey for experimental points
scatter_color = "grey" # orange for scatter plot

# Determine global x and y limits across all subplots for consistent ticks
all_cycles = []
all_soh = []
for ax, window_label in zip(axes, window_order):
    sub_window = overview_long_df.loc[overview_long_df["window"].astype(str).eq(window_label)].copy()
    if not sub_window.empty:
        all_cycles.extend(sub_window["exp_raw_cycle"].dropna().values)
        all_cycles.extend(sub_window["pred_raw_cycle"].dropna().values)
        all_soh.extend(sub_window["SOH"].dropna().values)

if all_cycles and all_soh:
    global_x_min = min(all_cycles)
    global_x_max = max(all_cycles)
    global_y_min = 79
    global_y_max = 101

for ax, window_label in zip(axes, window_order):
    sub_window = overview_long_df.loc[overview_long_df["window"].astype(str).eq(window_label)].copy()
    if sub_window.empty:
        ax.set_visible(False)
        continue

    for cell_id, cell_df in sub_window.groupby("cell", sort=True):
        cell_df = cell_df.sort_values("SOH", ascending=False)
        train_sub = cell_df.loc[cell_df["segment_role"].eq("train")].copy()
        pred_sub = cell_df.loc[cell_df["segment_role"].eq("pred")].copy()

        # ---- Connect pred to the last point of train ----
        if len(train_sub) >= 1 and len(pred_sub) >= 1:
            # Get the last point of the train segment (lowest SOH in train)
            last_train_point = train_sub.iloc[-1]
            # Create a new DataFrame for the connected pred segment
            pred_connected = pd.concat([
                pd.DataFrame([last_train_point[["pred_raw_cycle", "SOH"]]]),
                pred_sub[["pred_raw_cycle", "SOH"]]
            ], ignore_index=True)
        else:
            pred_connected = pred_sub[["pred_raw_cycle", "SOH"]].copy()

        # Experimental SOH points with triangle marker
        ax.scatter(
            cell_df["exp_raw_cycle"],
            cell_df["SOH"],
            s=20,
            marker="o",                 # Changed to triangle marker
            color=exp_color,
            alpha=0.3,
            edgecolors="none",
            zorder=1,
        )
        if len(train_sub) >= 2:
            ax.plot(
                train_sub["pred_raw_cycle"],
                train_sub["SOH"],
                color=train_color,      # Yellow line for train segment
                alpha=0.4,
                linewidth=1.5,
                zorder=2,
            )
        if len(pred_connected) >= 2:    # Use pred_connected instead of pred_sub
            ax.plot(
                pred_connected["pred_raw_cycle"],
                pred_connected["SOH"],
                color=pred_color,
                alpha=0.4,
                linewidth=1.5,
                zorder=2,
            )

    window_start = int(str(window_label).split("-")[0])
    ax.axhline(window_start, color="#9a9a9a", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.text(0.07, 0.92, f"Fit window 100-{window_start}% SOH", transform=ax.transAxes, fontsize=10, color=train_color)
    ax.text(0.03,0.1, f"Predict window \n {window_start - 1}-80% SOH", transform=ax.transAxes, linespacing=1.5, fontsize=10, color=pred_color)
    ax.set_xlabel("Cycle", fontsize=12)
    ax.set_ylabel("State of health (SOH%)", fontsize=12)
    ax.set_ylim(79, 101)
    ax.set_xlim(left=0)
    ax.set_yticks([80,85,90,95,100])

    # Set consistent x and y ticks across all subplots
    if all_cycles and all_soh:
        ax.set_xlim(global_x_min - 10, global_x_max + 10)
        ax.set_ylim(global_y_min, global_y_max)

    inset_df = overview_scatter_df.loc[overview_scatter_df["window"].astype(str).eq(window_label)].copy()
    inset_ax = ax.inset_axes([0.58, 0.6, 0.36, 0.34])
    if inset_df.empty:
        inset_ax.axis("off")
    else:
        lo = float(min(inset_df["true_raw_cycle"].min(), inset_df["pred_raw_cycle"].min()))
        hi = float(max(inset_df["true_raw_cycle"].max(), inset_df["pred_raw_cycle"].max()))
        pad = max((hi - lo) * 0.03, 1.0)
        inset_ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], linestyle="--", color="black", linewidth=0.8)
        inset_ax.scatter(
            inset_df["true_raw_cycle"],
            inset_df["pred_raw_cycle"],
            s=12,
            color=scatter_color,
            alpha=0.70,
            edgecolors="none",
        )
        rmse_raw = float(np.sqrt(np.mean((inset_df["pred_raw_cycle"].to_numpy(dtype=float) - inset_df["true_raw_cycle"].to_numpy(dtype=float)) ** 2)))
        inset_ax.text(
            0.05,
            0.92,
            f"RMSE={rmse_raw:.2f}",
            transform=inset_ax.transAxes,
            ha="left",
            va="top",
            fontsize=10,
        )
        inset_ax.set_xlim(lo - pad, hi + pad)
        inset_ax.set_ylim(lo - pad, hi + pad)
        inset_ax.set_xlabel("True cycle", fontsize=10)
        inset_ax.set_ylabel("Predicted cycle", fontsize=10)
        inset_ax.tick_params(axis="both", labelsize=7)
        inset_ax.grid(alpha=0.16)

legend_handles = [
    plt.Line2D([0], [0], linestyle="none", marker="^", markersize=10, color=exp_color, alpha=0.65, label="Experimental SOH"),
    plt.Line2D([0], [0], color=train_color, linewidth=2, label="Early-fit"),
    plt.Line2D([0], [0], color=pred_color, linewidth=2, label="Late-prediction"),
]
fig.legend(
    legend_handles,
    [h.get_label() for h in legend_handles],
    loc="upper center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 1.15),
    fontsize=12
)
plt.savefig(FIGURE_DIR / "F5a.tiff", dpi=500, bbox_inches="tight")
plt.show()


## Module 12A. Anchor full-trajectory fits used for prediction manifolds

This diagnostic mirrors Module 14C but is placed next to the actual Module 12 prediction logic. The anchor cells are arbitrary protocol-level reference cells selected by the shared anchor split plan. Each anchor is represented by its full-trajectory Module 10 PBS fit; this is the parameter manifold used to restrict prediction candidates for the other repeated cells.


In [ ]:

if "module12_anchor_split_plan_df" not in globals() or module12_anchor_split_plan_df.empty:
    raise ValueError("Run Module 12 before Module 12A.")
if "module10_hbi_df" not in globals() or module10_hbi_df.empty or "module10_hbi_long_df" not in globals() or module10_hbi_long_df.empty:
    raise ValueError("Module 12A requires Module 10 full-trajectory fit outputs.")

anchor_plan_plot_df = (
    module12_anchor_split_plan_df
    .sort_values(["anchor_split", "protocol_group", "anchor_cell"])
    .drop_duplicates(["anchor_split", "anchor_cell"])
    .head(5)
    .copy()
)

module12_anchor_fit_rows = []
module12_anchor_fit_long_rows = []
module10_meta = module10_hbi_df.copy()
module10_meta["cell"] = module10_meta["cell"].astype(str)
module10_long = module10_hbi_long_df.copy()
module10_long["cell"] = module10_long["cell"].astype(str)

for anchor_plan in anchor_plan_plot_df.itertuples(index=False):
    anchor_cell = str(anchor_plan.anchor_cell)
    meta = module10_meta.loc[module10_meta["cell"].eq(anchor_cell)]
    traj = module10_long.loc[module10_long["cell"].eq(anchor_cell)].sort_values("SOH", ascending=False)
    if meta.empty or traj.empty:
        continue
    meta0 = meta.iloc[0]
    module12_anchor_fit_rows.append({
        "anchor_split": int(anchor_plan.anchor_split),
        "anchor_cell": anchor_cell,
        "protocol_group": str(anchor_plan.protocol_group),
        "candidate_surrogate_row": int(meta0["best_surrogate_row"]),
        "full_spacing_rmse_norm": float(meta0.get("module10_curve_spacing_rmse", np.nan)),
        "full_shape_rmse_norm": float(meta0.get("module10_shape_slope_rmse", np.nan)),
        "full_fit_loss": float(meta0.get("module10_fit_loss", np.nan)),
        "module10_total_energy": float(meta0.get("module10_total_energy", np.nan)),
        "n_levels": int(traj["SOH"].nunique()),
    })
    for row in traj.itertuples(index=False):
        module12_anchor_fit_long_rows.append({
            "anchor_split": int(anchor_plan.anchor_split),
            "anchor_cell": anchor_cell,
            "cell": anchor_cell,
            "protocol_group": str(anchor_plan.protocol_group),
            "SOH": int(row.SOH),
            "exp_norm_cycle": float(row.tongji_exp_space_norm_cycle),
            "pred_norm_cycle": float(row.surrogate_space_norm_cycle),
        })

module12_anchor_fit_df = pd.DataFrame(module12_anchor_fit_rows)
module12_anchor_fit_long_df = pd.DataFrame(module12_anchor_fit_long_rows)
if module12_anchor_fit_df.empty:
    raise ValueError("No Module 12 anchor fits were found.")

display(module12_anchor_fit_df.sort_values(["anchor_split", "anchor_cell"]))

anchor_cells_order = module12_anchor_fit_df.sort_values(["anchor_split", "anchor_cell"])["anchor_cell"].astype(str).tolist()
fig, axes = plt.subplots(1, len(anchor_cells_order), figsize=(3.25 * len(anchor_cells_order), 3.2), sharey=True, constrained_layout=False)
axes = np.atleast_1d(axes)
for ax, anchor_cell in zip(axes, anchor_cells_order):
    sub = module12_anchor_fit_long_df.loc[module12_anchor_fit_long_df["anchor_cell"].astype(str).eq(anchor_cell)].sort_values("SOH", ascending=False)
    ax.plot(sub["exp_norm_cycle"], sub["SOH"], color="#7a7a7a", linewidth=1.3)
    ax.scatter(sub["exp_norm_cycle"], sub["SOH"], s=18, color="#7a7a7a", label="Exp")
    ax.plot(sub["pred_norm_cycle"], sub["SOH"], color="#1f77b4", linewidth=1.9, label="PBS full fit")
    ax.scatter(sub["pred_norm_cycle"], sub["SOH"], s=22, color="#1f77b4")
    row = module12_anchor_fit_df.loc[module12_anchor_fit_df["anchor_cell"].astype(str).eq(anchor_cell)].iloc[0]
    ax.set_title(f"split {int(row['anchor_split'])}\n{anchor_cell}", fontsize=8)
    ax.set_xlabel("Normalized cycle")
    ax.set_ylim(79, 101)
    ax.grid(alpha=0.22)
axes[0].set_ylabel("SOH (%)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("Module 12A: full-trajectory anchors used to define local PBS manifolds", y=1.18, fontsize=12)
fig.subplots_adjust(left=0.065, right=0.995, top=0.78, bottom=0.19, wspace=0.22)
plt.show()


## Module 12B. Module 12 prediction parameters with anchor parameters highlighted

This diagnostic mirrors Module 14D for the fixed-hyperparameter Module 12 prediction. Test-cell parameters are shown as filled points, while full-trajectory anchor parameters are shown as hollow stars. Because Module 12 now applies a hard anchor-manifold prefilter before early-trajectory matching, the test-cell points should stay closer to the anchor-defined local parameter region.


In [ ]:

if "module12_prediction_df" not in globals() or module12_prediction_df.empty:
    raise ValueError("Run Module 12 before Module 12B.")
if "module12_anchor_fit_df" not in globals() or module12_anchor_fit_df.empty:
    raise ValueError("Run Module 12A before Module 12B.")

def deterministic_spread_positions_module12(group_labels, category_order, width=0.22):
    x = np.zeros(len(group_labels), dtype=float)
    labels_arr = np.asarray(group_labels, dtype=object)
    for idx, cat in enumerate(category_order):
        member_idx = np.flatnonzero(labels_arr == cat)
        offsets = np.array([0.0]) if len(member_idx) == 1 else np.linspace(-width, width, len(member_idx))
        x[member_idx] = idx + offsets
    return x

def module12_major_from_protocol(protocol_group):
    parts = str(protocol_group).split(" | ")
    return parts[0] if len(parts) >= 3 else str(protocol_group).split("_")[0]

def module12_short_protocol_label(protocol_group):
    parts = str(protocol_group).split(" | ")
    return " | ".join(parts[1:]) if len(parts) >= 3 else str(protocol_group)

matched_param_cols = [col for col in module12_prediction_df.columns if col.startswith("matched_")]
if not matched_param_cols:
    raise ValueError("No Module 12 matched parameter columns found.")

module12_param_rows = []
for _, row in module12_prediction_df.iterrows():
    protocol = str(row["protocol_group"])
    for pcol in matched_param_cols:
        pname = pcol.replace("matched_", "")
        module12_param_rows.append({
            "source": "test",
            "window": str(row["window"]),
            "anchor_split": int(row["anchor_split"]),
            "cell": str(row["cell"]),
            "anchor_cell": str(row["anchor_cell"]),
            "protocol_group": protocol,
            "major_group": module12_major_from_protocol(protocol),
            "protocol_label": module12_short_protocol_label(protocol),
            "param_name": pname,
            "multiplier": float(row[pcol]),
        })

module12_anchor_param_rows = []
for row in module12_anchor_fit_df.itertuples(index=False):
    protocol = str(row.protocol_group)
    surrogate_row = int(row.candidate_surrogate_row)
    for window_label in sorted(module12_prediction_df["window"].astype(str).unique()):
        for pidx, pname in enumerate(param_names):
            module12_anchor_param_rows.append({
                "source": "anchor_full_fit",
                "window": str(window_label),
                "anchor_split": int(row.anchor_split),
                "cell": str(row.anchor_cell),
                "anchor_cell": str(row.anchor_cell),
                "protocol_group": protocol,
                "major_group": module12_major_from_protocol(protocol),
                "protocol_label": module12_short_protocol_label(protocol),
                "param_name": pname,
                "multiplier": float(X_expanded_multiplier[surrogate_row, pidx]),
            })

module12_param_long_df = pd.concat([pd.DataFrame(module12_param_rows), pd.DataFrame(module12_anchor_param_rows)], ignore_index=True)
display(
    module12_param_long_df
    .groupby(["source", "window", "major_group", "protocol_label", "param_name"], as_index=False)
    .agg(mean_multiplier=("multiplier", "mean"), std_multiplier=("multiplier", "std"), n=("cell", "nunique"))
    .sort_values(["window", "major_group", "protocol_label", "source", "param_name"])
    .head(50)
)

param_order = list(param_names)
n_params = len(param_order)
n_cols = 3
n_rows = int(np.ceil(n_params / n_cols))
window_order = sorted(module12_param_long_df["window"].astype(str).unique())
major_order = [group for group in ["Tongji2", "Tongji3"] if group in set(module12_param_long_df["major_group"].astype(str))]
if not major_order:
    major_order = sorted(module12_param_long_df["major_group"].dropna().astype(str).unique())

for window_label in window_order:
    for major_group in major_order:
        plot_df = module12_param_long_df.loc[
            module12_param_long_df["window"].astype(str).eq(window_label)
            & module12_param_long_df["major_group"].astype(str).eq(major_group)
        ].copy()
        if plot_df.empty:
            continue
        protocol_categories = sorted(plot_df["protocol_label"].dropna().astype(str).unique())
        color_cycle = plt.cm.tab10(np.linspace(0, 1, max(len(protocol_categories), 3)))
        color_map = {cat: color_cycle[i] for i, cat in enumerate(protocol_categories)}
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.3 * n_cols, 3.2 * n_rows), constrained_layout=False)
        axes = np.atleast_1d(axes).ravel()
        for ax, pname in zip(axes, param_order):
            sub = plot_df.loc[plot_df["param_name"].eq(pname)].copy()
            for cat in protocol_categories:
                test_ss = sub.loc[sub["source"].eq("test") & sub["protocol_label"].astype(str).eq(cat)]
                if not test_ss.empty:
                    x_test = deterministic_spread_positions_module12(test_ss["protocol_label"].astype(str).to_numpy(), protocol_categories, width=0.18)
                    ax.scatter(x_test, test_ss["multiplier"].to_numpy(dtype=float), s=24, alpha=0.58, color=color_map[cat], edgecolors="none")
                anchor_ss = sub.loc[sub["source"].eq("anchor_full_fit") & sub["protocol_label"].astype(str).eq(cat)]
                if not anchor_ss.empty:
                    x_anchor = deterministic_spread_positions_module12(anchor_ss["protocol_label"].astype(str).to_numpy(), protocol_categories, width=0.08)
                    ax.scatter(x_anchor, anchor_ss["multiplier"].to_numpy(dtype=float), marker="*", s=155, facecolors="white", edgecolors="black", linewidths=1.15, zorder=5)
            ax.set_title(pname, fontsize=9)
            ax.set_xticks(range(len(protocol_categories)))
            ax.set_xticklabels(protocol_categories, rotation=28, ha="right")
            ax.set_ylabel("Multiplier")
            ax.grid(axis="y", alpha=0.22)
        for ax in axes[n_params:]:
            ax.set_visible(False)
        protocol_handles = [plt.Line2D([0], [0], marker="o", linestyle="", markersize=6, color=color_map[cat], label=cat) for cat in protocol_categories]
        method_handles = [
            plt.Line2D([0], [0], marker="o", linestyle="", markersize=6, markerfacecolor="#777777", markeredgecolor="none", color="#777777", label="Module 12 test cells"),
            plt.Line2D([0], [0], marker="*", linestyle="", markersize=10, markerfacecolor="white", markeredgecolor="black", color="black", label="Full-trajectory anchor"),
        ]
        fig.legend(protocol_handles + method_handles, protocol_categories + ["Module 12 test cells", "Full-trajectory anchor"], loc="center left", bbox_to_anchor=(0.995, 0.5), frameon=False, title="Protocol")
        fig.suptitle(f"{major_group}: Module 12 inferred PBS parameters | {window_label}", y=0.995, fontsize=12)
        fig.subplots_adjust(right=0.80, wspace=0.30, hspace=0.48, top=0.91, bottom=0.16)
        plt.show()


## Module 13. Five-anchor direct-learning prediction of late-life trajectory

This module is a purely data-driven benchmark under the same five-anchor train/test split used by `Module 12`, but the model itself does not receive any anchor identity, anchor trajectory, anchor parameter, or protocol descriptor.

The anchor cells only define the training set:

- train: selected anchor cells in each split;
- test: all non-anchor cells in the same split.

For each prediction task:

1. `100 -> 95`, then predict `94 -> 80`
2. `100 -> 90`, then predict `89 -> 80`
3. `100 -> 85`, then predict `84 -> 80`

the model input is only:

```text
target-cell early normalized cycle sequence
```

The output is a monotonic future cycle trajectory. The network predicts positive increments through `softplus`, then accumulates them from the last observed cycle point. This enforces monotonicity during training and inference while keeping the data-driven baseline blind to anchor type.


In [ ]:
import copy
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from matplotlib.lines import Line2D

MODULE13_WINDOWS = [95, 90, 85]
MODULE13_MODEL_NAMES = ["cnn", "lstm", "transformer"]
MODULE13_REGIMES = ["anchor_only"]
MODULE13_BATCH_SIZE = 8
MODULE13_MAX_EPOCHS = 220
MODULE13_LR = 1e-3
MODULE13_WEIGHT_DECAY = 1e-4
MODULE13_SEED = 42
MODULE13_HIDDEN_DIM = 48
MODULE13_NUM_LAYERS = 2
MODULE13_DROPOUT = 0.10
MODULE13_INCREMENT_SCALE = 0.05
MODULE13_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODULE13_LEVEL_GRID = list(range(100, 79, -1))
MODULE13_NORM_BASIS = float(tongji_dataset_max_cycle)

torch.manual_seed(MODULE13_SEED)
np.random.seed(MODULE13_SEED)
random.seed(MODULE13_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MODULE13_SEED)

print(f"Module 13 device: {MODULE13_DEVICE}")
print(f"Tongji normalization basis: {MODULE13_NORM_BASIS:.2f}")
print("Module 13 input is target early trajectory only; anchors define train/test splits only.")


def format_crate_token_module13(token):
    token = str(token).strip()
    if token == "025":
        return "0.25C"
    if token == "05":
        return "0.5C"
    if token.isdigit():
        return f"{int(token)}C"
    return token


def parse_crate_numeric_module13(token):
    token = str(token).strip()
    if token == "025":
        return 0.25
    if token == "05":
        return 0.5
    try:
        return float(token)
    except Exception:
        return np.nan


def parse_tongji_condition_fields_module13(cell_name):
    cell_name = str(cell_name)
    out = {
        "major_group": "Unknown",
        "temperature_group": "Unknown",
        "protocol_group": "Unknown",
        "temperature_C": np.nan,
        "charge_rate_C": np.nan,
        "discharge_rate_C": np.nan,
    }
    try:
        parts = cell_name.split("_")
        if len(parts) >= 3:
            major, middle, suffix = parts[:3]
            out["major_group"] = major
            if middle.startswith("CY") and "-" in middle:
                temp_token, charge_token = middle[2:].split("-", 1)
                discharge_token = suffix.split("--")[0]
                if temp_token.isdigit():
                    temp_group = f"{int(temp_token)}C"
                    out["temperature_C"] = float(temp_token)
                else:
                    temp_group = temp_token
                charge_fmt = format_crate_token_module13(charge_token)
                discharge_fmt = format_crate_token_module13(discharge_token)
                out["temperature_group"] = temp_group
                out["charge_rate_C"] = parse_crate_numeric_module13(charge_token)
                out["discharge_rate_C"] = parse_crate_numeric_module13(discharge_token)
                out["protocol_group"] = f"{major} | {temp_group} | {charge_fmt}/{discharge_fmt}"
    except Exception:
        pass
    return out


def module13_interpolate_path(rec, level_grid):
    level_to_raw = {100: 0.0}
    for level, raw_cycle in zip(rec["compare_levels"], rec["tongji_relative_cycles"]):
        level_to_raw[int(level)] = float(raw_cycle)

    available_levels = sorted(level_to_raw.keys())
    available_raw = [level_to_raw[level] for level in available_levels]
    interp_levels = np.asarray(sorted(level_grid), dtype=float)
    interp_raw = np.interp(interp_levels, np.asarray(available_levels, dtype=float), np.asarray(available_raw, dtype=float))
    interp_map = {int(level): float(raw) for level, raw in zip(interp_levels.astype(int), interp_raw)}

    raw_seq = np.asarray([interp_map[int(level)] for level in level_grid], dtype=float)
    norm_seq = raw_seq / MODULE13_NORM_BASIS
    return raw_seq, norm_seq


def module13_make_window_arrays(cell_df, window_start):
    obs_levels = [level for level in MODULE13_LEVEL_GRID if level >= window_start]
    pred_levels = [level for level in MODULE13_LEVEL_GRID if level < window_start]
    X = np.stack(cell_df["norm_seq"].to_numpy())[:, : len(obs_levels)]
    Y = np.stack(cell_df["norm_seq"].to_numpy())[:, len(obs_levels) :]
    return obs_levels, pred_levels, X.astype(np.float32), Y.astype(np.float32), X.astype(np.float32)


def module13_feature_scale(train_arr, full_arr):
    mean = train_arr.mean(axis=0, keepdims=True)
    std = train_arr.std(axis=0, keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)
    return (full_arr - mean) / std, mean, std


def module13_positive_increment_tail_torch(prefix_norm, raw_increment_logits):
    increments = F.softplus(raw_increment_logits) * MODULE13_INCREMENT_SCALE + 1e-8
    return prefix_norm[:, [-1]] + torch.cumsum(increments, dim=1)


def module13_positive_increment_tail_numpy(prefix_norm, raw_increment_logits):
    increments = np.logaddexp(np.asarray(raw_increment_logits, dtype=float), 0.0) * MODULE13_INCREMENT_SCALE + 1e-8
    return np.asarray(prefix_norm, dtype=float)[:, [-1]] + np.cumsum(increments, axis=1)


class Module13Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = int(chomp_size)

    def forward(self, x):
        if self.chomp_size <= 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class Module13TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=padding, dilation=dilation),
            Module13Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, padding=padding, dilation=dilation),
            Module13Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class Module13CNN(nn.Module):
    def __init__(self, seq_len, out_dim, hidden_dim=48, dropout=0.10, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.network = nn.Sequential(
            nn.Conv1d(1, hidden_dim, kernel_size=kernel_size, padding=padding),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=kernel_size, padding=padding),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=kernel_size, padding=padding),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        feat = self.network(x)
        return self.head(feat)


class Module13LSTM(nn.Module):
    def __init__(self, seq_len, out_dim, hidden_dim=48, num_layers=2, dropout=0.10):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        x = x.unsqueeze(-1)
        out, _ = self.lstm(x)
        feat = out[:, -1, :]
        return self.head(feat)


class Module13Transformer(nn.Module):
    def __init__(self, seq_len, out_dim, hidden_dim=48, num_layers=2, dropout=0.10, nhead=4):
        super().__init__()
        self.input_proj = nn.Linear(1, hidden_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len, hidden_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.input_proj(x) + self.pos_embed[:, : x.shape[1], :]
        feat = self.encoder(x)
        feat = self.norm(feat.mean(dim=1))
        return self.head(feat)


def module13_build_model(model_name, seq_len, out_dim):
    if model_name == "cnn":
        return Module13CNN(seq_len=seq_len, out_dim=out_dim, hidden_dim=MODULE13_HIDDEN_DIM, dropout=MODULE13_DROPOUT)
    if model_name == "lstm":
        return Module13LSTM(seq_len=seq_len, out_dim=out_dim, hidden_dim=MODULE13_HIDDEN_DIM, num_layers=MODULE13_NUM_LAYERS, dropout=MODULE13_DROPOUT)
    if model_name == "transformer":
        return Module13Transformer(seq_len=seq_len, out_dim=out_dim, hidden_dim=MODULE13_HIDDEN_DIM, num_layers=MODULE13_NUM_LAYERS, dropout=MODULE13_DROPOUT)
    raise ValueError(f"Unsupported Module 13 model: {model_name}")


def module13_fit_model(model_name, X_train, Y_train, X_test, prefix_train, prefix_test):
    n_train = len(X_train)
    if n_train < 2:
        raise ValueError("Module 13 requires at least two training samples.")

    X_train_scaled, x_mean, x_std = module13_feature_scale(X_train, X_train)
    X_test_scaled = (X_test - x_mean) / x_std
    train_ds = TensorDataset(
        torch.tensor(X_train_scaled, dtype=torch.float32),
        torch.tensor(prefix_train, dtype=torch.float32),
        torch.tensor(Y_train, dtype=torch.float32),
    )
    train_loader = DataLoader(train_ds, batch_size=min(MODULE13_BATCH_SIZE, len(train_ds)), shuffle=True)

    model = module13_build_model(model_name, seq_len=X_train.shape[1], out_dim=Y_train.shape[1]).to(MODULE13_DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=MODULE13_LR, weight_decay=MODULE13_WEIGHT_DECAY)
    criterion = nn.MSELoss()

    history_rows = []
    best_train_loss = float("inf")
    best_epoch = 0
    best_state = None
    for epoch in range(1, MODULE13_MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for xb, prefix_b, yb in train_loader:
            xb = xb.to(MODULE13_DEVICE)
            prefix_b = prefix_b.to(MODULE13_DEVICE)
            yb = yb.to(MODULE13_DEVICE)
            optimizer.zero_grad()
            pred_tail = module13_positive_increment_tail_torch(prefix_b, model(xb))
            loss = criterion(pred_tail, yb)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.item()) * len(xb)
            train_count += len(xb)
        train_loss = train_loss_sum / max(train_count, 1)
        history_rows.append({"epoch": int(epoch), "train_loss": float(train_loss), "val_loss": np.nan})
        if train_loss < best_train_loss - 1e-10:
            best_train_loss = float(train_loss)
            best_epoch = int(epoch)
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        raw_logits = model(torch.tensor(X_test_scaled, dtype=torch.float32, device=MODULE13_DEVICE)).cpu().numpy()
    pred_norm = module13_positive_increment_tail_numpy(prefix_test, raw_logits)
    return {
        "pred_norm": pred_norm,
        "history": pd.DataFrame(history_rows),
        "best_val_loss": float(best_train_loss),
        "best_epoch": int(best_epoch),
        "model": model,
        "x_mean": x_mean,
        "x_std": x_std,
    }


def module13_predict_with_fit_result(fit_result, X_input, prefix_input):
    X_scaled = (X_input - fit_result["x_mean"]) / fit_result["x_std"]
    with torch.no_grad():
        raw_logits = fit_result["model"](torch.tensor(X_scaled, dtype=torch.float32, device=MODULE13_DEVICE)).cpu().numpy()
    return module13_positive_increment_tail_numpy(prefix_input, raw_logits)


module13_records = []
for rec in tongji_path_records:
    cell_id = str(rec["cell"])
    cond = parse_tongji_condition_fields_module13(cell_id)
    if cond["major_group"] not in INCLUDED_TONGJI_MAJOR_GROUPS:
        continue
    raw_seq, norm_seq = module13_interpolate_path(rec, MODULE13_LEVEL_GRID)
    module13_records.append({
        "cell": cell_id,
        "major_group": cond["major_group"],
        "temperature_group": cond["temperature_group"],
        "protocol_group": cond["protocol_group"],
        "temperature_C": float(cond["temperature_C"]),
        "charge_rate_C": float(cond["charge_rate_C"]),
        "discharge_rate_C": float(cond["discharge_rate_C"]),
        "raw_seq": raw_seq,
        "norm_seq": norm_seq,
    })

module13_cell_df = pd.DataFrame(module13_records)
if module13_cell_df.empty:
    raise ValueError("Module 13 found no Tongji2/Tongji3 cells.")

print(f"Module 13 prepared {len(module13_cell_df)} Tongji cells.")
print("The anchor-only split table is built in Module 13A from the shared Module 12 five-anchor split plan.")


## Module 13A. Train CNN, LSTM, and Transformer with five-anchor splits

This cell runs the direct-learning experiments:

- three early windows: `100 -> 95`, `100 -> 90`, `100 -> 85`
- three architectures: `CNN`, `LSTM`, `Transformer`
- one training regime: five-anchor `anchor_only` split inherited from `Module 12`

For each split, all selected anchor cells are used for training and all non-anchor cells are used for testing. No internal validation split is used.

The model does not know which anchor or protocol group a sample belongs to. This avoids the shortcut where a data-driven model implicitly copies a same-protocol reference trajectory.


In [ ]:
if module13_cell_df.empty:
    raise ValueError("Run Module 13 before Module 13A.")
if "module12_anchor_split_plan_df" not in globals() or module12_anchor_split_plan_df.empty:
    raise ValueError("Run the updated Module 12 first so Module 13 can reuse the same five anchor splits.")

MODULE13_REGIMES = ["anchor_only"]

split_map = {"anchor_only": []}
for split_idx, split_plan in module12_anchor_split_plan_df.groupby("anchor_split", sort=True):
    train_cells = sorted(set(split_plan["anchor_cell"].astype(str)))
    all_cells = sorted(module13_cell_df["cell"].astype(str).unique())
    test_cells = sorted([cell for cell in all_cells if cell not in set(train_cells)])
    split_map["anchor_only"].append({
        "anchor_split": int(split_idx),
        "train_cells": train_cells,
        "test_cells": test_cells,
    })

module13_split_df = pd.DataFrame([
    {
        "regime": regime_name,
        "anchor_split": int(split_def["anchor_split"]),
        "n_train": len(split_def["train_cells"]),
        "n_test": len(split_def["test_cells"]),
        "train_cells": ", ".join(split_def["train_cells"]),
        "test_cells": ", ".join(split_def["test_cells"]),
    }
    for regime_name, split_defs in split_map.items()
    for split_def in split_defs
])

display(module13_split_df)


def module13_full_curve_monotonic(full_norm):
    full_norm = np.asarray(full_norm, dtype=float)
    return bool(np.all(np.diff(full_norm) >= -1e-10))


result_rows = []
long_rows = []
scatter_rows = []
history_rows = []
summary_rows = []
anchor_long_rows = []

for window_start in MODULE13_WINDOWS:
    obs_levels, pred_levels, X_all, Y_all, prefix_all = module13_make_window_arrays(module13_cell_df, window_start)
    cells = module13_cell_df["cell"].astype(str).tolist()
    cell_to_idx = {cell: idx for idx, cell in enumerate(cells)}

    for regime_name in MODULE13_REGIMES:
        for split_def in split_map[regime_name]:
            anchor_split = int(split_def["anchor_split"])
            train_cells = [cell for cell in split_def["train_cells"] if cell in cell_to_idx]
            test_cells = [cell for cell in split_def["test_cells"] if cell in cell_to_idx]
            if len(train_cells) < 2 or len(test_cells) < 1:
                print(f"Skip Module 13 regime={regime_name}, split={anchor_split}, window={window_start}: insufficient train/test cells.")
                continue

            train_idx = np.asarray([cell_to_idx[cell] for cell in train_cells], dtype=int)
            test_idx = np.asarray([cell_to_idx[cell] for cell in test_cells], dtype=int)
            X_train = X_all[train_idx]
            Y_train = Y_all[train_idx]
            prefix_train = prefix_all[train_idx]
            X_test = X_all[test_idx]
            Y_test = Y_all[test_idx]
            prefix_test = prefix_all[test_idx]

            for model_name in MODULE13_MODEL_NAMES:
                fit_result = module13_fit_model(model_name, X_train, Y_train, X_test, prefix_train, prefix_test)
                pred_test_norm = fit_result["pred_norm"]
                true_test_norm = Y_test.copy()
                pred_test_raw = pred_test_norm * MODULE13_NORM_BASIS
                true_test_raw = true_test_norm * MODULE13_NORM_BASIS

                pred_train_norm = module13_predict_with_fit_result(fit_result, X_train, prefix_train)
                pred_train_raw = pred_train_norm * MODULE13_NORM_BASIS

                hist = fit_result["history"].copy()
                hist["anchor_split"] = anchor_split
                hist["window"] = f"{window_start}-80"
                hist["regime"] = regime_name
                hist["model"] = model_name
                history_rows.append(hist)

                point_errors_norm = pred_test_norm - true_test_norm
                point_errors_raw = pred_test_raw - true_test_raw
                overall_rmse_norm = float(np.sqrt(np.mean(point_errors_norm ** 2)))
                overall_mae_norm = float(np.mean(np.abs(point_errors_norm)))
                overall_rmse_raw = float(np.sqrt(np.mean(point_errors_raw ** 2)))
                overall_mae_raw = float(np.mean(np.abs(point_errors_raw)))
                overall_mape = float(np.mean(100.0 * np.abs(point_errors_raw) / np.maximum(np.abs(true_test_raw), 1e-12)))

                summary_rows.append({
                    "anchor_split": anchor_split,
                    "window": f"{window_start}-80",
                    "regime": regime_name,
                    "model": model_name,
                    "n_train": int(len(train_idx)),
                    "n_test": int(len(test_idx)),
                    "n_pred_levels": int(len(pred_levels)),
                    "rmse_norm": overall_rmse_norm,
                    "mae_norm": overall_mae_norm,
                    "rmse_raw": overall_rmse_raw,
                    "mae_raw": overall_mae_raw,
                    "mape": overall_mape,
                })

                for local_idx, cell_idx in enumerate(test_idx):
                    cell_row = module13_cell_df.iloc[int(cell_idx)]
                    cell_id = str(cell_row["cell"])
                    full_true_norm = np.asarray(cell_row["norm_seq"], dtype=float)
                    full_true_raw = np.asarray(cell_row["raw_seq"], dtype=float)
                    full_pred_norm = np.concatenate([prefix_test[local_idx], pred_test_norm[local_idx]])
                    full_pred_raw = full_pred_norm * MODULE13_NORM_BASIS
                    pred_err_norm = pred_test_norm[local_idx] - true_test_norm[local_idx]
                    pred_err_raw = pred_test_raw[local_idx] - true_test_raw[local_idx]
                    pred_mape = float(np.mean(100.0 * np.abs(pred_err_raw) / np.maximum(np.abs(true_test_raw[local_idx]), 1e-12)))

                    result_rows.append({
                        "anchor_split": anchor_split,
                        "cell": cell_id,
                        "major_group": str(cell_row["major_group"]),
                        "temperature_group": str(cell_row["temperature_group"]),
                        "protocol_group": str(cell_row["protocol_group"]),
                        "window": f"{window_start}-80",
                        "fit_basis": f"100_to_{window_start}",
                        "regime": regime_name,
                        "model": model_name,
                        "n_obs_levels": int(len(obs_levels)),
                        "n_pred_levels": int(len(pred_levels)),
                        "rmse_norm": float(np.sqrt(np.mean(pred_err_norm ** 2))),
                        "mae_norm": float(np.mean(np.abs(pred_err_norm))),
                        "rmse_raw": float(np.sqrt(np.mean(pred_err_raw ** 2))),
                        "mae_raw": float(np.mean(np.abs(pred_err_raw))),
                        "mape": pred_mape,
                    })

                    for level_idx, level in enumerate(MODULE13_LEVEL_GRID):
                        segment_role = "train" if level >= window_start else "pred"
                        long_rows.append({
                            "anchor_split": anchor_split,
                            "cell": cell_id,
                            "major_group": str(cell_row["major_group"]),
                            "temperature_group": str(cell_row["temperature_group"]),
                            "protocol_group": str(cell_row["protocol_group"]),
                            "window": f"{window_start}-80",
                            "regime": regime_name,
                            "model": model_name,
                            "SOH": int(level),
                            "exp_norm_cycle": float(full_true_norm[level_idx]),
                            "pred_norm_cycle": float(full_pred_norm[level_idx]),
                            "exp_raw_cycle": float(full_true_raw[level_idx]),
                            "pred_raw_cycle": float(full_pred_raw[level_idx]),
                            "segment_role": segment_role,
                        })
                        if level < window_start:
                            pred_pos = pred_levels.index(level)
                            scatter_rows.append({
                                "anchor_split": anchor_split,
                                "cell": cell_id,
                                "major_group": str(cell_row["major_group"]),
                                "temperature_group": str(cell_row["temperature_group"]),
                                "protocol_group": str(cell_row["protocol_group"]),
                                "window": f"{window_start}-80",
                                "regime": regime_name,
                                "model": model_name,
                                "SOH": int(level),
                                "true_norm_cycle": float(true_test_norm[local_idx, pred_pos]),
                                "pred_norm_cycle": float(pred_test_norm[local_idx, pred_pos]),
                                "true_raw_cycle": float(true_test_raw[local_idx, pred_pos]),
                                "pred_raw_cycle": float(pred_test_raw[local_idx, pred_pos]),
                            })

                for local_idx, cell_idx in enumerate(train_idx):
                    cell_row = module13_cell_df.iloc[int(cell_idx)]
                    cell_id = str(cell_row["cell"])
                    full_true_norm = np.asarray(cell_row["norm_seq"], dtype=float)
                    full_true_raw = np.asarray(cell_row["raw_seq"], dtype=float)
                    full_pred_norm = np.concatenate([prefix_train[local_idx], pred_train_norm[local_idx]])
                    full_pred_raw = full_pred_norm * MODULE13_NORM_BASIS
                    for level_idx, level in enumerate(MODULE13_LEVEL_GRID):
                        anchor_long_rows.append({
                            "anchor_split": anchor_split,
                            "cell": cell_id,
                            "major_group": str(cell_row["major_group"]),
                            "temperature_group": str(cell_row["temperature_group"]),
                            "protocol_group": str(cell_row["protocol_group"]),
                            "window": f"{window_start}-80",
                            "regime": regime_name,
                            "model": model_name,
                            "SOH": int(level),
                            "exp_norm_cycle": float(full_true_norm[level_idx]),
                            "pred_norm_cycle": float(full_pred_norm[level_idx]),
                            "exp_raw_cycle": float(full_true_raw[level_idx]),
                            "pred_raw_cycle": float(full_pred_raw[level_idx]),
                            "segment_role": "train" if level >= window_start else "pred",
                        })

module13_result_df = pd.DataFrame(result_rows)
module13_long_df = pd.DataFrame(long_rows)
module13_scatter_df = pd.DataFrame(scatter_rows)
module13_history_df = pd.concat(history_rows, ignore_index=True) if history_rows else pd.DataFrame()
module13_split_summary_df = pd.DataFrame(summary_rows)
module13_anchor_long_df = pd.DataFrame(anchor_long_rows)

if module13_split_summary_df.empty:
    raise ValueError("Module 13A produced no experiment summary.")

module13_summary_df = (
    module13_split_summary_df
    .groupby(["window", "regime", "model"], as_index=False)
    .agg(
        n_splits=("anchor_split", "nunique"),
        n_train=("n_train", "mean"),
        n_test=("n_test", "mean"),
        n_pred_levels=("n_pred_levels", "first"),
        rmse_norm=("rmse_norm", "mean"),
        rmse_norm_std=("rmse_norm", lambda s: float(np.std(s, ddof=0))),
        mae_norm=("mae_norm", "mean"),
        mae_norm_std=("mae_norm", lambda s: float(np.std(s, ddof=0))),
        rmse_raw=("rmse_raw", "mean"),
        rmse_raw_std=("rmse_raw", lambda s: float(np.std(s, ddof=0))),
        mae_raw=("mae_raw", "mean"),
        mae_raw_std=("mae_raw", lambda s: float(np.std(s, ddof=0))),
        mape=("mape", "mean"),
        mape_std=("mape", lambda s: float(np.std(s, ddof=0))),
    )
    .sort_values(["regime", "window", "rmse_raw", "model"])
    .reset_index(drop=True)
)

module13_monotonic_rows = []
for keys, sub in module13_long_df.groupby(["anchor_split", "window", "regime", "model", "cell"], sort=True):
    sub = sub.sort_values("SOH", ascending=False)
    pred_curve = sub["pred_norm_cycle"].to_numpy(dtype=float)
    min_step = float(np.min(np.diff(pred_curve))) if len(pred_curve) > 1 else float("nan")
    module13_monotonic_rows.append({
        "anchor_split": int(keys[0]),
        "window": str(keys[1]),
        "regime": str(keys[2]),
        "model": str(keys[3]),
        "cell": str(keys[4]),
        "min_pred_cycle_step": min_step,
        "is_monotonic": bool(np.all(np.diff(pred_curve) >= -1e-10)),
    })
module13_monotonic_check_df = pd.DataFrame(module13_monotonic_rows)
n_nonmonotonic = int((~module13_monotonic_check_df["is_monotonic"]).sum()) if not module13_monotonic_check_df.empty else 0
print(f"Module 13 monotonic-output check: {len(module13_monotonic_check_df) - n_nonmonotonic}/{len(module13_monotonic_check_df)} full predicted curves are monotonic.")
if n_nonmonotonic:
    display(module13_monotonic_check_df.loc[~module13_monotonic_check_df["is_monotonic"]].head(20))

display(
    module13_summary_df[[
        "window",
        "regime",
        "model",
        "n_splits",
        "n_train",
        "n_test",
        "n_pred_levels",
        "rmse_raw",
        "rmse_raw_std",
        "rmse_norm",
        "rmse_norm_std",
        "mae_raw",
        "mae_raw_std",
        "mae_norm",
        "mae_norm_std",
        "mape",
        "mape_std",
    ]].rename(columns={
        "rmse_raw": "rmse raw",
        "rmse_raw_std": "rmse raw std",
        "rmse_norm": "rmse norm",
        "rmse_norm_std": "rmse norm std",
        "mae_raw": "mae raw",
        "mae_raw_std": "mae raw std",
        "mae_norm": "mae norm",
        "mae_norm_std": "mae norm std",
        "mape_std": "mape std",
    })
)


## Module 13B. Quantitative comparison and pooled prediction scatter

This cell summarizes the direct-learning experiments and visualizes pooled prediction quality.

For each training regime and prediction window:

- the pooled scatter compares predicted vs true cycle positions on the held-out tail;
- the three architectures are overlaid in the same subplot for direct comparison.


In [ ]:
if module13_summary_df.empty or module13_scatter_df.empty:
    raise ValueError("Run Module 13A before Module 13B.")

display(module13_summary_df)

color_map = {"cnn": "tab:blue", "lstm": "tab:orange", "transformer": "tab:green"}
marker_map = {"cnn": "o", "lstm": "s", "transformer": "^"}

fig, axes = plt.subplots(len(MODULE13_REGIMES), len(MODULE13_WINDOWS), figsize=(13.8, 7.2), constrained_layout=True)
if len(MODULE13_REGIMES) == 1:
    axes = np.asarray([axes])

for row_idx, regime_name in enumerate(MODULE13_REGIMES):
    for col_idx, window_start in enumerate(MODULE13_WINDOWS):
        ax = axes[row_idx, col_idx]
        window_label = f"{window_start}-80"
        sub = module13_scatter_df.loc[
            module13_scatter_df["regime"].eq(regime_name) & module13_scatter_df["window"].eq(window_label)
        ].copy()
        if sub.empty:
            ax.set_visible(False)
            continue
        lo = float(min(sub["true_norm_cycle"].min(), sub["pred_norm_cycle"].min()))
        hi = float(max(sub["true_norm_cycle"].max(), sub["pred_norm_cycle"].max()))
        pad = max((hi - lo) * 0.05, 1e-3)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], linestyle="--", color="black", linewidth=1.0)
        for model_name in MODULE13_MODEL_NAMES:
            sm = sub.loc[sub["model"].eq(model_name)]
            if sm.empty:
                continue
            ax.scatter(
                sm["true_norm_cycle"],
                sm["pred_norm_cycle"],
                s=30,
                alpha=0.78,
                color=color_map[model_name],
                marker=marker_map[model_name],
                edgecolors="none",
                label=model_name.upper(),
            )
        best_row = module13_summary_df.loc[
            module13_summary_df["regime"].eq(regime_name) & module13_summary_df["window"].eq(window_label)
        ].sort_values("rmse_norm").iloc[0]
        ax.set_title(
            f"{regime_name} | {window_label}\nbest={best_row['model'].upper()} | RMSE={best_row['rmse_norm']:.3f}",
            fontsize=10,
        )
        ax.set_xlabel("True normalized cycle")
        ax.set_ylabel("Predicted normalized cycle")
        ax.grid(alpha=0.22)

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    uniq = {}
    for h, l in zip(handles, labels):
        if l and l not in uniq:
            uniq[l] = h
    fig.legend(list(uniq.values()), list(uniq.keys()), loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
plt.show()

fig, axes = plt.subplots(len(MODULE13_REGIMES), len(MODULE13_WINDOWS), figsize=(13.8, 7.2), constrained_layout=True)
if len(MODULE13_REGIMES) == 1:
    axes = np.asarray([axes])

for row_idx, regime_name in enumerate(MODULE13_REGIMES):
    for col_idx, window_start in enumerate(MODULE13_WINDOWS):
        ax = axes[row_idx, col_idx]
        window_label = f"{window_start}-80"
        sub = module13_scatter_df.loc[
            module13_scatter_df["regime"].eq(regime_name) & module13_scatter_df["window"].eq(window_label)
        ].copy()
        if sub.empty:
            ax.set_visible(False)
            continue
        lo = float(min(sub["true_raw_cycle"].min(), sub["pred_raw_cycle"].min()))
        hi = float(max(sub["true_raw_cycle"].max(), sub["pred_raw_cycle"].max()))
        pad = max((hi - lo) * 0.05, 1e-3)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], linestyle="--", color="black", linewidth=1.0)
        for model_name in MODULE13_MODEL_NAMES:
            sm = sub.loc[sub["model"].eq(model_name)]
            if sm.empty:
                continue
            ax.scatter(
                sm["true_raw_cycle"],
                sm["pred_raw_cycle"],
                s=30,
                alpha=0.78,
                color=color_map[model_name],
                marker=marker_map[model_name],
                edgecolors="none",
                label=model_name.upper(),
            )
        best_row = module13_summary_df.loc[
            module13_summary_df["regime"].eq(regime_name) & module13_summary_df["window"].eq(window_label)
        ].sort_values("rmse_raw").iloc[0]
        ax.set_title(
            f"{regime_name} | {window_label}\nbest={best_row['model'].upper()} | RMSEraw={best_row['rmse_raw']:.1f}",
            fontsize=10,
        )
        ax.set_xlabel("True raw cycle")
        ax.set_ylabel("Predicted raw cycle")
        ax.grid(alpha=0.22)

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    uniq = {}
    for h, l in zip(handles, labels):
        if l and l not in uniq:
            uniq[l] = h
    fig.legend(list(uniq.values()), list(uniq.keys()), loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
plt.show()


## Module 13C. Fit-vs-experiment overlays using the best direct-learning model in each setting

To keep the visualization readable, this cell selects the **best architecture** within each `(training regime, prediction window)` pair based on mean normalized RMSE, and then plots:

1. all test cells together in one subplot;
2. train segment in orange;
3. predicted tail in blue;
4. full experimental trajectory in gray.


In [ ]:
if module13_summary_df.empty or module13_long_df.empty:
    raise ValueError("Run Module 13A before Module 13C.")

best_lookup = (
    module13_summary_df.sort_values(["regime", "window", "rmse_norm", "model"])
    .groupby(["regime", "window"], as_index=False)
    .first()
)

# Module 13 metrics are averaged over five anchor splits. For trajectory lines, plot one split only;
# otherwise the same cell appears five times and separate split curves get incorrectly connected.
MODULE13_PLOT_SPLIT = int(module13_split_df["anchor_split"].min()) if "module13_split_df" in globals() and "anchor_split" in module13_split_df.columns else 0
module13_plot_long_df = module13_long_df.loc[module13_long_df["anchor_split"].eq(MODULE13_PLOT_SPLIT)].copy()
module13_plot_result_df = module13_result_df.loc[module13_result_df["anchor_split"].eq(MODULE13_PLOT_SPLIT)].copy()
module13_plot_anchor_long_df = (
    module13_anchor_long_df.loc[module13_anchor_long_df["anchor_split"].eq(MODULE13_PLOT_SPLIT)].copy()
    if "module13_anchor_long_df" in globals() and not module13_anchor_long_df.empty and "anchor_split" in module13_anchor_long_df.columns
    else pd.DataFrame()
)

print(f"Module 13C trajectory visualization uses anchor_split={MODULE13_PLOT_SPLIT}.")
print("Module 13 quantitative summaries above still use the five-split mean/std.")

fig, axes = plt.subplots(len(MODULE13_REGIMES), len(MODULE13_WINDOWS), figsize=(14.0, 7.4), constrained_layout=True)
if len(MODULE13_REGIMES) == 1:
    axes = np.asarray([axes])

legend_handles = None
for row_idx, regime_name in enumerate(MODULE13_REGIMES):
    for col_idx, window_start in enumerate(MODULE13_WINDOWS):
        ax = axes[row_idx, col_idx]
        window_label = f"{window_start}-80"
        best_row = best_lookup.loc[best_lookup["regime"].eq(regime_name) & best_lookup["window"].eq(window_label)]
        if best_row.empty:
            ax.set_visible(False)
            continue
        model_name = str(best_row.iloc[0]["model"])
        sub = module13_plot_long_df.loc[
            module13_plot_long_df["regime"].eq(regime_name)
            & module13_plot_long_df["window"].eq(window_label)
            & module13_plot_long_df["model"].eq(model_name)
        ].copy()
        if sub.empty:
            ax.set_visible(False)
            continue
        for cell_id, cell_sub in sub.groupby("cell", sort=True):
            cell_sub = cell_sub.sort_values("SOH", ascending=False).copy()
            train_sub = cell_sub.loc[cell_sub["segment_role"].eq("train")].copy()
            pred_sub = cell_sub.loc[cell_sub["segment_role"].eq("pred")].copy()
            ax.plot(cell_sub["exp_norm_cycle"], cell_sub["SOH"], color="0.72", linewidth=1.2, alpha=0.95)
            ax.scatter(cell_sub["exp_norm_cycle"], cell_sub["SOH"], color="0.65", s=16, alpha=0.9, edgecolors="none")
            ax.plot(train_sub["exp_norm_cycle"], train_sub["SOH"], color="tab:orange", linewidth=1.6, alpha=0.95)
            ax.plot(pred_sub["pred_norm_cycle"], pred_sub["SOH"], color="tab:blue", linewidth=1.6, alpha=0.95)

        ax.set_title(
            f"{regime_name} | {window_label} | split {MODULE13_PLOT_SPLIT}\nbest={model_name.upper()}",
            fontsize=10,
        )
        ax.set_xlabel("Normalized cycle")
        ax.set_ylabel("SOH (%)")
        ax.grid(alpha=0.18)
        ax.set_ylim(79, 101)

        if legend_handles is None:
            legend_handles = [
                Line2D([0], [0], color="0.72", linewidth=1.2, label="Exp curve"),
                Line2D([0], [0], marker="o", color="0.65", linestyle="None", markersize=4.5, label="Exp points"),
                Line2D([0], [0], color="tab:orange", linewidth=1.8, label="Observed train segment"),
                Line2D([0], [0], color="tab:blue", linewidth=1.8, label="Predicted tail"),
            ]

if legend_handles:
    fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.02))
plt.show()

for regime_name in MODULE13_REGIMES:
    for window_start in MODULE13_WINDOWS:
        window_label = f"{window_start}-80"
        best_row = best_lookup.loc[best_lookup["regime"].eq(regime_name) & best_lookup["window"].eq(window_label)]
        if best_row.empty:
            continue
        model_name = str(best_row.iloc[0]["model"])
        sub = module13_plot_long_df.loc[
            module13_plot_long_df["regime"].eq(regime_name)
            & module13_plot_long_df["window"].eq(window_label)
            & module13_plot_long_df["model"].eq(model_name)
        ].copy()
        if sub.empty:
            continue
        cell_ids = sorted(sub["cell"].astype(str).unique())
        n_cells = len(cell_ids)
        n_cols = 4
        n_rows = int(math.ceil(n_cells / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 3.0 * n_rows), constrained_layout=True)
        axes = np.atleast_1d(axes).ravel()
        for ax, cell_id in zip(axes, cell_ids):
            cell_sub = sub.loc[sub["cell"].eq(cell_id)].sort_values("SOH", ascending=False).copy()
            train_sub = cell_sub.loc[cell_sub["segment_role"].eq("train")].copy()
            pred_sub = cell_sub.loc[cell_sub["segment_role"].eq("pred")].copy()
            metric_row = module13_plot_result_df.loc[
                module13_plot_result_df["cell"].eq(cell_id)
                & module13_plot_result_df["regime"].eq(regime_name)
                & module13_plot_result_df["window"].eq(window_label)
                & module13_plot_result_df["model"].eq(model_name)
            ]
            rmse_norm = float(metric_row.iloc[0]["rmse_norm"]) if not metric_row.empty else float("nan")
            rmse_raw = float(metric_row.iloc[0]["rmse_raw"]) if not metric_row.empty else float("nan")

            ax.plot(cell_sub["exp_norm_cycle"], cell_sub["SOH"], color="0.72", linewidth=1.2, alpha=0.95)
            ax.scatter(cell_sub["exp_norm_cycle"], cell_sub["SOH"], color="0.65", s=16, alpha=0.9, edgecolors="none")
            ax.plot(train_sub["exp_norm_cycle"], train_sub["SOH"], color="tab:orange", linewidth=1.6, alpha=0.95)
            ax.plot(pred_sub["pred_norm_cycle"], pred_sub["SOH"], color="tab:blue", linewidth=1.6, alpha=0.95)
            ax.set_title(f"{cell_id}\nsplit={MODULE13_PLOT_SPLIT} | RMSEn={rmse_norm:.3f} | RMSER={rmse_raw:.1f}", fontsize=8.8)
            ax.set_xlabel("Normalized cycle")
            ax.set_ylabel("SOH (%)")
            ax.grid(alpha=0.18)
            ax.set_ylim(79, 101)
        for ax in axes[n_cells:]:
            ax.set_visible(False)
        fig.suptitle(f"Module 13 per-cell overlays | {regime_name} | {window_label} | {model_name.upper()} | split {MODULE13_PLOT_SPLIT}", fontsize=12)
        if legend_handles:
            fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.02))
        plt.show()

if not module13_plot_anchor_long_df.empty:
    anchor_only_plot_df = module13_plot_anchor_long_df.copy()
    for window_start in MODULE13_WINDOWS:
        window_label = f"{window_start}-80"
        fig, axes = plt.subplots(len(MODULE13_MODEL_NAMES), 1, figsize=(8.0, 3.3 * len(MODULE13_MODEL_NAMES)), constrained_layout=True)
        axes = np.atleast_1d(axes).ravel()
        plotted_any = False
        for ax, model_name in zip(axes, MODULE13_MODEL_NAMES):
            sub = anchor_only_plot_df.loc[
                anchor_only_plot_df["window"].eq(window_label)
                & anchor_only_plot_df["model"].eq(model_name)
                & anchor_only_plot_df["regime"].eq("anchor_only")
            ].copy()
            if sub.empty:
                ax.set_visible(False)
                continue
            plotted_any = True
            for cell_id, cell_sub in sub.groupby("cell", sort=True):
                cell_sub = cell_sub.sort_values("SOH", ascending=False).copy()
                train_sub = cell_sub.loc[cell_sub["segment_role"].eq("train")].copy()
                pred_sub = cell_sub.loc[cell_sub["segment_role"].eq("pred")].copy()
                ax.plot(cell_sub["exp_norm_cycle"], cell_sub["SOH"], color="0.72", linewidth=1.2, alpha=0.95)
                ax.scatter(cell_sub["exp_norm_cycle"], cell_sub["SOH"], color="0.65", s=16, alpha=0.9, edgecolors="none")
                ax.plot(train_sub["exp_norm_cycle"], train_sub["SOH"], color="tab:orange", linewidth=1.6, alpha=0.95)
                ax.plot(pred_sub["pred_norm_cycle"], pred_sub["SOH"], color="tab:blue", linewidth=1.6, alpha=0.95)
            ax.set_title(f"Anchor-only training anchors | {window_label} | {model_name.upper()} | split {MODULE13_PLOT_SPLIT}", fontsize=10)
            ax.set_xlabel("Normalized cycle")
            ax.set_ylabel("SOH (%)")
            ax.set_ylim(79, 101)
            ax.grid(alpha=0.18)
        if plotted_any:
            fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.02))
            plt.show()


## Module 13 overview. Best direct-learning model for each early-window task in raw cycle scale

This overview uses the best-performing direct-learning model for each prediction window and visualizes its early-input segment, late-life prediction, and raw-cycle true-versus-predicted agreement.


In [ ]:
if "module13_summary_df" not in globals() or module13_summary_df.empty:
    raise ValueError("Run Module 13A before the Module 13 overview plot.")
if "module13_long_df" not in globals() or module13_long_df.empty:
    raise ValueError("Run Module 13A before plotting Module 13 trajectory overlays.")
if "module13_scatter_df" not in globals() or module13_scatter_df.empty:
    raise ValueError("Run Module 13A before plotting Module 13 true-versus-predicted insets.")

module13_overview_regime = "anchor_only" if "anchor_only" in set(module13_summary_df["regime"].astype(str)) else str(module13_summary_df["regime"].astype(str).iloc[0])
module13_overview_summary = module13_summary_df.loc[module13_summary_df["regime"].astype(str).eq(module13_overview_regime)].copy()
if module13_overview_summary.empty:
    raise ValueError("No Module 13 summary rows are available for the selected overview regime.")

module13_best_by_window = (
    module13_overview_summary
    .sort_values(["window", "rmse_norm", "rmse_raw", "model"])
    .groupby("window", as_index=False)
    .first()
)
module13_best_model_lookup = dict(zip(module13_best_by_window["window"].astype(str), module13_best_by_window["model"].astype(str)))

module13_available_splits = sorted(module13_long_df["anchor_split"].dropna().unique()) if "anchor_split" in module13_long_df.columns else [0]
if not module13_available_splits:
    raise ValueError("No Module 13 anchor split is available for plotting.")
module13_overview_split = globals().get("MODULE13_PLOT_SPLIT", module13_available_splits[0])
if module13_overview_split not in module13_available_splits:
    module13_overview_split = module13_available_splits[0]

module13_overview_long_df = module13_long_df.loc[
    module13_long_df["regime"].astype(str).eq(module13_overview_regime)
    & module13_long_df["anchor_split"].eq(module13_overview_split)
].copy()
module13_overview_scatter_df = module13_scatter_df.loc[
    module13_scatter_df["regime"].astype(str).eq(module13_overview_regime)
    & module13_scatter_df["anchor_split"].eq(module13_overview_split)
].copy()

window_order = [f"{w}-80" for w in MODULE13_WINDOWS if f"{w}-80" in set(module13_best_by_window["window"].astype(str))]
if not window_order:
    window_order = sorted(module13_best_by_window["window"].astype(str).unique())

fig, axes = plt.subplots(1, len(window_order), figsize=(5.2 * len(window_order), 4.4), constrained_layout=True, dpi=500)
axes = np.atleast_1d(axes).ravel()

train_color = "darkorange"
pred_color = "cornflowerblue"
exp_color = "#7a7a7a"
scatter_color = "grey"

all_cycles = []
for window_label in window_order:
    model_name = module13_best_model_lookup.get(window_label)
    sub_window = module13_overview_long_df.loc[
        module13_overview_long_df["window"].astype(str).eq(window_label)
        & module13_overview_long_df["model"].astype(str).eq(model_name)
    ].copy()
    if not sub_window.empty:
        all_cycles.extend(sub_window["exp_raw_cycle"].dropna().to_numpy(dtype=float))
        all_cycles.extend(sub_window["pred_raw_cycle"].dropna().to_numpy(dtype=float))

if all_cycles:
    global_x_min = float(np.nanmin(all_cycles))
    global_x_max = float(np.nanmax(all_cycles))
    x_pad = max((global_x_max - global_x_min) * 0.03, 10.0)
else:
    global_x_min, global_x_max, x_pad = 0.0, 1.0, 0.0

for ax, window_label in zip(axes, window_order):
    model_name = module13_best_model_lookup.get(window_label)
    sub_window = module13_overview_long_df.loc[
        module13_overview_long_df["window"].astype(str).eq(window_label)
        & module13_overview_long_df["model"].astype(str).eq(model_name)
    ].copy()
    if sub_window.empty:
        ax.set_visible(False)
        continue

    for cell_id, cell_df in sub_window.groupby("cell", sort=True):
        cell_df = cell_df.sort_values("SOH", ascending=False).copy()
        train_sub = cell_df.loc[cell_df["segment_role"].eq("train")].copy()
        pred_sub = cell_df.loc[cell_df["segment_role"].eq("pred")].copy()

        if len(train_sub) >= 1 and len(pred_sub) >= 1:
            last_train_point = train_sub.iloc[-1]
            pred_connected = pd.concat([
                pd.DataFrame([last_train_point[["pred_raw_cycle", "SOH"]]]),
                pred_sub[["pred_raw_cycle", "SOH"]],
            ], ignore_index=True)
        else:
            pred_connected = pred_sub[["pred_raw_cycle", "SOH"]].copy()

        ax.scatter(
            cell_df["exp_raw_cycle"],
            cell_df["SOH"],
            s=20,
            marker="o",
            color=exp_color,
            alpha=0.30,
            edgecolors="none",
            zorder=1,
        )
        if len(train_sub) >= 2:
            ax.plot(
                train_sub["pred_raw_cycle"],
                train_sub["SOH"],
                color=train_color,
                alpha=0.42,
                linewidth=1.5,
                zorder=2,
            )
        if len(pred_connected) >= 2:
            ax.plot(
                pred_connected["pred_raw_cycle"],
                pred_connected["SOH"],
                color=pred_color,
                alpha=0.42,
                linewidth=1.5,
                zorder=2,
            )

    window_start = int(str(window_label).split("-")[0])
    ax.axhline(window_start, color="#9a9a9a", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.text(0.07, 0.92, f"Input window 100-{window_start}% SOH", transform=ax.transAxes, fontsize=10, color=train_color)
    ax.text(0.03, 0.10, f"Predict window\n{window_start - 1}-80% SOH", transform=ax.transAxes, linespacing=1.5, fontsize=10, color=pred_color)
    ax.set_title(f"{window_label} task\nbest model: {model_name.upper()}", fontsize=11)
    ax.set_xlabel("Cycle", fontsize=12)
    ax.set_ylabel("State of health (SOH%)", fontsize=12)
    ax.set_ylim(79, 101)
    ax.set_yticks([80, 85, 90, 95, 100])
    ax.set_xlim(global_x_min - x_pad, global_x_max + x_pad)
    ax.grid(alpha=0.16)

    inset_df = module13_overview_scatter_df.loc[
        module13_overview_scatter_df["window"].astype(str).eq(window_label)
        & module13_overview_scatter_df["model"].astype(str).eq(model_name)
    ].copy()
    inset_ax = ax.inset_axes([0.58, 0.60, 0.36, 0.34])
    if inset_df.empty:
        inset_ax.axis("off")
    else:
        lo = float(min(inset_df["true_raw_cycle"].min(), inset_df["pred_raw_cycle"].min()))
        hi = float(max(inset_df["true_raw_cycle"].max(), inset_df["pred_raw_cycle"].max()))
        pad = max((hi - lo) * 0.03, 1.0)
        inset_ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], linestyle="--", color="black", linewidth=0.8)
        inset_ax.scatter(
            inset_df["true_raw_cycle"],
            inset_df["pred_raw_cycle"],
            s=12,
            color=scatter_color,
            alpha=0.70,
            edgecolors="none",
        )
        rmse_raw = float(np.sqrt(np.mean((inset_df["pred_raw_cycle"].to_numpy(dtype=float) - inset_df["true_raw_cycle"].to_numpy(dtype=float)) ** 2)))
        inset_ax.text(0.05, 0.92, f"RMSE={rmse_raw:.2f}", transform=inset_ax.transAxes, ha="left", va="top", fontsize=10)
        inset_ax.set_xlim(lo - pad, hi + pad)
        inset_ax.set_ylim(lo - pad, hi + pad)
        inset_ax.set_xlabel("True cycle", fontsize=10)
        inset_ax.set_ylabel("Predicted cycle", fontsize=10)
        inset_ax.tick_params(axis="both", labelsize=7)
        inset_ax.grid(alpha=0.16)

legend_handles = [
    plt.Line2D([0], [0], linestyle="none", marker="o", markersize=6, color=exp_color, alpha=0.65, label="Experimental SOH"),
    plt.Line2D([0], [0], color=train_color, linewidth=2, label="Early input"),
    plt.Line2D([0], [0], color=pred_color, linewidth=2, label="Late prediction"),
]
fig.legend(
    legend_handles,
    [h.get_label() for h in legend_handles],
    loc="upper center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 1.15),
    fontsize=12,
)
plt.savefig(FIGURE_DIR / "F_module13_best_models_raw.tiff", dpi=500, bbox_inches="tight")
plt.show()


## Module 14A. Prediction-error comparison without retrospective finetuning

This visualization compares the fixed-hyperparameter PBS prediction from Module 12 against the three direct-learning baselines from Module 13. It intentionally excludes the Module 14 retrospective best-finetune result, because that setting uses late-life prediction error for hyperparameter selection and is not a deployable prediction protocol.

The bars report five-anchor-split mean performance with split-to-split standard deviation as error bars.


In [ ]:
if "module13_summary_df" not in globals() or module13_summary_df.empty:
    raise ValueError("Run Module 13A before Module 14A.")
if "module12_window_overall_df" not in globals() or module12_window_overall_df.empty:
    raise ValueError("Run Module 12 before Module 14A.")

module13_anchor_df = module13_summary_df.loc[module13_summary_df["regime"].eq("anchor_only")].copy()
if module13_anchor_df.empty:
    raise ValueError("Module 13 anchor_only results are required for Module 14A.")

module13_anchor_df["model_key"] = module13_anchor_df["model"].astype(str).str.lower()
model_specs = [
    {"label": "Module 13 CNN", "model_key": "cnn", "color": "#c46a1a"},
    {"label": "Module 13 LSTM", "model_key": "lstm", "color": "#8c564b"},
    {"label": "Module 13 Transformer", "model_key": "transformer", "color": "#9467bd"},
]
model_specs = [spec for spec in model_specs if spec["model_key"] in set(module13_anchor_df["model_key"])]
method_specs = model_specs + [
    {"label": "Module 12 PBS", "model_key": None, "color": "#2f6f9f"},
]

metric_specs = [
    ("rmse raw", "rmse_raw", "RMSE raw"),
    ("mae raw", "mae_raw", "MAE raw"),
    ("mape", "mape", "MAPE"),
]

window_order = sorted(module12_window_overall_df["window"].astype(str).unique())
xx = np.arange(len(window_order))
width = min(0.17, 0.78 / max(len(method_specs), 1))
offsets = (np.arange(len(method_specs)) - (len(method_specs) - 1) / 2.0) * width

fig, axes = plt.subplots(1, len(metric_specs), figsize=(5.1 * len(metric_specs), 4.4), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

comparison_rows = []
for ax, (module12_metric, module13_metric, metric_label) in zip(axes, metric_specs):
    for offset, spec in zip(offsets, method_specs):
        vals = np.full(len(window_order), np.nan, dtype=float)
        errs = np.zeros(len(window_order), dtype=float)
        for wi, window_label in enumerate(window_order):
            if spec["model_key"] is None:
                row = module12_window_overall_df.loc[module12_window_overall_df["window"].astype(str).eq(window_label)]
                if row.empty:
                    continue
                vals[wi] = float(row.iloc[0][module12_metric])
                errs[wi] = float(row.iloc[0].get(f"{module12_metric} std", 0.0))
                method_label = spec["label"]
            else:
                row = module13_anchor_df.loc[
                    module13_anchor_df["window"].astype(str).eq(window_label)
                    & module13_anchor_df["model_key"].eq(spec["model_key"])
                ]
                if row.empty:
                    continue
                vals[wi] = float(row.iloc[0][module13_metric])
                errs[wi] = float(row.iloc[0].get(f"{module13_metric}_std", 0.0))
                method_label = spec["label"]

            comparison_rows.append({
                "window": window_label,
                "method": method_label,
                "metric": metric_label,
                "value": vals[wi],
                "std": errs[wi],
            })

        ax.bar(
            xx + offset,
            vals,
            width=width * 0.92,
            yerr=errs,
            capsize=3,
            color=spec["color"],
            label=spec["label"],
        )

    ax.set_xticks(xx)
    ax.set_xticklabels(window_order, rotation=15, ha="right")
    ax.set_title(metric_label, fontsize=11)
    ax.set_xlabel("Prediction window")
    ax.set_ylabel(metric_label)
    ax.grid(axis="y", alpha=0.22)

handles, labels = axes[0].get_legend_handles_labels()
uniq = {}
for h, l in zip(handles, labels):
    if l and l not in uniq:
        uniq[l] = h
fig.legend(list(uniq.values()), list(uniq.keys()), loc="upper center", ncol=min(len(uniq), 4), frameon=False, bbox_to_anchor=(0.5, 1.04))
plt.show()

module14a_error_bar_df = pd.DataFrame(comparison_rows)
display(module14a_error_bar_df.pivot_table(index=["window", "method"], columns="metric", values=["value", "std"]))


## Module 14B. Endpoint prediction-error comparison at 80% SOH

This visualization uses the same comparison set as Module 14A, but evaluates only the predicted cycle at 80% SOH. The comparison intentionally includes only the deployable fixed-PBS prediction and the three direct-learning baselines, not retrospective best finetuning.


In [ ]:
if "module13_scatter_df" not in globals() or module13_scatter_df.empty:
    raise ValueError("Run Module 13A before Module 14B.")
if "module12_scatter_df" not in globals() or module12_scatter_df.empty:
    raise ValueError("Run Module 12 before Module 14B.")

model_specs = [
    {"label": "Module 13 CNN", "model_key": "cnn", "color": "#c46a1a"},
    {"label": "Module 13 LSTM", "model_key": "lstm", "color": "#8c564b"},
    {"label": "Module 13 Transformer", "model_key": "transformer", "color": "#9467bd"},
]
available_models = set(module13_scatter_df.loc[module13_scatter_df["regime"].eq("anchor_only"), "model"].astype(str).str.lower())
model_specs = [spec for spec in model_specs if spec["model_key"] in available_models]
method_specs = model_specs + [
    {"label": "Module 12 PBS", "model_key": None, "color": "#2f6f9f"},
]

def summarize_endpoint80_from_scatter(df, method_label, model_key=None):
    rows = []
    base = df.loc[df["SOH"].eq(80)].copy()
    if model_key is not None:
        base = base.loc[
            base["regime"].eq("anchor_only")
            & base["model"].astype(str).str.lower().eq(str(model_key).lower())
        ].copy()
    if base.empty:
        return pd.DataFrame()
    for (window_label, split_idx), sub in base.groupby(["window", "anchor_split"], sort=True):
        err_raw = sub["pred_raw_cycle"].to_numpy(dtype=float) - sub["true_raw_cycle"].to_numpy(dtype=float)
        err_norm = sub["pred_norm_cycle"].to_numpy(dtype=float) - sub["true_norm_cycle"].to_numpy(dtype=float)
        true_raw = sub["true_raw_cycle"].to_numpy(dtype=float)
        rows.append({
            "window": str(window_label),
            "anchor_split": int(split_idx),
            "method": method_label,
            "rmse raw @ 80": float(np.sqrt(np.mean(err_raw ** 2))),
            "mae raw @ 80": float(np.mean(np.abs(err_raw))),
            "mape @ 80": float(np.mean(100.0 * np.abs(err_raw) / np.maximum(np.abs(true_raw), 1e-12))),
            "rmse norm @ 80": float(np.sqrt(np.mean(err_norm ** 2))),
            "mae norm @ 80": float(np.mean(np.abs(err_norm))),
        })
    return pd.DataFrame(rows)

endpoint_parts = []
for spec in method_specs:
    if spec["model_key"] is None:
        endpoint_parts.append(summarize_endpoint80_from_scatter(module12_scatter_df, spec["label"], None))
    else:
        endpoint_parts.append(summarize_endpoint80_from_scatter(module13_scatter_df, spec["label"], spec["model_key"]))

endpoint_split_df = pd.concat([part for part in endpoint_parts if part is not None and not part.empty], ignore_index=True)
if endpoint_split_df.empty:
    raise ValueError("No endpoint-80 comparison rows were constructed.")

endpoint_summary_df = (
    endpoint_split_df
    .groupby(["window", "method"], as_index=False)
    .agg(
        **{
            "rmse raw @ 80": ("rmse raw @ 80", "mean"),
            "rmse raw @ 80 std": ("rmse raw @ 80", lambda s: float(np.std(s, ddof=0))),
            "mae raw @ 80": ("mae raw @ 80", "mean"),
            "mae raw @ 80 std": ("mae raw @ 80", lambda s: float(np.std(s, ddof=0))),
            "mape @ 80": ("mape @ 80", "mean"),
            "mape @ 80 std": ("mape @ 80", lambda s: float(np.std(s, ddof=0))),
        }
    )
)

metric_specs = [
    ("rmse raw @ 80", "RMSE raw @ 80"),
    ("mae raw @ 80", "MAE raw @ 80"),
    ("mape @ 80", "MAPE @ 80"),
]
window_order = sorted(endpoint_summary_df["window"].astype(str).unique())
method_order = [spec["label"] for spec in method_specs]
color_map = {spec["label"]: spec["color"] for spec in method_specs}

xx = np.arange(len(window_order))
width = min(0.17, 0.78 / max(len(method_order), 1))
offsets = (np.arange(len(method_order)) - (len(method_order) - 1) / 2.0) * width

fig, axes = plt.subplots(1, len(metric_specs), figsize=(5.1 * len(metric_specs), 4.4), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
for ax, (metric_col, metric_label) in zip(axes, metric_specs):
    for offset, method_label in zip(offsets, method_order):
        vals = np.full(len(window_order), np.nan, dtype=float)
        errs = np.zeros(len(window_order), dtype=float)
        for wi, window_label in enumerate(window_order):
            row = endpoint_summary_df.loc[
                endpoint_summary_df["window"].astype(str).eq(window_label)
                & endpoint_summary_df["method"].eq(method_label)
            ]
            if row.empty:
                continue
            vals[wi] = float(row.iloc[0][metric_col])
            errs[wi] = float(row.iloc[0].get(f"{metric_col} std", 0.0))
        ax.bar(
            xx + offset,
            vals,
            width=width * 0.92,
            yerr=errs,
            capsize=3,
            color=color_map[method_label],
            label=method_label,
        )
    ax.set_xticks(xx)
    ax.set_xticklabels(window_order, rotation=15, ha="right")
    ax.set_title(metric_label, fontsize=11)
    ax.set_xlabel("Prediction window")
    ax.set_ylabel(metric_label)
    ax.grid(axis="y", alpha=0.22)

handles, labels = axes[0].get_legend_handles_labels()
uniq = {}
for h, l in zip(handles, labels):
    if l and l not in uniq:
        uniq[l] = h
fig.legend(list(uniq.values()), list(uniq.keys()), loc="upper center", ncol=min(len(uniq), 4), frameon=False, bbox_to_anchor=(0.5, 1.04))
plt.show()

module14b_endpoint80_bar_df = endpoint_summary_df.copy()
display(module14b_endpoint80_bar_df.sort_values(["window", "method"]))


In [ ]:
import seaborn as sns

if "module13_summary_df" not in globals() or module13_summary_df.empty:
    raise ValueError("Run Module 13A first so module13_summary_df is available.")
if "module13_scatter_df" not in globals() or module13_scatter_df.empty:
    raise ValueError("Run Module 13A first so module13_scatter_df is available.")
if "module12_window_overall_df" not in globals() or module12_window_overall_df.empty:
    raise ValueError("Run Module 12 first so module12_window_overall_df is available.")
if "module12_scatter_df" not in globals() or module12_scatter_df.empty:
    raise ValueError("Run Module 12 first so module12_scatter_df is available.")

method_order = [
    "CNN",
    "LSTM",
    "Transformer",
    "PBS",
]
method_colors = dict(zip(method_order, list(sns.color_palette("PuBu", 4)[1:4]) + ["#f89191"]))

window_order = sorted(
    set(module13_summary_df["window"].astype(str))
    | set(module12_window_overall_df["window"].astype(str))
)

trajectory_rows = []
module13_anchor_summary = module13_summary_df.loc[module13_summary_df["regime"].astype(str).eq("anchor_only")].copy()
for model_key, method_label in [
    ("cnn", "CNN"),
    ("lstm", "LSTM"),
    ("transformer", "Transformer"),
]:
    sub = module13_anchor_summary.loc[module13_anchor_summary["model"].astype(str).str.lower().eq(model_key)].copy()
    for row in sub.itertuples(index=False):
        trajectory_rows.append({
            "window": str(row.window),
            "method": method_label,
            "value": float(getattr(row, "rmse_raw")),
            "std": float(getattr(row, "rmse_raw_std", 0.0)),
        })

for _, row in module12_window_overall_df.iterrows():
    trajectory_rows.append({
        "window": str(row["window"]),
        "method": "PBS",
        "value": float(row["rmse raw"]),
        "std": float(row.get("rmse raw std", 0.0)),
    })

trajectory_plot_df = pd.DataFrame(trajectory_rows)


def compact_endpoint80_mape(scatter_df, method_label, model_key=None):
    base = scatter_df.loc[scatter_df["SOH"].eq(80)].copy()
    if model_key is not None:
        base = base.loc[
            base["regime"].astype(str).eq("anchor_only")
            & base["model"].astype(str).str.lower().eq(model_key)
        ].copy()
    rows = []
    for (window_label, split_idx), sub in base.groupby(["window", "anchor_split"], sort=True):
        true_raw = sub["true_raw_cycle"].to_numpy(dtype=float)
        pred_raw = sub["pred_raw_cycle"].to_numpy(dtype=float)
        rows.append({
            "window": str(window_label),
            "anchor_split": int(split_idx),
            "method": method_label,
            "mape80": float(np.mean(100.0 * np.abs(pred_raw - true_raw) / np.maximum(np.abs(true_raw), 1e-12))),
        })
    return pd.DataFrame(rows)


endpoint_parts = []
for model_key, method_label in [
    ("cnn", "CNN"),
    ("lstm", "LSTM"),
    ("transformer", "Transformer"),
]:
    endpoint_parts.append(compact_endpoint80_mape(module13_scatter_df, method_label, model_key))
endpoint_parts.append(compact_endpoint80_mape(module12_scatter_df, "PBS"))

endpoint_split_df = pd.concat([part for part in endpoint_parts if part is not None and not part.empty], ignore_index=True)
endpoint_plot_df = (
    endpoint_split_df
    .groupby(["window", "method"], as_index=False)
    .agg(
        value=("mape80", "mean"),
        std=("mape80", lambda s: float(np.std(s, ddof=0))),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=500, constrained_layout=True)
plot_specs = [
    (axes[0], trajectory_plot_df, "Trajectory RMSE raw", "raw RMSE"),
    (axes[1], endpoint_plot_df, "80% SOH MAPE", "80% SOH MAPE (%)"),
]

xx = np.arange(len(window_order), dtype=float)
width = min(0.18, 0.80 / max(len(method_order), 1))
offsets = (np.arange(len(method_order)) - (len(method_order) - 1) / 2.0) * width

for ax, plot_df, title, ylabel in plot_specs:
    for offset, method in zip(offsets, method_order):
        vals = []
        errs = []
        for window in window_order:
            row = plot_df.loc[
                plot_df["window"].astype(str).eq(window)
                & plot_df["method"].astype(str).eq(method)
            ]
            vals.append(float(row.iloc[0]["value"]) if not row.empty else np.nan)
            errs.append(float(row.iloc[0]["std"]) if not row.empty else 0.0)
        ax.bar(
            xx + offset,
            vals,
            width=width * 0.92,
            yerr=errs,
            capsize=3,
            color=method_colors[method],
            alpha=0.8,
            edgecolor="none",
            label=method,
        )
    ax.set_xlabel("Prediction window (SOH%)", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xticks(xx)
    ax.set_xticklabels(window_order, fontsize=10)

# Set yticks/ylim for the second subplot
axes[1].set_yticks([0, 5, 10, 15, 20])
axes[1].set_ylim(0, 22)

# ---------- Add legend to each subplot at upper right ----------
axes[0].legend(loc='upper left', frameon=False)
axes[1].legend(loc='upper left', frameon=False)

plt.savefig(FIGURE_DIR / "F5b.tiff", dpi=500, bbox_inches="tight")
plt.show()

display(trajectory_plot_df.sort_values(["window", "method"]))
display(endpoint_plot_df.sort_values(["window", "method"]))
